# Dynamics 08 — Programmes transparents comme institutions : CUPOD, DUPOC, PrudentBot, CIMCIC (Rust)

**Navigation** : [<< Dynamics-07 (Python)](Dynamics-07-Equilibres-par-Simulation-Python.ipynb) | [Index](README.md) | [06i-Capstone >>](../GameTheory-06i-Ensembles-Limites-Poincare-Bendixson-Python.ipynb) | [09-BiensPublics >>](Dynamics-09-Biens-Publics-N-Personnes-Rust.ipynb) | [Python](Dynamics-08-Institutions-Transparentes-Python.ipynb) | [C#](Dynamics-08-Institutions-Transparentes-CSharp.ipynb)

Dans le jeu d'actions du Dilemme du prisonnier (PD), `D` domine strictement `C` et
`(D, D)` est l'unique équilibre de Nash, même si les joueurs connaissent les codes. Le jeu
de **programmes** modifie l'espace des stratégies : remplacer son programme peut changer
la réaction adverse au code publié. Critch, Dennis et Russell (2022) étudient deux
**institutions** — ou deux programmes — qui
lisent mutuellement leurs statuts publics (« bylaws », code source) avant de jouer une partie
unique. Ils y montrent que des institutions formelles qui devraient se trahir coopèrent, et
réciproquement (arXiv:2208.07006, p. 1 et 3).

Ce notebook en tire un modèle **borné et terminant** dont les quatre agents sont `CUPOD`,
`DUPOC`, `PrudentBot` et `CIMCIC`, exécutable pas à pas, sans prétendre trancher une question
ouverte.

**Lien** : Voir aussi [la version Python](Dynamics-08-Institutions-Transparentes-Python.ipynb) — jumeau de référence du présent carnet : les sorties numériques Rust sont calibrées sur ses sorties committées (registre de conformité `scripts/notebook_tools/rust_conformance.d/dynamics-08-institutions-transparentes.yaml`).


## 0. Source primaire et conventions de nommage

**Source primaire de ce travail** : le carnet Python versionné. La notice bibliographique héritée ci-dessous est conservée sans revendiquer une nouvelle lecture du texte : Andrew Critch, Michael Dennis, Stuart Russell, « Cooperative and
uncooperative institution designs: Surprises and problems in open-source game theory »,
arXiv:2208.07006 (v1, 15 août 2022), Center for Human-Compatible AI, UC Berkeley.

| Élément cité | Localisation exacte |
| --- | --- |
| Cadre « open-source game theory », lecture du code source avant l'action | p. 3 |
| Définition de `CUPOD` (coopérer sauf preuve de défection) | p. 9 |
| Définition de `DUPOC` (faire défection sauf preuve de coopération) | p. 9 et 10 |
| Exemples 2.1, 3.1 et 3.2 (rôle de la borne `k`) | p. 7, 10 et notes 3 et 4 |
| Propositions 3.1 et 3.2 (non-exploitabilité de `CUPOD` et de `DUPOC`) | p. 11 |
| Théorème 3.4 : pour `k` grand, `outcome(CUPOD(k), CUPOD(k)) == (D, D)` | p. 13 |
| Théorème 3.7 : pour `k` grand, `outcome(DUPOC(k), DUPOC(k)) == (C, C)` | p. 14 |
| Lemme 3.6 (PBLT, Parametric Bounded Löb Theorem) | p. 13 |
| `PDUPOC` et Théorème 4.1 | p. 19 |
| Définition de `CIMCIC` et Proposition 5.1 | p. 20 et 21 |
| Théorème 5.2 : (a) `CIMCIC` contre `CIMCIC`, (b) `DUPOC` contre `CIMCIC` | p. 21 et 22 |
| « Cooperative affidavit » pour des institutions de type DUPOC | p. 16 |
| `PrudentBot` (agent non borné de LaVictoire et al. 2017) et Open Problem 9 | p. 26 |
| Les dix problèmes ouverts | p. 15, 18, 20, 22, 24, 25, 26 et 27 |

**Conventions de nommage** : les analogues bornés des définitions publiées portent le nom publié
sans suffixe (`CUPOD`, `DUPOC`, `CIMCIC`). Les sigles anglais d'origine sont `CUPOD`
(`Cooperate Unless Proof Of Defection`) et `DUPOC` (`Defect Unless Proof Of Cooperation`, p. 9).
`PrudentBot_borne` est un **candidat** : l'existence
même d'une version bornée de PrudentBot est l'Open Problem 9 (p. 26), qui n'est pas résolu ici.
`CooperateBot`, `DefectBot` et `CooperateBotOpake` sont des agents de référence ou de contrôle ;
les deux premiers reprennent les noms de la p. 6, le troisième est un contrôle local.

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :

- énoncer pourquoi `(D, D)` est l'unique équilibre de Nash du PD canonique, et pourquoi la
  transparence des programmes change ce verdict (Critch-Dennis-Russell 2022, p. 3) ;
- définir une sémantique **bornée, totale et terminante** des agents `CUPOD`, `DUPOC`,
  `PrudentBot` et `CIMCIC`, à partir d'un certificat public et d'une clôture finie ;
- distinguer rigoureusement trois régimes de preuve : **observation finie**, **résultat
  reproduit**, **théorème cité** ;
- reproduire une matrice de confrontations avec **deux implémentations indépendantes** et
  comparer mécaniquement les relations et les matrices obtenues ;
- exhiber un **contrôle négatif** où la transparence seule ne crée pas la coopération ;
- lire le tableau des **dix problèmes ouverts** de l'article, dont l'Open Problem 3, laissé
  explicitement ouvert — ni exercice à solution attendue, ni résultat de notebook.

### Prérequis

- Rust (kernel `rust` d'evcxr Jupyter) et sa bibliothèque standard seulement : le portage n'utilise aucune caisse externe.
- La notion d'équilibre de Nash en stratégies pures, et le PD sous forme normale.
- Lecture utile (facultative) : `Dynamics-04-Open-Source-Game-Theory-Python.ipynb` pour le cadre
  des jeux-programmes, et `Dynamics-07-Equilibres-par-Simulation-Python.ipynb` pour un
  analogue fondé sur la simulation.

### Durée estimée : 45 minutes

## 1. Le Dilemme du prisonnier canonique

On pose la matrice canonique `(T=5, R=3, P=1, S=0)` avec `T > R > P > S` et `2R > T + S`.

| | C | D |
| --- | --- | --- |
| **C** | R, R | S, T |
| **D** | T, S | P, P |

`D` domine strictement `C` pour chaque joueur, quelle que soit l'action adverse : l'unique
équilibre de Nash en stratégies pures est `(D, D)`, quelle que soit la connaissance du code.
Dans le jeu de programmes étudié ensuite, une déviation change le code publié et peut
changer la réponse adverse : elle n'est donc pas une simple substitution d'action à réponse fixée.

In [6]:
// Parametrage canonique du Dilemme du prisonnier (Shoham & Leyton-Brown 2009, §3.4.2).
const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

// Gain du joueur dont l'action est `mien`, face a l'action `adverse`.
fn gain(mien: &str, adverse: &str) -> i64 {
    if mien == "C" { if adverse == "C" { R } else { S } } else if adverse == "C" { T } else { P }
}

assert!(T > R && R > P && P > S, "parametres PD non canoniques");
assert!(2 * R > T + S, "PD strict : la cooperation mutuelle bat l'alternance");

for mien in ["C", "D"] {
    let ligne: Vec<String> = ["C", "D"].iter()
        .map(|adv| format!("{}/{} -> {}", mien, adv, gain(mien, adv)))
        .collect();
    println!("{}", ligne.join("  "));
}

for _mien in ["C", "D"] {
    for adv in ["C", "D"] {
        assert!(gain("D", adv) >= gain("C", adv), "D doit dominer C");
    }
}
println!("D domine C pour chaque joueur : (D, D) est l'unique equilibre de Nash en strategies pures.");


C/C -> 3  C/D -> 0


D/C -> 5  D/D -> 1


D domine C pour chaque joueur : (D, D) est l'unique equilibre de Nash en strategies pures.


## 2. Sémantique bornée et terminante du modèle

L'article définit les agents au moyen d'un `proof_search` (p. 7, §2.1) : un agent cherche une
**preuve formelle**, bornée en longueur par `k` caractères, d'un énoncé portant sur le code de
l'adversaire. Cette recherche présuppose un système de preuve (arithmétique de Peano ou une
extension) et une énumération des preuves ; elle n'est pas exécutable telle quelle dans un
notebook, et elle n'est pas décidable dans le cas non borné.

Ce notebook remplace la recherche de preuve par une **clôture finie et totale** :

1. chaque programme publie un **certificat** — un ensemble fini d'atomes tirés d'un vocabulaire
   fixe. C'est l'analogue des « bylaws » d'une institution, que la partie adverse peut lire ;
2. une liste finie de **règles d'inférence** dérive des faits sur l'adversaire à partir de son
   **seul** certificat ;
3. `proof_search(k, source, but)` teste si le but appartient à la clôture obtenue en **au plus
   `k` tours** d'application des règles ;
4. `k` compte des **tours d'inférence**, et non des caractères de preuve comme dans l'article :
   les seuils numériques ne sont donc **pas comparables**, seule la **monotonie** (`k` plus grand
   rend davantage de faits dérivables) est reproduite.

Trois limites de ce choix, à garder en tête pendant toute la lecture :

- la clôture est **finie** — l'univers des atomes dérivables est fixé à l'avance — donc la
  sémantique est **totale** et **terminante**, contrairement au `proof_search` non borné ;
- elle est **incomplète** vis-à-vis de l'article : elle ne peut pas dériver les faits qui
  exigent le lemme PBLT (lemme 3.6, p. 13), dont dépendent les théorèmes 3.4 et 3.7 ;
- un certificat **opaque** (vide) ne fournit aucun fait initial à la clôture. Dans le contrôle
  `DUPOC/CooperateBotOpake`, `DUPOC` fait défection : c'est l'analogue ciblé du point (3)
  de la p. 18. Ce n'est pas une propriété de tous les agents : `CooperateBot` et `CUPOD`
  coopèrent avec cet adversaire opaque dans la matrice finie.

In [7]:
// ---------------------------------------------------------------------------
// Modele borne : univers fini d'atomes certifies et regles d'inference.
// ---------------------------------------------------------------------------
const K_DEFAUT: usize = 3;

const ATOMES_CERTIFIES: [&str; 6] = [
    "uncond_coop",             // cooperer inconditionnellement
    "uncond_defect",           // faire defection inconditionnellement
    "self_cond_coop",          // condition de cooperation auto-referentielle (CIMCIC)
    "needs_proof_of_coop",     // cooperer seulement avec une preuve de cooperation (DUPOC)
    "needs_proof_of_defect",   // faire defection seulement sur preuve de defection (CUPOD)
    "prudent",                 // exiger en plus une preuve que l'adversaire fait defection contre DefectBot
];

const REGLES: [(&[&str], &[&str]); 9] = [
    (&["uncond_coop"], &["intent_coop"]),
    (&["uncond_defect"], &["intent_defect"]),
    (&["self_cond_coop"], &["implies_own_coop"]),
    (&["intent_coop"], &["coops_against_any", "coops_with_db"]),
    (&["intent_defect"], &["defects_against_any", "defects_with_db"]),
    (&["needs_proof_of_coop"], &["defects_with_db"]),
    (&["needs_proof_of_defect"], &["defects_with_db"]),
    (&["self_cond_coop"], &["defects_with_db"]),
    (&["prudent"], &["defects_with_db"]),
];

println!("Atomes certifies : {} | regles : {}", ATOMES_CERTIFIES.len(), REGLES.len());
for (premisse, conclusions) in REGLES.iter() {
    let mut p: Vec<&str> = premisse.to_vec();
    let mut c: Vec<&str> = conclusions.to_vec();
    p.sort_unstable();
    c.sort_unstable();
    println!("  {}  =>  {}", p.join(","), c.join(","));
}
println!();
println!("Les quatre regles a conclusion `defects_with_db` encodent un fait relatif au DefectBot");
println!("canonique : un agent qui n'obtient jamais de preuve de cooperation fait defection contre lui.");
println!("C'est l'ancrage utilise par PrudentBot dans LaVictoire et al. 2017 (cite p. 26).");


Atomes certifies : 6 | regles : 9


  uncond_coop  =>  intent_coop


  uncond_defect  =>  intent_defect


  self_cond_coop  =>  implies_own_coop


  intent_coop  =>  coops_against_any,coops_with_db


  intent_defect  =>  defects_against_any,defects_with_db


... (3 outputs left out)

  prudent  =>  defects_with_db


Les quatre regles a conclusion `defects_with_db` encodent un fait relatif au DefectBot


canonique : un agent qui n'obtient jamais de preuve de cooperation fait defection contre lui.


C'est l'ancrage utilise par PrudentBot dans LaVictoire et al. 2017 (cite p. 26).


### 2.1 Les sept programmes et leur certificat

Chaque programme est un texte Python : une fonction `nom(opp_source, k)` et un champ
`bylaws_json` qui publie son certificat. Le certificat est **la** source d'information de
l'adversaire — un agent sans certificat est opaque.

| Programme | Rôle | Certificat publié |
| --- | --- | --- |
| `CooperateBot` | coopère toujours (p. 6) | `uncond_coop` |
| `DefectBot` | fait toujours défection (p. 6) | `uncond_defect` |
| `CooperateBotOpake` | coopère toujours, mais ne publie rien | aucun atome |
| `CUPOD` | fait défection s'il détient une preuve de défection (p. 9) | `needs_proof_of_defect` |
| `DUPOC` | coopère seulement sur preuve de coopération (p. 9 et 10) | `needs_proof_of_coop` |
| `CIMCIC` | coopère si sa coopération implique celle de l'autre (p. 20) | `self_cond_coop` |
| `PrudentBot_borne` | candidat borné de PrudentBot (Open Problem 9, p. 26) | `prudent`, `needs_proof_of_coop` |

Le certificat est publié **deux fois**, sous deux formes destinées à deux lecteurs distincts :
des marqueurs `# BYLAW:` lus par le moteur principal, et le champ `bylaws_json` lu par le
vérificateur indépendant. Cette redondance est délibérée : elle rend la comparaison des deux
implémentations significative (cf. section 8).

Dans ce jumeau Rust, ces textes restent des **données** : un interprète du sous-langage publié (retour littéral, `chr(...)`, affectation, `if proof_search`) en exécute la décision, pour qu'elle vienne du texte publié plutôt que d'une réécriture à côté.


In [8]:
// Les marqueurs et le JSON décrivent les mêmes certificats pour deux lecteurs indépendants.
// Textes publies, ordre d'insertion du dict source (valeurs exactes des litteraux Python).
const SOURCES: [(&str, &str); 7] = [
    ("CooperateBot", r#"def CooperateBot(opp_source, k):
    return "C"
# BYLAW: uncond_coop
bylaws_json = '{"atoms": ["uncond_coop"]}'
"#),
    ("DefectBot", r#"def DefectBot(opp_source, k):
    return "D"
# BYLAW: uncond_defect
bylaws_json = '{"atoms": ["uncond_defect"]}'
"#),
    ("CooperateBotOpake", r#"def CooperateBotOpake(opp_source, k):
    _code = 67
    return chr(_code)
# BYLAW:
bylaws_json = '{"atoms": []}'
"#),
    ("CUPOD", r#"def CUPOD(opp_source, k):
    if proof_search(k, opp_source, "defects_against_any"):
        return "D"
    return "C"
# BYLAW: needs_proof_of_defect
bylaws_json = '{"atoms": ["needs_proof_of_defect"]}'
"#),
    ("DUPOC", r#"def DUPOC(opp_source, k):
    if proof_search(k, opp_source, "coops_against_any"):
        return "C"
    return "D"
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["needs_proof_of_coop"]}'
"#),
    ("CIMCIC", r#"def CIMCIC(opp_source, k):
    if proof_search(k, opp_source, "implies_own_coop"):
        return "C"
    return "D"
# BYLAW: self_cond_coop
bylaws_json = '{"atoms": ["self_cond_coop"]}'
"#),
    ("PrudentBot_borne", r#"def PrudentBot_borne(opp_source, k):
    if not proof_search(k, opp_source, "coops_against_any"):
        return "D"
    if not proof_search(k, opp_source, "defects_with_db"):
        return "D"
    return "C"
# BYLAW: prudent
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["prudent", "needs_proof_of_coop"]}'
"#),
];

const ORDRE: [&str; 7] = ["CooperateBot", "DefectBot", "CooperateBotOpake",
                          "CUPOD", "DUPOC", "CIMCIC", "PrudentBot_borne"];

fn source(nom: &str) -> &'static str {
    SOURCES.iter().find(|(n, _)| *n == nom).map(|(_, s)| *s).expect("programme publie")
}

for nom in ORDRE.iter() {
    println!("{:<18} {}", nom, source(nom).lines().next().expect("texte publie non vide"));
}
println!();
println!("{} programmes publies, chacun avec sa fonction et son certificat.", SOURCES.len());


CooperateBot       def CooperateBot(opp_source, k):


DefectBot          def DefectBot(opp_source, k):


CooperateBotOpake  def CooperateBotOpake(opp_source, k):


CUPOD              def CUPOD(opp_source, k):


DUPOC              def DUPOC(opp_source, k):


CIMCIC             def CIMCIC(opp_source, k):


PrudentBot_borne   def PrudentBot_borne(opp_source, k):


7 programmes publies, chacun avec sa fonction et son certificat.


## 3. Moteur principal (implémentation 1)

Le moteur lit le certificat par ses marqueurs `# BYLAW:`, calcule la clôture bornée et expose
`proof_search(k, source, but)` aux programmes. La clôture s'arrête dès qu'un tour n'ajoute
plus rien : elle est donc terminante, et le résultat est **monotone** en `k`.

Dans ce jumeau Rust, l'exécution se fait sans `exec` : c'est l'interprète du texte publié qui tient `proof_search` à disposition des programmes chargés.


In [9]:
// ---- Bloc moteur Dynamics-08 (std seul) — a recopier tel quel en tete des cellules 10 a 35 ----
use std::collections::{BTreeMap, BTreeSet, HashMap};

const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

fn gain(mien: &str, adverse: &str) -> i64 {
    if mien == "C" { if adverse == "C" { R } else { S } } else if adverse == "C" { T } else { P }
}

const K_DEFAUT: usize = 3;

const REGLES: [(&[&str], &[&str]); 9] = [
    (&["uncond_coop"], &["intent_coop"]),
    (&["uncond_defect"], &["intent_defect"]),
    (&["self_cond_coop"], &["implies_own_coop"]),
    (&["intent_coop"], &["coops_against_any", "coops_with_db"]),
    (&["intent_defect"], &["defects_against_any", "defects_with_db"]),
    (&["needs_proof_of_coop"], &["defects_with_db"]),
    (&["needs_proof_of_defect"], &["defects_with_db"]),
    (&["self_cond_coop"], &["defects_with_db"]),
    (&["prudent"], &["defects_with_db"]),
];

// Textes publies, ordre d'insertion du dict source (valeurs exactes des litteraux Python).
const SOURCES: [(&str, &str); 7] = [
    ("CooperateBot", r#"def CooperateBot(opp_source, k):
    return "C"
# BYLAW: uncond_coop
bylaws_json = '{"atoms": ["uncond_coop"]}'
"#),
    ("DefectBot", r#"def DefectBot(opp_source, k):
    return "D"
# BYLAW: uncond_defect
bylaws_json = '{"atoms": ["uncond_defect"]}'
"#),
    ("CooperateBotOpake", r#"def CooperateBotOpake(opp_source, k):
    _code = 67
    return chr(_code)
# BYLAW:
bylaws_json = '{"atoms": []}'
"#),
    ("CUPOD", r#"def CUPOD(opp_source, k):
    if proof_search(k, opp_source, "defects_against_any"):
        return "D"
    return "C"
# BYLAW: needs_proof_of_defect
bylaws_json = '{"atoms": ["needs_proof_of_defect"]}'
"#),
    ("DUPOC", r#"def DUPOC(opp_source, k):
    if proof_search(k, opp_source, "coops_against_any"):
        return "C"
    return "D"
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["needs_proof_of_coop"]}'
"#),
    ("CIMCIC", r#"def CIMCIC(opp_source, k):
    if proof_search(k, opp_source, "implies_own_coop"):
        return "C"
    return "D"
# BYLAW: self_cond_coop
bylaws_json = '{"atoms": ["self_cond_coop"]}'
"#),
    ("PrudentBot_borne", r#"def PrudentBot_borne(opp_source, k):
    if not proof_search(k, opp_source, "coops_against_any"):
        return "D"
    if not proof_search(k, opp_source, "defects_with_db"):
        return "D"
    return "C"
# BYLAW: prudent
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["prudent", "needs_proof_of_coop"]}'
"#),
];

const ORDRE: [&str; 7] = ["CooperateBot", "DefectBot", "CooperateBotOpake",
                          "CUPOD", "DUPOC", "CIMCIC", "PrudentBot_borne"];

fn source(nom: &str) -> &'static str {
    SOURCES.iter().find(|(n, _)| *n == nom).map(|(_, s)| *s).expect("programme publie")
}

// Equivalent ligne a ligne de re.compile(r"^#\s*BYLAW:\s*([a-z_]+)\s*$", re.MULTILINE).
fn atome_bylaw(ligne: &str) -> Option<String> {
    let reste = ligne.strip_prefix('#')?.trim_start().strip_prefix("BYLAW:")?.trim();
    if !reste.is_empty() && reste.chars().all(|c| c.is_ascii_lowercase() || c == '_') {
        Some(reste.to_string())
    } else {
        None
    }
}

fn parse_certificat_a(source: &str) -> BTreeSet<String> {
    source.lines().filter_map(atome_bylaw).collect()
}

fn cloture_a(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut grossi = faits.clone();
        for (premisse, conclusions) in REGLES.iter() {
            if premisse.iter().all(|p| faits.contains(*p)) {
                grossi.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if grossi == faits {
            break;
        }
        faits = grossi;
    }
    faits
}

fn proof_search(k: usize, opp_source: &str, but: &str) -> bool {
    cloture_a(&parse_certificat_a(opp_source), k).contains(but)
}

// Interprete du sous-langage des textes publies (remplace exec : la decision vient du texte).
#[derive(Debug)]
enum Instr {
    Retour(String),                                     // return "X"
    Affecte(String, i64),                               // _code = 67
    RetourChr(String),                                  // return chr(_code) | return chr(67)
    Si { negation: bool, but: String, retour: String }, // if [not] proof_search(k, opp_source, "but"): return "X"
}

#[derive(Debug)]
struct Programme { nom: String, corps: Vec<Instr> }

fn chaine_python(s: &str) -> Option<String> {
    let s = s.trim();
    let q = s.chars().next()?;
    if (q == '"' || q == '\'') && s.len() >= 2 && s.ends_with(q) {
        let interieur = &s[1..s.len() - 1];
        if !interieur.contains(q) && !interieur.contains('\\') {
            return Some(interieur.to_string());
        }
    }
    None
}

fn charger(source: &str) -> Result<Programme, String> {
    let mut lignes = source.lines();
    let tete = lignes.next().ok_or("source vide")?;
    let nom = tete.strip_prefix("def ").and_then(|r| r.strip_suffix("(opp_source, k):"))
        .ok_or(format!("en-tete non reconnu : {}", tete))?.to_string();
    let corps_lignes: Vec<&str> = lignes.take_while(|l| l.starts_with("    ")).collect();
    let mut corps = Vec::new();
    let mut i = 0;
    while i < corps_lignes.len() {
        let l = &corps_lignes[i][4..];
        if let Some(cond) = l.strip_prefix("if ").and_then(|r| r.strip_suffix(':')) {
            let (negation, appel) = match cond.strip_prefix("not ") {
                Some(r) => (true, r),
                None => (false, cond),
            };
            let but = appel.strip_prefix("proof_search(k, opp_source, ").and_then(|r| r.strip_suffix(')'))
                .and_then(chaine_python).ok_or(format!("condition non reconnue : {}", l))?;
            let retour = corps_lignes.get(i + 1).and_then(|s| s.strip_prefix("        return "))
                .and_then(chaine_python).ok_or(format!("bloc if non reconnu apres : {}", l))?;
            corps.push(Instr::Si { negation, but, retour });
            i += 2;
            continue;
        }
        if let Some(e) = l.strip_prefix("return ") {
            if let Some(v) = chaine_python(e) {
                corps.push(Instr::Retour(v));
            } else if let Some(arg) = e.strip_prefix("chr(").and_then(|r| r.strip_suffix(')')) {
                corps.push(Instr::RetourChr(arg.trim().to_string()));
            } else {
                return Err(format!("retour non reconnu : {}", l));
            }
        } else if let Some((var, val)) = l.split_once(" = ") {
            let v: i64 = val.trim().parse().map_err(|_| format!("affectation non reconnue : {}", l))?;
            corps.push(Instr::Affecte(var.trim().to_string(), v));
        } else {
            return Err(format!("instruction non reconnue : {}", l));
        }
        i += 1;
    }
    Ok(Programme { nom, corps })
}

fn executer(p: &Programme, opp_source: &str, k: usize) -> Result<String, String> {
    let mut vars: HashMap<String, i64> = HashMap::new();
    for ins in &p.corps {
        match ins {
            Instr::Retour(v) => return Ok(v.clone()),
            Instr::Affecte(n, v) => {
                vars.insert(n.clone(), *v);
            }
            Instr::RetourChr(arg) => {
                let code = match arg.parse::<i64>() {
                    Ok(v) => v,
                    Err(_) => *vars.get(arg).ok_or(format!("nom inconnu : {}", arg))?,
                };
                return char::from_u32(code as u32).map(|c| c.to_string()).ok_or(format!("chr hors domaine : {}", code));
            }
            Instr::Si { negation, but, retour } => {
                if proof_search(k, opp_source, but) != *negation {
                    return Ok(retour.clone());
                }
            }
        }
    }
    Err(format!("{} se termine sans return", p.nom))
}

type Issue = (String, String, (i64, i64));

struct Moteur { programmes: Vec<(String, Programme)> }

impl Moteur {
    // PROGRAMMES = {nom: charger(SOURCES[nom])[nom] for nom in ORDRE}
    fn charger_tout() -> Moteur {
        let programmes = ORDRE.iter().map(|nom| {
            let p = charger(source(nom)).expect("texte publie interpretable");
            assert!(p.nom == *nom, "la source ne definit pas {}", nom);
            (nom.to_string(), p)
        }).collect();
        Moteur { programmes }
    }
    fn programme(&self, nom: &str) -> &Programme {
        &self.programmes.iter().find(|(n, _)| n == nom).expect("programme charge").1
    }
    fn action(&self, nom: &str, adverse: &str, k: usize) -> String {
        executer(self.programme(nom), source(adverse), k).expect("execution du programme")
    }
    fn issue(&self, nom_a: &str, nom_b: &str, k: usize) -> Issue {
        let act_a = self.action(nom_a, nom_b, k);
        let act_b = self.action(nom_b, nom_a, k);
        let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
        (act_a, act_b, g)
    }
}

fn paires() -> Vec<(String, String)> {
    ORDRE.iter().flat_map(|a| ORDRE.iter().map(move |b| (a.to_string(), b.to_string()))).collect()
}

// ISSUE_MOTEUR = {(a, b): issue(a, b) for a, b in itertools.product(ORDRE, repeat=2)} (ordre d'insertion)
fn table_moteur(m: &Moteur) -> Vec<((String, String), Issue)> {
    paires().into_iter().map(|(a, b)| {
        let i = m.issue(&a, &b, K_DEFAUT);
        ((a, b), i)
    }).collect()
}

fn chercher<'a>(table: &'a [((String, String), Issue)], a: &str, b: &str) -> &'a Issue {
    &table.iter().find(|((x, y), _)| x == a && y == b).expect("paire du registre").1
}

fn py_list(items: &BTreeSet<String>) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn py_pair(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }
fn py_gains(g: (i64, i64)) -> String { format!("({}, {})", g.0, g.1) }
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
// ---- fin du bloc moteur ----

for nom in ["CooperateBot", "DUPOC", "CIMCIC"] {
    println!("{:<14} cert={}", nom, py_list(&parse_certificat_a(source(nom))));
    println!("{:<14} clot(k=3)={}", "", py_list(&cloture_a(&parse_certificat_a(source(nom)), K_DEFAUT)));
};


CooperateBot   cert=['uncond_coop']


               clot(k=3)=['coops_against_any', 'coops_with_db', 'intent_coop', 'uncond_coop']


DUPOC          cert=['needs_proof_of_coop']


               clot(k=3)=['defects_with_db', 'needs_proof_of_coop']


CIMCIC         cert=['self_cond_coop']


               clot(k=3)=['defects_with_db', 'implies_own_coop', 'self_cond_coop']


### 3.1 Exécution des programmes

La transparence est ici **effective** : chaque fonction reçoit le **texte** du programme adverse
et s'en sert pour interroger `proof_search`. On charge les sept programmes depuis leur texte : l'interprète du jumeau Rust en exécute le corps, avec le seul `proof_search` borné à disposition.

In [10]:
// ---- Bloc moteur Dynamics-08 (std seul) — a recopier tel quel en tete des cellules 10 a 35 ----
use std::collections::{BTreeMap, BTreeSet, HashMap};

const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

fn gain(mien: &str, adverse: &str) -> i64 {
    if mien == "C" { if adverse == "C" { R } else { S } } else if adverse == "C" { T } else { P }
}

const K_DEFAUT: usize = 3;

const REGLES: [(&[&str], &[&str]); 9] = [
    (&["uncond_coop"], &["intent_coop"]),
    (&["uncond_defect"], &["intent_defect"]),
    (&["self_cond_coop"], &["implies_own_coop"]),
    (&["intent_coop"], &["coops_against_any", "coops_with_db"]),
    (&["intent_defect"], &["defects_against_any", "defects_with_db"]),
    (&["needs_proof_of_coop"], &["defects_with_db"]),
    (&["needs_proof_of_defect"], &["defects_with_db"]),
    (&["self_cond_coop"], &["defects_with_db"]),
    (&["prudent"], &["defects_with_db"]),
];

// Textes publies, ordre d'insertion du dict source (valeurs exactes des litteraux Python).
const SOURCES: [(&str, &str); 7] = [
    ("CooperateBot", r#"def CooperateBot(opp_source, k):
    return "C"
# BYLAW: uncond_coop
bylaws_json = '{"atoms": ["uncond_coop"]}'
"#),
    ("DefectBot", r#"def DefectBot(opp_source, k):
    return "D"
# BYLAW: uncond_defect
bylaws_json = '{"atoms": ["uncond_defect"]}'
"#),
    ("CooperateBotOpake", r#"def CooperateBotOpake(opp_source, k):
    _code = 67
    return chr(_code)
# BYLAW:
bylaws_json = '{"atoms": []}'
"#),
    ("CUPOD", r#"def CUPOD(opp_source, k):
    if proof_search(k, opp_source, "defects_against_any"):
        return "D"
    return "C"
# BYLAW: needs_proof_of_defect
bylaws_json = '{"atoms": ["needs_proof_of_defect"]}'
"#),
    ("DUPOC", r#"def DUPOC(opp_source, k):
    if proof_search(k, opp_source, "coops_against_any"):
        return "C"
    return "D"
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["needs_proof_of_coop"]}'
"#),
    ("CIMCIC", r#"def CIMCIC(opp_source, k):
    if proof_search(k, opp_source, "implies_own_coop"):
        return "C"
    return "D"
# BYLAW: self_cond_coop
bylaws_json = '{"atoms": ["self_cond_coop"]}'
"#),
    ("PrudentBot_borne", r#"def PrudentBot_borne(opp_source, k):
    if not proof_search(k, opp_source, "coops_against_any"):
        return "D"
    if not proof_search(k, opp_source, "defects_with_db"):
        return "D"
    return "C"
# BYLAW: prudent
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["prudent", "needs_proof_of_coop"]}'
"#),
];

const ORDRE: [&str; 7] = ["CooperateBot", "DefectBot", "CooperateBotOpake",
                          "CUPOD", "DUPOC", "CIMCIC", "PrudentBot_borne"];

fn source(nom: &str) -> &'static str {
    SOURCES.iter().find(|(n, _)| *n == nom).map(|(_, s)| *s).expect("programme publie")
}

// Equivalent ligne a ligne de re.compile(r"^#\s*BYLAW:\s*([a-z_]+)\s*$", re.MULTILINE).
fn atome_bylaw(ligne: &str) -> Option<String> {
    let reste = ligne.strip_prefix('#')?.trim_start().strip_prefix("BYLAW:")?.trim();
    if !reste.is_empty() && reste.chars().all(|c| c.is_ascii_lowercase() || c == '_') {
        Some(reste.to_string())
    } else {
        None
    }
}

fn parse_certificat_a(source: &str) -> BTreeSet<String> {
    source.lines().filter_map(atome_bylaw).collect()
}

fn cloture_a(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut grossi = faits.clone();
        for (premisse, conclusions) in REGLES.iter() {
            if premisse.iter().all(|p| faits.contains(*p)) {
                grossi.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if grossi == faits {
            break;
        }
        faits = grossi;
    }
    faits
}

fn proof_search(k: usize, opp_source: &str, but: &str) -> bool {
    cloture_a(&parse_certificat_a(opp_source), k).contains(but)
}

// Interprete du sous-langage des textes publies (remplace exec : la decision vient du texte).
#[derive(Debug)]
enum Instr {
    Retour(String),                                     // return "X"
    Affecte(String, i64),                               // _code = 67
    RetourChr(String),                                  // return chr(_code) | return chr(67)
    Si { negation: bool, but: String, retour: String }, // if [not] proof_search(k, opp_source, "but"): return "X"
}

#[derive(Debug)]
struct Programme { nom: String, corps: Vec<Instr> }

fn chaine_python(s: &str) -> Option<String> {
    let s = s.trim();
    let q = s.chars().next()?;
    if (q == '"' || q == '\'') && s.len() >= 2 && s.ends_with(q) {
        let interieur = &s[1..s.len() - 1];
        if !interieur.contains(q) && !interieur.contains('\\') {
            return Some(interieur.to_string());
        }
    }
    None
}

fn charger(source: &str) -> Result<Programme, String> {
    let mut lignes = source.lines();
    let tete = lignes.next().ok_or("source vide")?;
    let nom = tete.strip_prefix("def ").and_then(|r| r.strip_suffix("(opp_source, k):"))
        .ok_or(format!("en-tete non reconnu : {}", tete))?.to_string();
    let corps_lignes: Vec<&str> = lignes.take_while(|l| l.starts_with("    ")).collect();
    let mut corps = Vec::new();
    let mut i = 0;
    while i < corps_lignes.len() {
        let l = &corps_lignes[i][4..];
        if let Some(cond) = l.strip_prefix("if ").and_then(|r| r.strip_suffix(':')) {
            let (negation, appel) = match cond.strip_prefix("not ") {
                Some(r) => (true, r),
                None => (false, cond),
            };
            let but = appel.strip_prefix("proof_search(k, opp_source, ").and_then(|r| r.strip_suffix(')'))
                .and_then(chaine_python).ok_or(format!("condition non reconnue : {}", l))?;
            let retour = corps_lignes.get(i + 1).and_then(|s| s.strip_prefix("        return "))
                .and_then(chaine_python).ok_or(format!("bloc if non reconnu apres : {}", l))?;
            corps.push(Instr::Si { negation, but, retour });
            i += 2;
            continue;
        }
        if let Some(e) = l.strip_prefix("return ") {
            if let Some(v) = chaine_python(e) {
                corps.push(Instr::Retour(v));
            } else if let Some(arg) = e.strip_prefix("chr(").and_then(|r| r.strip_suffix(')')) {
                corps.push(Instr::RetourChr(arg.trim().to_string()));
            } else {
                return Err(format!("retour non reconnu : {}", l));
            }
        } else if let Some((var, val)) = l.split_once(" = ") {
            let v: i64 = val.trim().parse().map_err(|_| format!("affectation non reconnue : {}", l))?;
            corps.push(Instr::Affecte(var.trim().to_string(), v));
        } else {
            return Err(format!("instruction non reconnue : {}", l));
        }
        i += 1;
    }
    Ok(Programme { nom, corps })
}

fn executer(p: &Programme, opp_source: &str, k: usize) -> Result<String, String> {
    let mut vars: HashMap<String, i64> = HashMap::new();
    for ins in &p.corps {
        match ins {
            Instr::Retour(v) => return Ok(v.clone()),
            Instr::Affecte(n, v) => {
                vars.insert(n.clone(), *v);
            }
            Instr::RetourChr(arg) => {
                let code = match arg.parse::<i64>() {
                    Ok(v) => v,
                    Err(_) => *vars.get(arg).ok_or(format!("nom inconnu : {}", arg))?,
                };
                return char::from_u32(code as u32).map(|c| c.to_string()).ok_or(format!("chr hors domaine : {}", code));
            }
            Instr::Si { negation, but, retour } => {
                if proof_search(k, opp_source, but) != *negation {
                    return Ok(retour.clone());
                }
            }
        }
    }
    Err(format!("{} se termine sans return", p.nom))
}

type Issue = (String, String, (i64, i64));

struct Moteur { programmes: Vec<(String, Programme)> }

impl Moteur {
    // PROGRAMMES = {nom: charger(SOURCES[nom])[nom] for nom in ORDRE}
    fn charger_tout() -> Moteur {
        let programmes = ORDRE.iter().map(|nom| {
            let p = charger(source(nom)).expect("texte publie interpretable");
            assert!(p.nom == *nom, "la source ne definit pas {}", nom);
            (nom.to_string(), p)
        }).collect();
        Moteur { programmes }
    }
    fn programme(&self, nom: &str) -> &Programme {
        &self.programmes.iter().find(|(n, _)| n == nom).expect("programme charge").1
    }
    fn action(&self, nom: &str, adverse: &str, k: usize) -> String {
        executer(self.programme(nom), source(adverse), k).expect("execution du programme")
    }
    fn issue(&self, nom_a: &str, nom_b: &str, k: usize) -> Issue {
        let act_a = self.action(nom_a, nom_b, k);
        let act_b = self.action(nom_b, nom_a, k);
        let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
        (act_a, act_b, g)
    }
}

fn paires() -> Vec<(String, String)> {
    ORDRE.iter().flat_map(|a| ORDRE.iter().map(move |b| (a.to_string(), b.to_string()))).collect()
}

// ISSUE_MOTEUR = {(a, b): issue(a, b) for a, b in itertools.product(ORDRE, repeat=2)} (ordre d'insertion)
fn table_moteur(m: &Moteur) -> Vec<((String, String), Issue)> {
    paires().into_iter().map(|(a, b)| {
        let i = m.issue(&a, &b, K_DEFAUT);
        ((a, b), i)
    }).collect()
}

fn chercher<'a>(table: &'a [((String, String), Issue)], a: &str, b: &str) -> &'a Issue {
    &table.iter().find(|((x, y), _)| x == a && y == b).expect("paire du registre").1
}

fn py_list(items: &BTreeSet<String>) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn py_pair(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }
fn py_gains(g: (i64, i64)) -> String { format!("({}, {})", g.0, g.1) }
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
// ---- fin du bloc moteur ----

let m = Moteur::charger_tout();
for nom in ORDRE.iter() {
    let adversaire = "DefectBot";
    let action = m.action(nom, adversaire, K_DEFAUT);
    println!("{:<18} contre {:<11} -> {}", nom, adversaire, action);
}
println!();
println!("{} programmes charges et appelables.", m.programmes.len());


CooperateBot       contre DefectBot   -> C


DefectBot          contre DefectBot   -> D


CooperateBotOpake  contre DefectBot   -> C


CUPOD              contre DefectBot   -> D


DUPOC              contre DefectBot   -> D


CIMCIC             contre DefectBot   -> D


PrudentBot_borne   contre DefectBot   -> D


7 programmes charges et appelables.


### 3.2 Issue d'une confrontation en un coup

Une confrontation `(A, B)` fournit à chaque programme le texte de l'autre. L'issue est le
couple d'actions, suivi des deux gains.

In [11]:
// ---- Bloc moteur Dynamics-08 (std seul) — a recopier tel quel en tete des cellules 10 a 35 ----
use std::collections::{BTreeMap, BTreeSet, HashMap};

const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

fn gain(mien: &str, adverse: &str) -> i64 {
    if mien == "C" { if adverse == "C" { R } else { S } } else if adverse == "C" { T } else { P }
}

const K_DEFAUT: usize = 3;

const REGLES: [(&[&str], &[&str]); 9] = [
    (&["uncond_coop"], &["intent_coop"]),
    (&["uncond_defect"], &["intent_defect"]),
    (&["self_cond_coop"], &["implies_own_coop"]),
    (&["intent_coop"], &["coops_against_any", "coops_with_db"]),
    (&["intent_defect"], &["defects_against_any", "defects_with_db"]),
    (&["needs_proof_of_coop"], &["defects_with_db"]),
    (&["needs_proof_of_defect"], &["defects_with_db"]),
    (&["self_cond_coop"], &["defects_with_db"]),
    (&["prudent"], &["defects_with_db"]),
];

// Textes publies, ordre d'insertion du dict source (valeurs exactes des litteraux Python).
const SOURCES: [(&str, &str); 7] = [
    ("CooperateBot", r#"def CooperateBot(opp_source, k):
    return "C"
# BYLAW: uncond_coop
bylaws_json = '{"atoms": ["uncond_coop"]}'
"#),
    ("DefectBot", r#"def DefectBot(opp_source, k):
    return "D"
# BYLAW: uncond_defect
bylaws_json = '{"atoms": ["uncond_defect"]}'
"#),
    ("CooperateBotOpake", r#"def CooperateBotOpake(opp_source, k):
    _code = 67
    return chr(_code)
# BYLAW:
bylaws_json = '{"atoms": []}'
"#),
    ("CUPOD", r#"def CUPOD(opp_source, k):
    if proof_search(k, opp_source, "defects_against_any"):
        return "D"
    return "C"
# BYLAW: needs_proof_of_defect
bylaws_json = '{"atoms": ["needs_proof_of_defect"]}'
"#),
    ("DUPOC", r#"def DUPOC(opp_source, k):
    if proof_search(k, opp_source, "coops_against_any"):
        return "C"
    return "D"
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["needs_proof_of_coop"]}'
"#),
    ("CIMCIC", r#"def CIMCIC(opp_source, k):
    if proof_search(k, opp_source, "implies_own_coop"):
        return "C"
    return "D"
# BYLAW: self_cond_coop
bylaws_json = '{"atoms": ["self_cond_coop"]}'
"#),
    ("PrudentBot_borne", r#"def PrudentBot_borne(opp_source, k):
    if not proof_search(k, opp_source, "coops_against_any"):
        return "D"
    if not proof_search(k, opp_source, "defects_with_db"):
        return "D"
    return "C"
# BYLAW: prudent
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["prudent", "needs_proof_of_coop"]}'
"#),
];

const ORDRE: [&str; 7] = ["CooperateBot", "DefectBot", "CooperateBotOpake",
                          "CUPOD", "DUPOC", "CIMCIC", "PrudentBot_borne"];

fn source(nom: &str) -> &'static str {
    SOURCES.iter().find(|(n, _)| *n == nom).map(|(_, s)| *s).expect("programme publie")
}

// Equivalent ligne a ligne de re.compile(r"^#\s*BYLAW:\s*([a-z_]+)\s*$", re.MULTILINE).
fn atome_bylaw(ligne: &str) -> Option<String> {
    let reste = ligne.strip_prefix('#')?.trim_start().strip_prefix("BYLAW:")?.trim();
    if !reste.is_empty() && reste.chars().all(|c| c.is_ascii_lowercase() || c == '_') {
        Some(reste.to_string())
    } else {
        None
    }
}

fn parse_certificat_a(source: &str) -> BTreeSet<String> {
    source.lines().filter_map(atome_bylaw).collect()
}

fn cloture_a(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut grossi = faits.clone();
        for (premisse, conclusions) in REGLES.iter() {
            if premisse.iter().all(|p| faits.contains(*p)) {
                grossi.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if grossi == faits {
            break;
        }
        faits = grossi;
    }
    faits
}

fn proof_search(k: usize, opp_source: &str, but: &str) -> bool {
    cloture_a(&parse_certificat_a(opp_source), k).contains(but)
}

// Interprete du sous-langage des textes publies (remplace exec : la decision vient du texte).
#[derive(Debug)]
enum Instr {
    Retour(String),                                     // return "X"
    Affecte(String, i64),                               // _code = 67
    RetourChr(String),                                  // return chr(_code) | return chr(67)
    Si { negation: bool, but: String, retour: String }, // if [not] proof_search(k, opp_source, "but"): return "X"
}

#[derive(Debug)]
struct Programme { nom: String, corps: Vec<Instr> }

fn chaine_python(s: &str) -> Option<String> {
    let s = s.trim();
    let q = s.chars().next()?;
    if (q == '"' || q == '\'') && s.len() >= 2 && s.ends_with(q) {
        let interieur = &s[1..s.len() - 1];
        if !interieur.contains(q) && !interieur.contains('\\') {
            return Some(interieur.to_string());
        }
    }
    None
}

fn charger(source: &str) -> Result<Programme, String> {
    let mut lignes = source.lines();
    let tete = lignes.next().ok_or("source vide")?;
    let nom = tete.strip_prefix("def ").and_then(|r| r.strip_suffix("(opp_source, k):"))
        .ok_or(format!("en-tete non reconnu : {}", tete))?.to_string();
    let corps_lignes: Vec<&str> = lignes.take_while(|l| l.starts_with("    ")).collect();
    let mut corps = Vec::new();
    let mut i = 0;
    while i < corps_lignes.len() {
        let l = &corps_lignes[i][4..];
        if let Some(cond) = l.strip_prefix("if ").and_then(|r| r.strip_suffix(':')) {
            let (negation, appel) = match cond.strip_prefix("not ") {
                Some(r) => (true, r),
                None => (false, cond),
            };
            let but = appel.strip_prefix("proof_search(k, opp_source, ").and_then(|r| r.strip_suffix(')'))
                .and_then(chaine_python).ok_or(format!("condition non reconnue : {}", l))?;
            let retour = corps_lignes.get(i + 1).and_then(|s| s.strip_prefix("        return "))
                .and_then(chaine_python).ok_or(format!("bloc if non reconnu apres : {}", l))?;
            corps.push(Instr::Si { negation, but, retour });
            i += 2;
            continue;
        }
        if let Some(e) = l.strip_prefix("return ") {
            if let Some(v) = chaine_python(e) {
                corps.push(Instr::Retour(v));
            } else if let Some(arg) = e.strip_prefix("chr(").and_then(|r| r.strip_suffix(')')) {
                corps.push(Instr::RetourChr(arg.trim().to_string()));
            } else {
                return Err(format!("retour non reconnu : {}", l));
            }
        } else if let Some((var, val)) = l.split_once(" = ") {
            let v: i64 = val.trim().parse().map_err(|_| format!("affectation non reconnue : {}", l))?;
            corps.push(Instr::Affecte(var.trim().to_string(), v));
        } else {
            return Err(format!("instruction non reconnue : {}", l));
        }
        i += 1;
    }
    Ok(Programme { nom, corps })
}

fn executer(p: &Programme, opp_source: &str, k: usize) -> Result<String, String> {
    let mut vars: HashMap<String, i64> = HashMap::new();
    for ins in &p.corps {
        match ins {
            Instr::Retour(v) => return Ok(v.clone()),
            Instr::Affecte(n, v) => {
                vars.insert(n.clone(), *v);
            }
            Instr::RetourChr(arg) => {
                let code = match arg.parse::<i64>() {
                    Ok(v) => v,
                    Err(_) => *vars.get(arg).ok_or(format!("nom inconnu : {}", arg))?,
                };
                return char::from_u32(code as u32).map(|c| c.to_string()).ok_or(format!("chr hors domaine : {}", code));
            }
            Instr::Si { negation, but, retour } => {
                if proof_search(k, opp_source, but) != *negation {
                    return Ok(retour.clone());
                }
            }
        }
    }
    Err(format!("{} se termine sans return", p.nom))
}

type Issue = (String, String, (i64, i64));

struct Moteur { programmes: Vec<(String, Programme)> }

impl Moteur {
    // PROGRAMMES = {nom: charger(SOURCES[nom])[nom] for nom in ORDRE}
    fn charger_tout() -> Moteur {
        let programmes = ORDRE.iter().map(|nom| {
            let p = charger(source(nom)).expect("texte publie interpretable");
            assert!(p.nom == *nom, "la source ne definit pas {}", nom);
            (nom.to_string(), p)
        }).collect();
        Moteur { programmes }
    }
    fn programme(&self, nom: &str) -> &Programme {
        &self.programmes.iter().find(|(n, _)| n == nom).expect("programme charge").1
    }
    fn action(&self, nom: &str, adverse: &str, k: usize) -> String {
        executer(self.programme(nom), source(adverse), k).expect("execution du programme")
    }
    fn issue(&self, nom_a: &str, nom_b: &str, k: usize) -> Issue {
        let act_a = self.action(nom_a, nom_b, k);
        let act_b = self.action(nom_b, nom_a, k);
        let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
        (act_a, act_b, g)
    }
}

fn paires() -> Vec<(String, String)> {
    ORDRE.iter().flat_map(|a| ORDRE.iter().map(move |b| (a.to_string(), b.to_string()))).collect()
}

// ISSUE_MOTEUR = {(a, b): issue(a, b) for a, b in itertools.product(ORDRE, repeat=2)} (ordre d'insertion)
fn table_moteur(m: &Moteur) -> Vec<((String, String), Issue)> {
    paires().into_iter().map(|(a, b)| {
        let i = m.issue(&a, &b, K_DEFAUT);
        ((a, b), i)
    }).collect()
}

fn chercher<'a>(table: &'a [((String, String), Issue)], a: &str, b: &str) -> &'a Issue {
    &table.iter().find(|((x, y), _)| x == a && y == b).expect("paire du registre").1
}

fn py_list(items: &BTreeSet<String>) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn py_pair(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }
fn py_gains(g: (i64, i64)) -> String { format!("({}, {})", g.0, g.1) }
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
// ---- fin du bloc moteur ----

let etiquette: [(&str, &str); 7] = [("CooperateBot", "CB"), ("DefectBot", "DB"),
                                    ("CooperateBotOpake", "CBo"), ("CUPOD", "CU"),
                                    ("DUPOC", "DU"), ("CIMCIC", "CI"), ("PrudentBot_borne", "PB")];
fn etiq<'a>(table: &[(&'a str, &'a str); 7], n: &str) -> &'a str {
    table.iter().find(|(x, _)| *x == n).map(|(_, e)| *e).expect("etiquette connue")
}

let m = Moteur::charger_tout();
let im = table_moteur(&m);
let legende: Vec<String> = ORDRE.iter().map(|n| format!("{}={}", etiq(&etiquette, n), n)).collect();
println!("Legende : {}", legende.join(", "));
println!();
println!("Action du programme en ligne contre le programme en colonne :");
print!("      ");
for n in ORDRE.iter() { print!("{:>5}", etiq(&etiquette, n)); }
println!();
for a in ORDRE.iter() {
    print!("{:>4}  ", etiq(&etiquette, a));
    for b in ORDRE.iter() { print!("{:>5}", chercher(&im, a, b).0); }
    println!();
}
println!();
println!("Gain du programme en ligne (celui du programme en colonne s'en deduit par transposition,");
println!("le jeu etant symetrique : gain_colonne(A, B) == gain_ligne(B, A)) :");
for a in ORDRE.iter() {
    print!("{:>4}  ", etiq(&etiquette, a));
    for b in ORDRE.iter() { print!("{:>5}", chercher(&im, a, b).2.0); }
    println!();
};


Legende : CB=CooperateBot, DB=DefectBot, CBo=CooperateBotOpake, CU=CUPOD, DU=DUPOC, CI=CIMCIC, PB=PrudentBot_borne


Action du programme en ligne contre le programme en colonne :


         CB   DB  CBo   CU   DU   CI   PB


  CB      C    C    C    C    C    C    C


  DB      D    D    D    D    D    D    D


 CBo      C    C    C    C    C    C    C


... (9 outputs left out)

  CU      3    1    3    3    0    0    0


  DU      3    1    5    5    1    1    1


  CI      5    1    5    5    1    3    1


  PB      5    1    5    5    1    1    1


### Lecture de la matrice

Quatre phénomènes sont visibles dans cette table (verdict **calculé** dans le modèle borné,
jamais une preuve) :

1. **Coopération mutuelle** : `DUPOC` contre `CooperateBot` donne `(C, C)` — le patient
   coopère parce que le certificat de `CooperateBot` est lisible, et réciproquement.
2. **Inexploitation** : `DUPOC` contre `CooperateBotOpake` donne `(D, C)`. Le contrôle opaque
   se comporte pourtant exactement comme `CooperateBot` ; seule sa **représentation** diffère.
3. **Défection révélée par la lecture** : `CUPOD` contre `DefectBot` donne `(D, D)` : `CUPOD`
   trouve dans le certificat de `DefectBot` la preuve qu'il va faire défection.
4. **Non-coopération sous transparence totale** : `DUPOC` contre `DUPOC` donne `(D, D)` — les
   deux programmes lisent tout le texte de l'autre et ne coopèrent pas pour autant. C'est le
   contrôle négatif de la section 5.

Les théorèmes 3.4 et 3.7 de l'article prévoient respectivement `(D, D)` pour
`CUPOD/CUPOD` et `(C, C)` pour `DUPOC/DUPOC` **quand `k` est grand** : l'écart entre ces énoncés et la table ci-dessus est
le sujet du classement de la section 4.

## 4. Classification : observation finie, résultat reproduit, théorème cité

Chaque affirmation de ce notebook appartient à **un seul** de ces trois régimes.

| Régime | Définition | Exemple dans ce notebook |
| --- | --- | --- |
| **Résultat reproduit** | Une affirmation de la source que le modèle borné retrouve, et qui **ne dépend pas** du lemme PBLT | Exemple 2.1, exemples 3.1 et 3.2, propositions 3.1 et 3.2, proposition 5.1, théorème 5.2(a) |
| **Observation finie** | Un calcul du modèle borné, **sans** énoncé correspondant dans la source, ou avec un énoncé qui en diffère | La table complète des 49 paires, la ligne de `PrudentBot_borne`, l'Open Problem 3 (laissé ouvert, section 9) |
| **Théorème cité** | Un énoncé de la source **cité** pour mémoire, que le modèle borné **ne reproduit pas** | Théorèmes 3.4 et 3.7 (PBLT), théorème 5.2(b) |

Le théorème 5.2(a) mérite une note : l'article le démontre **sans** invoquer PBLT pour l'essentiel,
car la preuve requise se réduit à l'implication `X => X` — une tautologie (p. 21). C'est pourquoi
le modèle borné le retrouve, alors qu'il échoue sur les théorèmes 3.4 et 3.7, dont la preuve passe
par le point fixe de PBLT.

In [12]:
// ---- Bloc moteur Dynamics-08 (std seul) — a recopier tel quel en tete des cellules 10 a 35 ----
use std::collections::{BTreeMap, BTreeSet, HashMap};

const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

fn gain(mien: &str, adverse: &str) -> i64 {
    if mien == "C" { if adverse == "C" { R } else { S } } else if adverse == "C" { T } else { P }
}

const K_DEFAUT: usize = 3;

const REGLES: [(&[&str], &[&str]); 9] = [
    (&["uncond_coop"], &["intent_coop"]),
    (&["uncond_defect"], &["intent_defect"]),
    (&["self_cond_coop"], &["implies_own_coop"]),
    (&["intent_coop"], &["coops_against_any", "coops_with_db"]),
    (&["intent_defect"], &["defects_against_any", "defects_with_db"]),
    (&["needs_proof_of_coop"], &["defects_with_db"]),
    (&["needs_proof_of_defect"], &["defects_with_db"]),
    (&["self_cond_coop"], &["defects_with_db"]),
    (&["prudent"], &["defects_with_db"]),
];

// Textes publies, ordre d'insertion du dict source (valeurs exactes des litteraux Python).
const SOURCES: [(&str, &str); 7] = [
    ("CooperateBot", r#"def CooperateBot(opp_source, k):
    return "C"
# BYLAW: uncond_coop
bylaws_json = '{"atoms": ["uncond_coop"]}'
"#),
    ("DefectBot", r#"def DefectBot(opp_source, k):
    return "D"
# BYLAW: uncond_defect
bylaws_json = '{"atoms": ["uncond_defect"]}'
"#),
    ("CooperateBotOpake", r#"def CooperateBotOpake(opp_source, k):
    _code = 67
    return chr(_code)
# BYLAW:
bylaws_json = '{"atoms": []}'
"#),
    ("CUPOD", r#"def CUPOD(opp_source, k):
    if proof_search(k, opp_source, "defects_against_any"):
        return "D"
    return "C"
# BYLAW: needs_proof_of_defect
bylaws_json = '{"atoms": ["needs_proof_of_defect"]}'
"#),
    ("DUPOC", r#"def DUPOC(opp_source, k):
    if proof_search(k, opp_source, "coops_against_any"):
        return "C"
    return "D"
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["needs_proof_of_coop"]}'
"#),
    ("CIMCIC", r#"def CIMCIC(opp_source, k):
    if proof_search(k, opp_source, "implies_own_coop"):
        return "C"
    return "D"
# BYLAW: self_cond_coop
bylaws_json = '{"atoms": ["self_cond_coop"]}'
"#),
    ("PrudentBot_borne", r#"def PrudentBot_borne(opp_source, k):
    if not proof_search(k, opp_source, "coops_against_any"):
        return "D"
    if not proof_search(k, opp_source, "defects_with_db"):
        return "D"
    return "C"
# BYLAW: prudent
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["prudent", "needs_proof_of_coop"]}'
"#),
];

const ORDRE: [&str; 7] = ["CooperateBot", "DefectBot", "CooperateBotOpake",
                          "CUPOD", "DUPOC", "CIMCIC", "PrudentBot_borne"];

fn source(nom: &str) -> &'static str {
    SOURCES.iter().find(|(n, _)| *n == nom).map(|(_, s)| *s).expect("programme publie")
}

// Equivalent ligne a ligne de re.compile(r"^#\s*BYLAW:\s*([a-z_]+)\s*$", re.MULTILINE).
fn atome_bylaw(ligne: &str) -> Option<String> {
    let reste = ligne.strip_prefix('#')?.trim_start().strip_prefix("BYLAW:")?.trim();
    if !reste.is_empty() && reste.chars().all(|c| c.is_ascii_lowercase() || c == '_') {
        Some(reste.to_string())
    } else {
        None
    }
}

fn parse_certificat_a(source: &str) -> BTreeSet<String> {
    source.lines().filter_map(atome_bylaw).collect()
}

fn cloture_a(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut grossi = faits.clone();
        for (premisse, conclusions) in REGLES.iter() {
            if premisse.iter().all(|p| faits.contains(*p)) {
                grossi.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if grossi == faits {
            break;
        }
        faits = grossi;
    }
    faits
}

fn proof_search(k: usize, opp_source: &str, but: &str) -> bool {
    cloture_a(&parse_certificat_a(opp_source), k).contains(but)
}

// Interprete du sous-langage des textes publies (remplace exec : la decision vient du texte).
#[derive(Debug)]
enum Instr {
    Retour(String),                                     // return "X"
    Affecte(String, i64),                               // _code = 67
    RetourChr(String),                                  // return chr(_code) | return chr(67)
    Si { negation: bool, but: String, retour: String }, // if [not] proof_search(k, opp_source, "but"): return "X"
}

#[derive(Debug)]
struct Programme { nom: String, corps: Vec<Instr> }

fn chaine_python(s: &str) -> Option<String> {
    let s = s.trim();
    let q = s.chars().next()?;
    if (q == '"' || q == '\'') && s.len() >= 2 && s.ends_with(q) {
        let interieur = &s[1..s.len() - 1];
        if !interieur.contains(q) && !interieur.contains('\\') {
            return Some(interieur.to_string());
        }
    }
    None
}

fn charger(source: &str) -> Result<Programme, String> {
    let mut lignes = source.lines();
    let tete = lignes.next().ok_or("source vide")?;
    let nom = tete.strip_prefix("def ").and_then(|r| r.strip_suffix("(opp_source, k):"))
        .ok_or(format!("en-tete non reconnu : {}", tete))?.to_string();
    let corps_lignes: Vec<&str> = lignes.take_while(|l| l.starts_with("    ")).collect();
    let mut corps = Vec::new();
    let mut i = 0;
    while i < corps_lignes.len() {
        let l = &corps_lignes[i][4..];
        if let Some(cond) = l.strip_prefix("if ").and_then(|r| r.strip_suffix(':')) {
            let (negation, appel) = match cond.strip_prefix("not ") {
                Some(r) => (true, r),
                None => (false, cond),
            };
            let but = appel.strip_prefix("proof_search(k, opp_source, ").and_then(|r| r.strip_suffix(')'))
                .and_then(chaine_python).ok_or(format!("condition non reconnue : {}", l))?;
            let retour = corps_lignes.get(i + 1).and_then(|s| s.strip_prefix("        return "))
                .and_then(chaine_python).ok_or(format!("bloc if non reconnu apres : {}", l))?;
            corps.push(Instr::Si { negation, but, retour });
            i += 2;
            continue;
        }
        if let Some(e) = l.strip_prefix("return ") {
            if let Some(v) = chaine_python(e) {
                corps.push(Instr::Retour(v));
            } else if let Some(arg) = e.strip_prefix("chr(").and_then(|r| r.strip_suffix(')')) {
                corps.push(Instr::RetourChr(arg.trim().to_string()));
            } else {
                return Err(format!("retour non reconnu : {}", l));
            }
        } else if let Some((var, val)) = l.split_once(" = ") {
            let v: i64 = val.trim().parse().map_err(|_| format!("affectation non reconnue : {}", l))?;
            corps.push(Instr::Affecte(var.trim().to_string(), v));
        } else {
            return Err(format!("instruction non reconnue : {}", l));
        }
        i += 1;
    }
    Ok(Programme { nom, corps })
}

fn executer(p: &Programme, opp_source: &str, k: usize) -> Result<String, String> {
    let mut vars: HashMap<String, i64> = HashMap::new();
    for ins in &p.corps {
        match ins {
            Instr::Retour(v) => return Ok(v.clone()),
            Instr::Affecte(n, v) => {
                vars.insert(n.clone(), *v);
            }
            Instr::RetourChr(arg) => {
                let code = match arg.parse::<i64>() {
                    Ok(v) => v,
                    Err(_) => *vars.get(arg).ok_or(format!("nom inconnu : {}", arg))?,
                };
                return char::from_u32(code as u32).map(|c| c.to_string()).ok_or(format!("chr hors domaine : {}", code));
            }
            Instr::Si { negation, but, retour } => {
                if proof_search(k, opp_source, but) != *negation {
                    return Ok(retour.clone());
                }
            }
        }
    }
    Err(format!("{} se termine sans return", p.nom))
}

type Issue = (String, String, (i64, i64));

struct Moteur { programmes: Vec<(String, Programme)> }

impl Moteur {
    // PROGRAMMES = {nom: charger(SOURCES[nom])[nom] for nom in ORDRE}
    fn charger_tout() -> Moteur {
        let programmes = ORDRE.iter().map(|nom| {
            let p = charger(source(nom)).expect("texte publie interpretable");
            assert!(p.nom == *nom, "la source ne definit pas {}", nom);
            (nom.to_string(), p)
        }).collect();
        Moteur { programmes }
    }
    fn programme(&self, nom: &str) -> &Programme {
        &self.programmes.iter().find(|(n, _)| n == nom).expect("programme charge").1
    }
    fn action(&self, nom: &str, adverse: &str, k: usize) -> String {
        executer(self.programme(nom), source(adverse), k).expect("execution du programme")
    }
    fn issue(&self, nom_a: &str, nom_b: &str, k: usize) -> Issue {
        let act_a = self.action(nom_a, nom_b, k);
        let act_b = self.action(nom_b, nom_a, k);
        let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
        (act_a, act_b, g)
    }
}

fn paires() -> Vec<(String, String)> {
    ORDRE.iter().flat_map(|a| ORDRE.iter().map(move |b| (a.to_string(), b.to_string()))).collect()
}

// ISSUE_MOTEUR = {(a, b): issue(a, b) for a, b in itertools.product(ORDRE, repeat=2)} (ordre d'insertion)
fn table_moteur(m: &Moteur) -> Vec<((String, String), Issue)> {
    paires().into_iter().map(|(a, b)| {
        let i = m.issue(&a, &b, K_DEFAUT);
        ((a, b), i)
    }).collect()
}

fn chercher<'a>(table: &'a [((String, String), Issue)], a: &str, b: &str) -> &'a Issue {
    &table.iter().find(|((x, y), _)| x == a && y == b).expect("paire du registre").1
}

fn py_list(items: &BTreeSet<String>) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn py_pair(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }
fn py_gains(g: (i64, i64)) -> String { format!("({}, {})", g.0, g.1) }
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
// ---- fin du bloc moteur ----

// ---------------------------------------------------------------------------
// Verification mecanique des affirmations classees « resultat reproduit ».
// ---------------------------------------------------------------------------
let m = Moteur::charger_tout();
let ex21 = m.issue("CooperateBot", "DefectBot", K_DEFAUT);
assert!((ex21.0.as_str(), ex21.1.as_str()) == ("C", "D"), "{:?}", ex21);
println!("Exemple 2.1 reproduit : outcome(CB, DB) = {} {}", py_pair(&ex21.0, &ex21.1), py_gains(ex21.2));

let bas = m.issue("CUPOD", "DefectBot", 1);
let haut = m.issue("CUPOD", "DefectBot", K_DEFAUT);
assert!((bas.0.as_str(), bas.1.as_str()) == ("C", "D") && (haut.0.as_str(), haut.1.as_str()) == ("D", "D"));
println!("Exemples 3.1 et 3.2 reproduits qualitativement : k=1 -> {} | k=3 -> {}",
         py_pair(&bas.0, &bas.1), py_pair(&haut.0, &haut.1));
println!("  (l'article mesure k en caracteres de preuve, ici en tours d'inference)");

let ci = m.issue("CIMCIC", "CIMCIC", K_DEFAUT);
assert!((ci.0.as_str(), ci.1.as_str()) == ("C", "C"), "{:?}", ci);
println!("Theoreme 5.2(a) reproduit : outcome(CIMCIC, CIMCIC) = {} {}", py_pair(&ci.0, &ci.1), py_gains(ci.2));

// Rend une paire temoin si `agent` est dans l'issue interdite, sinon None (ordre d'insertion).
fn exploite_par(table: &[((String, String), Issue)], agent: &str, issue_interdite: &str) -> Option<(String, String)> {
    for ((x, y), (ax, ay, _)) in table {
        if x == agent && format!("{}{}", ax, ay) == issue_interdite {
            return Some((x.clone(), y.clone()));
        }
    }
    None
}

let im = table_moteur(&m);
for (agent, interdite, src) in [("CUPOD", "DC", "Proposition 3.1 (p. 11)"),
                                ("DUPOC", "CD", "Proposition 3.2 (p. 11)"),
                                ("CIMCIC", "CD", "Proposition 5.1 (p. 20)")] {
    let temoin = exploite_par(&im, agent, interdite);
    assert!(temoin.is_none(), "{:?} {:?} {:?}", agent, interdite, temoin);
    println!("{} : {} n'est jamais dans l'issue {} sur les paires du registre.", src, agent, interdite);
};


Exemple 2.1 reproduit : outcome(CB, DB) = ('C', 'D') (0, 5)


Exemples 3.1 et 3.2 reproduits qualitativement : k=1 -> ('C', 'D') | k=3 -> ('D', 'D')


  (l'article mesure k en caracteres de preuve, ici en tours d'inference)


Theoreme 5.2(a) reproduit : outcome(CIMCIC, CIMCIC) = ('C', 'C') (3, 3)


Proposition 3.1 (p. 11) : CUPOD n'est jamais dans l'issue DC sur les paires du registre.


Proposition 3.2 (p. 11) : DUPOC n'est jamais dans l'issue CD sur les paires du registre.


Proposition 5.1 (p. 20) : CIMCIC n'est jamais dans l'issue CD sur les paires du registre.


## 5. Contrôle négatif : la transparence seule ne crée pas la coopération

Les programmes lisent ici **tout** le texte de l'adversaire. La question est de savoir si cette
transparence suffit à produire la coopération mutuelle. La réponse du modèle borné est non, et
c'est précisément l'endroit où il se sépare de l'article.

In [13]:
// ---- Bloc moteur Dynamics-08 (std seul) — a recopier tel quel en tete des cellules 10 a 35 ----
use std::collections::{BTreeMap, BTreeSet, HashMap};

const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

fn gain(mien: &str, adverse: &str) -> i64 {
    if mien == "C" { if adverse == "C" { R } else { S } } else if adverse == "C" { T } else { P }
}

const K_DEFAUT: usize = 3;

const REGLES: [(&[&str], &[&str]); 9] = [
    (&["uncond_coop"], &["intent_coop"]),
    (&["uncond_defect"], &["intent_defect"]),
    (&["self_cond_coop"], &["implies_own_coop"]),
    (&["intent_coop"], &["coops_against_any", "coops_with_db"]),
    (&["intent_defect"], &["defects_against_any", "defects_with_db"]),
    (&["needs_proof_of_coop"], &["defects_with_db"]),
    (&["needs_proof_of_defect"], &["defects_with_db"]),
    (&["self_cond_coop"], &["defects_with_db"]),
    (&["prudent"], &["defects_with_db"]),
];

// Textes publies, ordre d'insertion du dict source (valeurs exactes des litteraux Python).
const SOURCES: [(&str, &str); 7] = [
    ("CooperateBot", r#"def CooperateBot(opp_source, k):
    return "C"
# BYLAW: uncond_coop
bylaws_json = '{"atoms": ["uncond_coop"]}'
"#),
    ("DefectBot", r#"def DefectBot(opp_source, k):
    return "D"
# BYLAW: uncond_defect
bylaws_json = '{"atoms": ["uncond_defect"]}'
"#),
    ("CooperateBotOpake", r#"def CooperateBotOpake(opp_source, k):
    _code = 67
    return chr(_code)
# BYLAW:
bylaws_json = '{"atoms": []}'
"#),
    ("CUPOD", r#"def CUPOD(opp_source, k):
    if proof_search(k, opp_source, "defects_against_any"):
        return "D"
    return "C"
# BYLAW: needs_proof_of_defect
bylaws_json = '{"atoms": ["needs_proof_of_defect"]}'
"#),
    ("DUPOC", r#"def DUPOC(opp_source, k):
    if proof_search(k, opp_source, "coops_against_any"):
        return "C"
    return "D"
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["needs_proof_of_coop"]}'
"#),
    ("CIMCIC", r#"def CIMCIC(opp_source, k):
    if proof_search(k, opp_source, "implies_own_coop"):
        return "C"
    return "D"
# BYLAW: self_cond_coop
bylaws_json = '{"atoms": ["self_cond_coop"]}'
"#),
    ("PrudentBot_borne", r#"def PrudentBot_borne(opp_source, k):
    if not proof_search(k, opp_source, "coops_against_any"):
        return "D"
    if not proof_search(k, opp_source, "defects_with_db"):
        return "D"
    return "C"
# BYLAW: prudent
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["prudent", "needs_proof_of_coop"]}'
"#),
];

const ORDRE: [&str; 7] = ["CooperateBot", "DefectBot", "CooperateBotOpake",
                          "CUPOD", "DUPOC", "CIMCIC", "PrudentBot_borne"];

fn source(nom: &str) -> &'static str {
    SOURCES.iter().find(|(n, _)| *n == nom).map(|(_, s)| *s).expect("programme publie")
}

// Equivalent ligne a ligne de re.compile(r"^#\s*BYLAW:\s*([a-z_]+)\s*$", re.MULTILINE).
fn atome_bylaw(ligne: &str) -> Option<String> {
    let reste = ligne.strip_prefix('#')?.trim_start().strip_prefix("BYLAW:")?.trim();
    if !reste.is_empty() && reste.chars().all(|c| c.is_ascii_lowercase() || c == '_') {
        Some(reste.to_string())
    } else {
        None
    }
}

fn parse_certificat_a(source: &str) -> BTreeSet<String> {
    source.lines().filter_map(atome_bylaw).collect()
}

fn cloture_a(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut grossi = faits.clone();
        for (premisse, conclusions) in REGLES.iter() {
            if premisse.iter().all(|p| faits.contains(*p)) {
                grossi.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if grossi == faits {
            break;
        }
        faits = grossi;
    }
    faits
}

fn proof_search(k: usize, opp_source: &str, but: &str) -> bool {
    cloture_a(&parse_certificat_a(opp_source), k).contains(but)
}

// Interprete du sous-langage des textes publies (remplace exec : la decision vient du texte).
#[derive(Debug)]
enum Instr {
    Retour(String),                                     // return "X"
    Affecte(String, i64),                               // _code = 67
    RetourChr(String),                                  // return chr(_code) | return chr(67)
    Si { negation: bool, but: String, retour: String }, // if [not] proof_search(k, opp_source, "but"): return "X"
}

#[derive(Debug)]
struct Programme { nom: String, corps: Vec<Instr> }

fn chaine_python(s: &str) -> Option<String> {
    let s = s.trim();
    let q = s.chars().next()?;
    if (q == '"' || q == '\'') && s.len() >= 2 && s.ends_with(q) {
        let interieur = &s[1..s.len() - 1];
        if !interieur.contains(q) && !interieur.contains('\\') {
            return Some(interieur.to_string());
        }
    }
    None
}

fn charger(source: &str) -> Result<Programme, String> {
    let mut lignes = source.lines();
    let tete = lignes.next().ok_or("source vide")?;
    let nom = tete.strip_prefix("def ").and_then(|r| r.strip_suffix("(opp_source, k):"))
        .ok_or(format!("en-tete non reconnu : {}", tete))?.to_string();
    let corps_lignes: Vec<&str> = lignes.take_while(|l| l.starts_with("    ")).collect();
    let mut corps = Vec::new();
    let mut i = 0;
    while i < corps_lignes.len() {
        let l = &corps_lignes[i][4..];
        if let Some(cond) = l.strip_prefix("if ").and_then(|r| r.strip_suffix(':')) {
            let (negation, appel) = match cond.strip_prefix("not ") {
                Some(r) => (true, r),
                None => (false, cond),
            };
            let but = appel.strip_prefix("proof_search(k, opp_source, ").and_then(|r| r.strip_suffix(')'))
                .and_then(chaine_python).ok_or(format!("condition non reconnue : {}", l))?;
            let retour = corps_lignes.get(i + 1).and_then(|s| s.strip_prefix("        return "))
                .and_then(chaine_python).ok_or(format!("bloc if non reconnu apres : {}", l))?;
            corps.push(Instr::Si { negation, but, retour });
            i += 2;
            continue;
        }
        if let Some(e) = l.strip_prefix("return ") {
            if let Some(v) = chaine_python(e) {
                corps.push(Instr::Retour(v));
            } else if let Some(arg) = e.strip_prefix("chr(").and_then(|r| r.strip_suffix(')')) {
                corps.push(Instr::RetourChr(arg.trim().to_string()));
            } else {
                return Err(format!("retour non reconnu : {}", l));
            }
        } else if let Some((var, val)) = l.split_once(" = ") {
            let v: i64 = val.trim().parse().map_err(|_| format!("affectation non reconnue : {}", l))?;
            corps.push(Instr::Affecte(var.trim().to_string(), v));
        } else {
            return Err(format!("instruction non reconnue : {}", l));
        }
        i += 1;
    }
    Ok(Programme { nom, corps })
}

fn executer(p: &Programme, opp_source: &str, k: usize) -> Result<String, String> {
    let mut vars: HashMap<String, i64> = HashMap::new();
    for ins in &p.corps {
        match ins {
            Instr::Retour(v) => return Ok(v.clone()),
            Instr::Affecte(n, v) => {
                vars.insert(n.clone(), *v);
            }
            Instr::RetourChr(arg) => {
                let code = match arg.parse::<i64>() {
                    Ok(v) => v,
                    Err(_) => *vars.get(arg).ok_or(format!("nom inconnu : {}", arg))?,
                };
                return char::from_u32(code as u32).map(|c| c.to_string()).ok_or(format!("chr hors domaine : {}", code));
            }
            Instr::Si { negation, but, retour } => {
                if proof_search(k, opp_source, but) != *negation {
                    return Ok(retour.clone());
                }
            }
        }
    }
    Err(format!("{} se termine sans return", p.nom))
}

type Issue = (String, String, (i64, i64));

struct Moteur { programmes: Vec<(String, Programme)> }

impl Moteur {
    // PROGRAMMES = {nom: charger(SOURCES[nom])[nom] for nom in ORDRE}
    fn charger_tout() -> Moteur {
        let programmes = ORDRE.iter().map(|nom| {
            let p = charger(source(nom)).expect("texte publie interpretable");
            assert!(p.nom == *nom, "la source ne definit pas {}", nom);
            (nom.to_string(), p)
        }).collect();
        Moteur { programmes }
    }
    fn programme(&self, nom: &str) -> &Programme {
        &self.programmes.iter().find(|(n, _)| n == nom).expect("programme charge").1
    }
    fn action(&self, nom: &str, adverse: &str, k: usize) -> String {
        executer(self.programme(nom), source(adverse), k).expect("execution du programme")
    }
    fn issue(&self, nom_a: &str, nom_b: &str, k: usize) -> Issue {
        let act_a = self.action(nom_a, nom_b, k);
        let act_b = self.action(nom_b, nom_a, k);
        let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
        (act_a, act_b, g)
    }
}

fn paires() -> Vec<(String, String)> {
    ORDRE.iter().flat_map(|a| ORDRE.iter().map(move |b| (a.to_string(), b.to_string()))).collect()
}

// ISSUE_MOTEUR = {(a, b): issue(a, b) for a, b in itertools.product(ORDRE, repeat=2)} (ordre d'insertion)
fn table_moteur(m: &Moteur) -> Vec<((String, String), Issue)> {
    paires().into_iter().map(|(a, b)| {
        let i = m.issue(&a, &b, K_DEFAUT);
        ((a, b), i)
    }).collect()
}

fn chercher<'a>(table: &'a [((String, String), Issue)], a: &str, b: &str) -> &'a Issue {
    &table.iter().find(|((x, y), _)| x == a && y == b).expect("paire du registre").1
}

fn py_list(items: &BTreeSet<String>) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn py_pair(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }
fn py_gains(g: (i64, i64)) -> String { format!("({}, {})", g.0, g.1) }
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
// ---- fin du bloc moteur ----

// ---------------------------------------------------------------------------
// Controle negatif : transparence mutuelle totale, sans cooperation.
// ---------------------------------------------------------------------------
let m = Moteur::charger_tout();
let du = m.issue("DUPOC", "DUPOC", K_DEFAUT);
let cu = m.issue("CUPOD", "CUPOD", K_DEFAUT);
assert!((du.0.as_str(), du.1.as_str()) == ("D", "D"), "{:?}", du);
assert!((cu.0.as_str(), cu.1.as_str()) == ("C", "C"), "{:?}", cu);

println!("Observation finie : outcome(DUPOC, DUPOC) = {} {}", py_pair(&du.0, &du.1), py_gains(du.2));
println!("Theoreme cite      : Theoreme 3.7 (p. 14) : pour k grand,");
println!("                     outcome(DUPOC(k), DUPOC(k)) == (C, C), via PBLT.");
println!("  -> sans PBLT, la transparence mutuelle ne suffit pas : les deux font defection.");
println!();
println!("Observation finie : outcome(CUPOD, CUPOD) = {} {}", py_pair(&cu.0, &cu.1), py_gains(cu.2));
println!("Theoreme cite      : Theoreme 3.4 (p. 13) : pour k grand,");
println!("                     outcome(CUPOD(k), CUPOD(k)) == (D, D), via PBLT.");
println!("  -> ecart en miroir : le modele borne est incomplet dans les deux sens.");


Observation finie : outcome(DUPOC, DUPOC) = ('D', 'D') (1, 1)


Theoreme cite      : Theoreme 3.7 (p. 14) : pour k grand,


                     outcome(DUPOC(k), DUPOC(k)) == (C, C), via PBLT.


  -> sans PBLT, la transparence mutuelle ne suffit pas : les deux font defection.


Observation finie : outcome(CUPOD, CUPOD) = ('C', 'C') (3, 3)


Theoreme cite      : Theoreme 3.4 (p. 13) : pour k grand,


                     outcome(CUPOD(k), CUPOD(k)) == (D, D), via PBLT.


  -> ecart en miroir : le modele borne est incomplet dans les deux sens.


## 6. Sortie non triviale : la représentation change une issue

`CooperateBot` et `CooperateBotOpake` ont un **comportement observable identique** : ils
coopèrent contre n'importe quel adversaire. Ils ne diffèrent que par le certificat publié.
Faites les confronter à `DUPOC`, et observez si la transparence du certificat change l'issue.

In [14]:
// ---- Bloc moteur Dynamics-08 (std seul) — a recopier tel quel en tete des cellules 10 a 35 ----
use std::collections::{BTreeMap, BTreeSet, HashMap};

const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

fn gain(mien: &str, adverse: &str) -> i64 {
    if mien == "C" { if adverse == "C" { R } else { S } } else if adverse == "C" { T } else { P }
}

const K_DEFAUT: usize = 3;

const REGLES: [(&[&str], &[&str]); 9] = [
    (&["uncond_coop"], &["intent_coop"]),
    (&["uncond_defect"], &["intent_defect"]),
    (&["self_cond_coop"], &["implies_own_coop"]),
    (&["intent_coop"], &["coops_against_any", "coops_with_db"]),
    (&["intent_defect"], &["defects_against_any", "defects_with_db"]),
    (&["needs_proof_of_coop"], &["defects_with_db"]),
    (&["needs_proof_of_defect"], &["defects_with_db"]),
    (&["self_cond_coop"], &["defects_with_db"]),
    (&["prudent"], &["defects_with_db"]),
];

// Textes publies, ordre d'insertion du dict source (valeurs exactes des litteraux Python).
const SOURCES: [(&str, &str); 7] = [
    ("CooperateBot", r#"def CooperateBot(opp_source, k):
    return "C"
# BYLAW: uncond_coop
bylaws_json = '{"atoms": ["uncond_coop"]}'
"#),
    ("DefectBot", r#"def DefectBot(opp_source, k):
    return "D"
# BYLAW: uncond_defect
bylaws_json = '{"atoms": ["uncond_defect"]}'
"#),
    ("CooperateBotOpake", r#"def CooperateBotOpake(opp_source, k):
    _code = 67
    return chr(_code)
# BYLAW:
bylaws_json = '{"atoms": []}'
"#),
    ("CUPOD", r#"def CUPOD(opp_source, k):
    if proof_search(k, opp_source, "defects_against_any"):
        return "D"
    return "C"
# BYLAW: needs_proof_of_defect
bylaws_json = '{"atoms": ["needs_proof_of_defect"]}'
"#),
    ("DUPOC", r#"def DUPOC(opp_source, k):
    if proof_search(k, opp_source, "coops_against_any"):
        return "C"
    return "D"
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["needs_proof_of_coop"]}'
"#),
    ("CIMCIC", r#"def CIMCIC(opp_source, k):
    if proof_search(k, opp_source, "implies_own_coop"):
        return "C"
    return "D"
# BYLAW: self_cond_coop
bylaws_json = '{"atoms": ["self_cond_coop"]}'
"#),
    ("PrudentBot_borne", r#"def PrudentBot_borne(opp_source, k):
    if not proof_search(k, opp_source, "coops_against_any"):
        return "D"
    if not proof_search(k, opp_source, "defects_with_db"):
        return "D"
    return "C"
# BYLAW: prudent
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["prudent", "needs_proof_of_coop"]}'
"#),
];

const ORDRE: [&str; 7] = ["CooperateBot", "DefectBot", "CooperateBotOpake",
                          "CUPOD", "DUPOC", "CIMCIC", "PrudentBot_borne"];

fn source(nom: &str) -> &'static str {
    SOURCES.iter().find(|(n, _)| *n == nom).map(|(_, s)| *s).expect("programme publie")
}

// Equivalent ligne a ligne de re.compile(r"^#\s*BYLAW:\s*([a-z_]+)\s*$", re.MULTILINE).
fn atome_bylaw(ligne: &str) -> Option<String> {
    let reste = ligne.strip_prefix('#')?.trim_start().strip_prefix("BYLAW:")?.trim();
    if !reste.is_empty() && reste.chars().all(|c| c.is_ascii_lowercase() || c == '_') {
        Some(reste.to_string())
    } else {
        None
    }
}

fn parse_certificat_a(source: &str) -> BTreeSet<String> {
    source.lines().filter_map(atome_bylaw).collect()
}

fn cloture_a(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut grossi = faits.clone();
        for (premisse, conclusions) in REGLES.iter() {
            if premisse.iter().all(|p| faits.contains(*p)) {
                grossi.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if grossi == faits {
            break;
        }
        faits = grossi;
    }
    faits
}

fn proof_search(k: usize, opp_source: &str, but: &str) -> bool {
    cloture_a(&parse_certificat_a(opp_source), k).contains(but)
}

// Interprete du sous-langage des textes publies (remplace exec : la decision vient du texte).
#[derive(Debug)]
enum Instr {
    Retour(String),                                     // return "X"
    Affecte(String, i64),                               // _code = 67
    RetourChr(String),                                  // return chr(_code) | return chr(67)
    Si { negation: bool, but: String, retour: String }, // if [not] proof_search(k, opp_source, "but"): return "X"
}

#[derive(Debug)]
struct Programme { nom: String, corps: Vec<Instr> }

fn chaine_python(s: &str) -> Option<String> {
    let s = s.trim();
    let q = s.chars().next()?;
    if (q == '"' || q == '\'') && s.len() >= 2 && s.ends_with(q) {
        let interieur = &s[1..s.len() - 1];
        if !interieur.contains(q) && !interieur.contains('\\') {
            return Some(interieur.to_string());
        }
    }
    None
}

fn charger(source: &str) -> Result<Programme, String> {
    let mut lignes = source.lines();
    let tete = lignes.next().ok_or("source vide")?;
    let nom = tete.strip_prefix("def ").and_then(|r| r.strip_suffix("(opp_source, k):"))
        .ok_or(format!("en-tete non reconnu : {}", tete))?.to_string();
    let corps_lignes: Vec<&str> = lignes.take_while(|l| l.starts_with("    ")).collect();
    let mut corps = Vec::new();
    let mut i = 0;
    while i < corps_lignes.len() {
        let l = &corps_lignes[i][4..];
        if let Some(cond) = l.strip_prefix("if ").and_then(|r| r.strip_suffix(':')) {
            let (negation, appel) = match cond.strip_prefix("not ") {
                Some(r) => (true, r),
                None => (false, cond),
            };
            let but = appel.strip_prefix("proof_search(k, opp_source, ").and_then(|r| r.strip_suffix(')'))
                .and_then(chaine_python).ok_or(format!("condition non reconnue : {}", l))?;
            let retour = corps_lignes.get(i + 1).and_then(|s| s.strip_prefix("        return "))
                .and_then(chaine_python).ok_or(format!("bloc if non reconnu apres : {}", l))?;
            corps.push(Instr::Si { negation, but, retour });
            i += 2;
            continue;
        }
        if let Some(e) = l.strip_prefix("return ") {
            if let Some(v) = chaine_python(e) {
                corps.push(Instr::Retour(v));
            } else if let Some(arg) = e.strip_prefix("chr(").and_then(|r| r.strip_suffix(')')) {
                corps.push(Instr::RetourChr(arg.trim().to_string()));
            } else {
                return Err(format!("retour non reconnu : {}", l));
            }
        } else if let Some((var, val)) = l.split_once(" = ") {
            let v: i64 = val.trim().parse().map_err(|_| format!("affectation non reconnue : {}", l))?;
            corps.push(Instr::Affecte(var.trim().to_string(), v));
        } else {
            return Err(format!("instruction non reconnue : {}", l));
        }
        i += 1;
    }
    Ok(Programme { nom, corps })
}

fn executer(p: &Programme, opp_source: &str, k: usize) -> Result<String, String> {
    let mut vars: HashMap<String, i64> = HashMap::new();
    for ins in &p.corps {
        match ins {
            Instr::Retour(v) => return Ok(v.clone()),
            Instr::Affecte(n, v) => {
                vars.insert(n.clone(), *v);
            }
            Instr::RetourChr(arg) => {
                let code = match arg.parse::<i64>() {
                    Ok(v) => v,
                    Err(_) => *vars.get(arg).ok_or(format!("nom inconnu : {}", arg))?,
                };
                return char::from_u32(code as u32).map(|c| c.to_string()).ok_or(format!("chr hors domaine : {}", code));
            }
            Instr::Si { negation, but, retour } => {
                if proof_search(k, opp_source, but) != *negation {
                    return Ok(retour.clone());
                }
            }
        }
    }
    Err(format!("{} se termine sans return", p.nom))
}

type Issue = (String, String, (i64, i64));

struct Moteur { programmes: Vec<(String, Programme)> }

impl Moteur {
    // PROGRAMMES = {nom: charger(SOURCES[nom])[nom] for nom in ORDRE}
    fn charger_tout() -> Moteur {
        let programmes = ORDRE.iter().map(|nom| {
            let p = charger(source(nom)).expect("texte publie interpretable");
            assert!(p.nom == *nom, "la source ne definit pas {}", nom);
            (nom.to_string(), p)
        }).collect();
        Moteur { programmes }
    }
    fn programme(&self, nom: &str) -> &Programme {
        &self.programmes.iter().find(|(n, _)| n == nom).expect("programme charge").1
    }
    fn action(&self, nom: &str, adverse: &str, k: usize) -> String {
        executer(self.programme(nom), source(adverse), k).expect("execution du programme")
    }
    fn issue(&self, nom_a: &str, nom_b: &str, k: usize) -> Issue {
        let act_a = self.action(nom_a, nom_b, k);
        let act_b = self.action(nom_b, nom_a, k);
        let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
        (act_a, act_b, g)
    }
}

fn paires() -> Vec<(String, String)> {
    ORDRE.iter().flat_map(|a| ORDRE.iter().map(move |b| (a.to_string(), b.to_string()))).collect()
}

// ISSUE_MOTEUR = {(a, b): issue(a, b) for a, b in itertools.product(ORDRE, repeat=2)} (ordre d'insertion)
fn table_moteur(m: &Moteur) -> Vec<((String, String), Issue)> {
    paires().into_iter().map(|(a, b)| {
        let i = m.issue(&a, &b, K_DEFAUT);
        ((a, b), i)
    }).collect()
}

fn chercher<'a>(table: &'a [((String, String), Issue)], a: &str, b: &str) -> &'a Issue {
    &table.iter().find(|((x, y), _)| x == a && y == b).expect("paire du registre").1
}

fn py_list(items: &BTreeSet<String>) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn py_pair(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }
fn py_gains(g: (i64, i64)) -> String { format!("({}, {})", g.0, g.1) }
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
// ---- fin du bloc moteur ----

// ---------------------------------------------------------------------------
// La representation publiee change une issue de PD en un coup.
// ---------------------------------------------------------------------------
let m = Moteur::charger_tout();
for adversaire in ORDRE.iter() {
    assert!(m.action("CooperateBot", adversaire, K_DEFAUT) == "C", "{}", adversaire);
    assert!(m.action("CooperateBotOpake", adversaire, K_DEFAUT) == "C", "{}", adversaire);
}
println!("Comportement identique verifie : les deux programmes cooperent contre tous les adversaires.");

let legible = m.issue("DUPOC", "CooperateBot", K_DEFAUT);
let opaque = m.issue("DUPOC", "CooperateBotOpake", K_DEFAUT);
assert!((legible.0.as_str(), legible.1.as_str()) != (opaque.0.as_str(), opaque.1.as_str()));
println!();
println!("DUPOC vs CooperateBot      : {} {}", py_pair(&legible.0, &legible.1), py_gains(legible.2));
println!("DUPOC vs CooperateBotOpake : {} {}", py_pair(&opaque.0, &opaque.1), py_gains(opaque.2));
println!();
println!("La seule difference est la representation publiee : certificat non vide contre certificat vide.");
println!("C'est le point (3) de la p. 18 : un adversaire illisible n'est pas recompense.");


Comportement identique verifie : les deux programmes cooperent contre tous les adversaires.


DUPOC vs CooperateBot      : ('C', 'C') (3, 3)


DUPOC vs CooperateBotOpake : ('D', 'C') (5, 0)


La seule difference est la representation publiee : certificat non vide contre certificat vide.


C'est le point (3) de la p. 18 : un adversaire illisible n'est pas recompense.


### Exercice 1 — une paire lisible / opaque, et la bascule qu'elle provoque

L'exemple guidé de la section 6 montre la bascule `(C, C)` vers `(D, C)` sur une paire fournie.
À vous de la reconstruire avec **vos propres** sources.

Objectif : écrire deux programmes de comportement identique (« coopérer toujours ») dont l'un
publie un certificat `uncond_coop` et l'autre rien, les charger, puis confronter `DUPOC` à
chacun et montrer que les issues diffèrent. Retourner une `struct Reponse1 { lisible, opaque, issues }` (les deux textes et les deux couples d'actions) pour permettre leur vérification.

Contraintes :

- les deux fonctions doivent porter **le même nom** `CooperateBot` : c'est une convention
  du contrat de chargement. `DUPOC` lit le certificat, pas ce nom ;
- le programme opaque ne doit pas contenir le littéral `"C"` dans un `return` : utilisez
  `chr(67)` ;
- ne pas modifier `SOURCES` ni le bloc moteur.

Indices : `charger(texte)` rend le programme interprété ; `Moteur::action` exige un nom présent dans `SOURCES`, donc appelez directement `executer(charger(texte), source("DUPOC"), K_DEFAUT)` et sa contrepartie.

In [15]:
// ---- Bloc moteur Dynamics-08 (std seul) — a recopier tel quel en tete des cellules 10 a 35 ----
use std::collections::{BTreeMap, BTreeSet, HashMap};

const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

fn gain(mien: &str, adverse: &str) -> i64 {
    if mien == "C" { if adverse == "C" { R } else { S } } else if adverse == "C" { T } else { P }
}

const K_DEFAUT: usize = 3;

const REGLES: [(&[&str], &[&str]); 9] = [
    (&["uncond_coop"], &["intent_coop"]),
    (&["uncond_defect"], &["intent_defect"]),
    (&["self_cond_coop"], &["implies_own_coop"]),
    (&["intent_coop"], &["coops_against_any", "coops_with_db"]),
    (&["intent_defect"], &["defects_against_any", "defects_with_db"]),
    (&["needs_proof_of_coop"], &["defects_with_db"]),
    (&["needs_proof_of_defect"], &["defects_with_db"]),
    (&["self_cond_coop"], &["defects_with_db"]),
    (&["prudent"], &["defects_with_db"]),
];

// Textes publies, ordre d'insertion du dict source (valeurs exactes des litteraux Python).
const SOURCES: [(&str, &str); 7] = [
    ("CooperateBot", r#"def CooperateBot(opp_source, k):
    return "C"
# BYLAW: uncond_coop
bylaws_json = '{"atoms": ["uncond_coop"]}'
"#),
    ("DefectBot", r#"def DefectBot(opp_source, k):
    return "D"
# BYLAW: uncond_defect
bylaws_json = '{"atoms": ["uncond_defect"]}'
"#),
    ("CooperateBotOpake", r#"def CooperateBotOpake(opp_source, k):
    _code = 67
    return chr(_code)
# BYLAW:
bylaws_json = '{"atoms": []}'
"#),
    ("CUPOD", r#"def CUPOD(opp_source, k):
    if proof_search(k, opp_source, "defects_against_any"):
        return "D"
    return "C"
# BYLAW: needs_proof_of_defect
bylaws_json = '{"atoms": ["needs_proof_of_defect"]}'
"#),
    ("DUPOC", r#"def DUPOC(opp_source, k):
    if proof_search(k, opp_source, "coops_against_any"):
        return "C"
    return "D"
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["needs_proof_of_coop"]}'
"#),
    ("CIMCIC", r#"def CIMCIC(opp_source, k):
    if proof_search(k, opp_source, "implies_own_coop"):
        return "C"
    return "D"
# BYLAW: self_cond_coop
bylaws_json = '{"atoms": ["self_cond_coop"]}'
"#),
    ("PrudentBot_borne", r#"def PrudentBot_borne(opp_source, k):
    if not proof_search(k, opp_source, "coops_against_any"):
        return "D"
    if not proof_search(k, opp_source, "defects_with_db"):
        return "D"
    return "C"
# BYLAW: prudent
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["prudent", "needs_proof_of_coop"]}'
"#),
];

const ORDRE: [&str; 7] = ["CooperateBot", "DefectBot", "CooperateBotOpake",
                          "CUPOD", "DUPOC", "CIMCIC", "PrudentBot_borne"];

fn source(nom: &str) -> &'static str {
    SOURCES.iter().find(|(n, _)| *n == nom).map(|(_, s)| *s).expect("programme publie")
}

// Equivalent ligne a ligne de re.compile(r"^#\s*BYLAW:\s*([a-z_]+)\s*$", re.MULTILINE).
fn atome_bylaw(ligne: &str) -> Option<String> {
    let reste = ligne.strip_prefix('#')?.trim_start().strip_prefix("BYLAW:")?.trim();
    if !reste.is_empty() && reste.chars().all(|c| c.is_ascii_lowercase() || c == '_') {
        Some(reste.to_string())
    } else {
        None
    }
}

fn parse_certificat_a(source: &str) -> BTreeSet<String> {
    source.lines().filter_map(atome_bylaw).collect()
}

fn cloture_a(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut grossi = faits.clone();
        for (premisse, conclusions) in REGLES.iter() {
            if premisse.iter().all(|p| faits.contains(*p)) {
                grossi.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if grossi == faits {
            break;
        }
        faits = grossi;
    }
    faits
}

fn proof_search(k: usize, opp_source: &str, but: &str) -> bool {
    cloture_a(&parse_certificat_a(opp_source), k).contains(but)
}

// Interprete du sous-langage des textes publies (remplace exec : la decision vient du texte).
#[derive(Debug)]
enum Instr {
    Retour(String),                                     // return "X"
    Affecte(String, i64),                               // _code = 67
    RetourChr(String),                                  // return chr(_code) | return chr(67)
    Si { negation: bool, but: String, retour: String }, // if [not] proof_search(k, opp_source, "but"): return "X"
}

#[derive(Debug)]
struct Programme { nom: String, corps: Vec<Instr> }

fn chaine_python(s: &str) -> Option<String> {
    let s = s.trim();
    let q = s.chars().next()?;
    if (q == '"' || q == '\'') && s.len() >= 2 && s.ends_with(q) {
        let interieur = &s[1..s.len() - 1];
        if !interieur.contains(q) && !interieur.contains('\\') {
            return Some(interieur.to_string());
        }
    }
    None
}

fn charger(source: &str) -> Result<Programme, String> {
    let mut lignes = source.lines();
    let tete = lignes.next().ok_or("source vide")?;
    let nom = tete.strip_prefix("def ").and_then(|r| r.strip_suffix("(opp_source, k):"))
        .ok_or(format!("en-tete non reconnu : {}", tete))?.to_string();
    let corps_lignes: Vec<&str> = lignes.take_while(|l| l.starts_with("    ")).collect();
    let mut corps = Vec::new();
    let mut i = 0;
    while i < corps_lignes.len() {
        let l = &corps_lignes[i][4..];
        if let Some(cond) = l.strip_prefix("if ").and_then(|r| r.strip_suffix(':')) {
            let (negation, appel) = match cond.strip_prefix("not ") {
                Some(r) => (true, r),
                None => (false, cond),
            };
            let but = appel.strip_prefix("proof_search(k, opp_source, ").and_then(|r| r.strip_suffix(')'))
                .and_then(chaine_python).ok_or(format!("condition non reconnue : {}", l))?;
            let retour = corps_lignes.get(i + 1).and_then(|s| s.strip_prefix("        return "))
                .and_then(chaine_python).ok_or(format!("bloc if non reconnu apres : {}", l))?;
            corps.push(Instr::Si { negation, but, retour });
            i += 2;
            continue;
        }
        if let Some(e) = l.strip_prefix("return ") {
            if let Some(v) = chaine_python(e) {
                corps.push(Instr::Retour(v));
            } else if let Some(arg) = e.strip_prefix("chr(").and_then(|r| r.strip_suffix(')')) {
                corps.push(Instr::RetourChr(arg.trim().to_string()));
            } else {
                return Err(format!("retour non reconnu : {}", l));
            }
        } else if let Some((var, val)) = l.split_once(" = ") {
            let v: i64 = val.trim().parse().map_err(|_| format!("affectation non reconnue : {}", l))?;
            corps.push(Instr::Affecte(var.trim().to_string(), v));
        } else {
            return Err(format!("instruction non reconnue : {}", l));
        }
        i += 1;
    }
    Ok(Programme { nom, corps })
}

fn executer(p: &Programme, opp_source: &str, k: usize) -> Result<String, String> {
    let mut vars: HashMap<String, i64> = HashMap::new();
    for ins in &p.corps {
        match ins {
            Instr::Retour(v) => return Ok(v.clone()),
            Instr::Affecte(n, v) => {
                vars.insert(n.clone(), *v);
            }
            Instr::RetourChr(arg) => {
                let code = match arg.parse::<i64>() {
                    Ok(v) => v,
                    Err(_) => *vars.get(arg).ok_or(format!("nom inconnu : {}", arg))?,
                };
                return char::from_u32(code as u32).map(|c| c.to_string()).ok_or(format!("chr hors domaine : {}", code));
            }
            Instr::Si { negation, but, retour } => {
                if proof_search(k, opp_source, but) != *negation {
                    return Ok(retour.clone());
                }
            }
        }
    }
    Err(format!("{} se termine sans return", p.nom))
}

type Issue = (String, String, (i64, i64));

struct Moteur { programmes: Vec<(String, Programme)> }

impl Moteur {
    // PROGRAMMES = {nom: charger(SOURCES[nom])[nom] for nom in ORDRE}
    fn charger_tout() -> Moteur {
        let programmes = ORDRE.iter().map(|nom| {
            let p = charger(source(nom)).expect("texte publie interpretable");
            assert!(p.nom == *nom, "la source ne definit pas {}", nom);
            (nom.to_string(), p)
        }).collect();
        Moteur { programmes }
    }
    fn programme(&self, nom: &str) -> &Programme {
        &self.programmes.iter().find(|(n, _)| n == nom).expect("programme charge").1
    }
    fn action(&self, nom: &str, adverse: &str, k: usize) -> String {
        executer(self.programme(nom), source(adverse), k).expect("execution du programme")
    }
    fn issue(&self, nom_a: &str, nom_b: &str, k: usize) -> Issue {
        let act_a = self.action(nom_a, nom_b, k);
        let act_b = self.action(nom_b, nom_a, k);
        let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
        (act_a, act_b, g)
    }
}

fn paires() -> Vec<(String, String)> {
    ORDRE.iter().flat_map(|a| ORDRE.iter().map(move |b| (a.to_string(), b.to_string()))).collect()
}

// ISSUE_MOTEUR = {(a, b): issue(a, b) for a, b in itertools.product(ORDRE, repeat=2)} (ordre d'insertion)
fn table_moteur(m: &Moteur) -> Vec<((String, String), Issue)> {
    paires().into_iter().map(|(a, b)| {
        let i = m.issue(&a, &b, K_DEFAUT);
        ((a, b), i)
    }).collect()
}

fn chercher<'a>(table: &'a [((String, String), Issue)], a: &str, b: &str) -> &'a Issue {
    &table.iter().find(|((x, y), _)| x == a && y == b).expect("paire du registre").1
}

fn py_list(items: &BTreeSet<String>) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn py_pair(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }
fn py_gains(g: (i64, i64)) -> String { format!("({}, {})", g.0, g.1) }
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
// ---- fin du bloc moteur ----

#[derive(Debug)]
struct Reponse1 {
    lisible: String,
    opaque: String,
    issues: Vec<(String, String)>,
}

fn exercice_1() -> Option<Reponse1> {
    // Etape 1 : ecrire SOURCE_LEGIBLE (certificat `uncond_coop`, retour "C").
    // Etape 2 : ecrire SOURCE_OPAQUE (certificat vide, retour chr(67)).
    // Etape 3 : charger les deux sources.
    // Etape 4 : confronter DUPOC a chacune et renvoyer les deux issues.
    // TODO etudiant : completer les etapes ci-dessus.
    let result: Option<Reponse1> = None;
    result
}

// Affichage stable : None tant que l'exercice est a completer.
fn py_none<T>(o: &Option<T>) -> &'static str {
    if o.is_some() { "Some(...)" } else { "None" }
}

println!("Exercice 1 a completer -> {}", py_none(&exercice_1()));
// Vérifier les textes fournis et les deux exécutions réelles, pas seulement les valeurs annoncées.
if let Some(reponse1) = exercice_1() {
    let lisible = reponse1.lisible.clone();
    let opaque = reponse1.opaque.clone();
    assert!(lisible != opaque, "les deux textes doivent differer");
    assert!(!opaque.contains("\"C\"") && !opaque.contains("'C'") && opaque.contains("chr(67)"),
            "l'opaque ne doit exposer C que via chr(67)");
    assert!(parse_certificat_a(&lisible) == ["uncond_coop".to_string()].into_iter().collect(),
            "certificat lisible attendu : {{uncond_coop}}");
    assert!(parse_certificat_a(&opaque).is_empty(), "certificat opaque attendu : vide");
    let p1 = charger(&lisible).expect("texte lisible interpretable");
    let p2 = charger(&opaque).expect("texte opaque interpretable");
    assert!(p1.nom == "CooperateBot" && p2.nom == "CooperateBot", "meme nom de programme attendu");
    let m = Moteur::charger_tout();
    for a in ORDRE.iter() {
        let a1 = executer(&p1, source(a), K_DEFAUT).expect("execution du texte lisible");
        let a2 = executer(&p2, source(a), K_DEFAUT).expect("execution du texte opaque");
        assert!(a1 == "C" && a2 == "C", "les deux doivent cooperer contre {}", a);
    }
    let dupoc = m.programme("DUPOC");
    let vrais: Vec<(String, String)> = vec![
        (executer(dupoc, &lisible, K_DEFAUT).expect("DUPOC contre le lisible"),
         executer(&p1, source("DUPOC"), K_DEFAUT).expect("le lisible contre DUPOC")),
        (executer(dupoc, &opaque, K_DEFAUT).expect("DUPOC contre l'opaque"),
         executer(&p2, source("DUPOC"), K_DEFAUT).expect("l'opaque contre DUPOC")),
    ];
    assert!(reponse1.issues == vrais
            && vrais == vec![("C".to_string(), "C".to_string()), ("D".to_string(), "C".to_string())],
            "issues {:?} != executions reelles {:?}", reponse1.issues, vrais);
};


Exercice 1 a completer -> None


## 7. Rôle de la borne `k`

La borne `k` est le seul paramètre libre du modèle. Un tour d'inférence ne peut produire un fait
que si ses prémisses sont déjà présentes : les bascules se produisent donc à des seuils précis.
L'exemple guidé ci-dessous balaie `k` de 0 à 4 sur quatre confrontations.

In [24]:
// ---- Bloc moteur Dynamics-08 (std seul) — a recopier tel quel en tete des cellules 10 a 35 ----
use std::collections::{BTreeMap, BTreeSet, HashMap};

const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

fn gain(mien: &str, adverse: &str) -> i64 {
    if mien == "C" { if adverse == "C" { R } else { S } } else if adverse == "C" { T } else { P }
}

const K_DEFAUT: usize = 3;

const REGLES: [(&[&str], &[&str]); 9] = [
    (&["uncond_coop"], &["intent_coop"]),
    (&["uncond_defect"], &["intent_defect"]),
    (&["self_cond_coop"], &["implies_own_coop"]),
    (&["intent_coop"], &["coops_against_any", "coops_with_db"]),
    (&["intent_defect"], &["defects_against_any", "defects_with_db"]),
    (&["needs_proof_of_coop"], &["defects_with_db"]),
    (&["needs_proof_of_defect"], &["defects_with_db"]),
    (&["self_cond_coop"], &["defects_with_db"]),
    (&["prudent"], &["defects_with_db"]),
];

// Textes publies, ordre d'insertion du dict source (valeurs exactes des litteraux Python).
const SOURCES: [(&str, &str); 7] = [
    ("CooperateBot", r#"def CooperateBot(opp_source, k):
    return "C"
# BYLAW: uncond_coop
bylaws_json = '{"atoms": ["uncond_coop"]}'
"#),
    ("DefectBot", r#"def DefectBot(opp_source, k):
    return "D"
# BYLAW: uncond_defect
bylaws_json = '{"atoms": ["uncond_defect"]}'
"#),
    ("CooperateBotOpake", r#"def CooperateBotOpake(opp_source, k):
    _code = 67
    return chr(_code)
# BYLAW:
bylaws_json = '{"atoms": []}'
"#),
    ("CUPOD", r#"def CUPOD(opp_source, k):
    if proof_search(k, opp_source, "defects_against_any"):
        return "D"
    return "C"
# BYLAW: needs_proof_of_defect
bylaws_json = '{"atoms": ["needs_proof_of_defect"]}'
"#),
    ("DUPOC", r#"def DUPOC(opp_source, k):
    if proof_search(k, opp_source, "coops_against_any"):
        return "C"
    return "D"
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["needs_proof_of_coop"]}'
"#),
    ("CIMCIC", r#"def CIMCIC(opp_source, k):
    if proof_search(k, opp_source, "implies_own_coop"):
        return "C"
    return "D"
# BYLAW: self_cond_coop
bylaws_json = '{"atoms": ["self_cond_coop"]}'
"#),
    ("PrudentBot_borne", r#"def PrudentBot_borne(opp_source, k):
    if not proof_search(k, opp_source, "coops_against_any"):
        return "D"
    if not proof_search(k, opp_source, "defects_with_db"):
        return "D"
    return "C"
# BYLAW: prudent
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["prudent", "needs_proof_of_coop"]}'
"#),
];

const ORDRE: [&str; 7] = ["CooperateBot", "DefectBot", "CooperateBotOpake",
                          "CUPOD", "DUPOC", "CIMCIC", "PrudentBot_borne"];

fn source(nom: &str) -> &'static str {
    SOURCES.iter().find(|(n, _)| *n == nom).map(|(_, s)| *s).expect("programme publie")
}

// Equivalent ligne a ligne de re.compile(r"^#\s*BYLAW:\s*([a-z_]+)\s*$", re.MULTILINE).
fn atome_bylaw(ligne: &str) -> Option<String> {
    let reste = ligne.strip_prefix('#')?.trim_start().strip_prefix("BYLAW:")?.trim();
    if !reste.is_empty() && reste.chars().all(|c| c.is_ascii_lowercase() || c == '_') {
        Some(reste.to_string())
    } else {
        None
    }
}

fn parse_certificat_a(source: &str) -> BTreeSet<String> {
    source.lines().filter_map(atome_bylaw).collect()
}

fn cloture_a(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut grossi = faits.clone();
        for (premisse, conclusions) in REGLES.iter() {
            if premisse.iter().all(|p| faits.contains(*p)) {
                grossi.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if grossi == faits {
            break;
        }
        faits = grossi;
    }
    faits
}

fn proof_search(k: usize, opp_source: &str, but: &str) -> bool {
    cloture_a(&parse_certificat_a(opp_source), k).contains(but)
}

// Interprete du sous-langage des textes publies (remplace exec : la decision vient du texte).
#[derive(Debug)]
enum Instr {
    Retour(String),                                     // return "X"
    Affecte(String, i64),                               // _code = 67
    RetourChr(String),                                  // return chr(_code) | return chr(67)
    Si { negation: bool, but: String, retour: String }, // if [not] proof_search(k, opp_source, "but"): return "X"
}

#[derive(Debug)]
struct Programme { nom: String, corps: Vec<Instr> }

fn chaine_python(s: &str) -> Option<String> {
    let s = s.trim();
    let q = s.chars().next()?;
    if (q == '"' || q == '\'') && s.len() >= 2 && s.ends_with(q) {
        let interieur = &s[1..s.len() - 1];
        if !interieur.contains(q) && !interieur.contains('\\') {
            return Some(interieur.to_string());
        }
    }
    None
}

fn charger(source: &str) -> Result<Programme, String> {
    let mut lignes = source.lines();
    let tete = lignes.next().ok_or("source vide")?;
    let nom = tete.strip_prefix("def ").and_then(|r| r.strip_suffix("(opp_source, k):"))
        .ok_or(format!("en-tete non reconnu : {}", tete))?.to_string();
    let corps_lignes: Vec<&str> = lignes.take_while(|l| l.starts_with("    ")).collect();
    let mut corps = Vec::new();
    let mut i = 0;
    while i < corps_lignes.len() {
        let l = &corps_lignes[i][4..];
        if let Some(cond) = l.strip_prefix("if ").and_then(|r| r.strip_suffix(':')) {
            let (negation, appel) = match cond.strip_prefix("not ") {
                Some(r) => (true, r),
                None => (false, cond),
            };
            let but = appel.strip_prefix("proof_search(k, opp_source, ").and_then(|r| r.strip_suffix(')'))
                .and_then(chaine_python).ok_or(format!("condition non reconnue : {}", l))?;
            let retour = corps_lignes.get(i + 1).and_then(|s| s.strip_prefix("        return "))
                .and_then(chaine_python).ok_or(format!("bloc if non reconnu apres : {}", l))?;
            corps.push(Instr::Si { negation, but, retour });
            i += 2;
            continue;
        }
        if let Some(e) = l.strip_prefix("return ") {
            if let Some(v) = chaine_python(e) {
                corps.push(Instr::Retour(v));
            } else if let Some(arg) = e.strip_prefix("chr(").and_then(|r| r.strip_suffix(')')) {
                corps.push(Instr::RetourChr(arg.trim().to_string()));
            } else {
                return Err(format!("retour non reconnu : {}", l));
            }
        } else if let Some((var, val)) = l.split_once(" = ") {
            let v: i64 = val.trim().parse().map_err(|_| format!("affectation non reconnue : {}", l))?;
            corps.push(Instr::Affecte(var.trim().to_string(), v));
        } else {
            return Err(format!("instruction non reconnue : {}", l));
        }
        i += 1;
    }
    Ok(Programme { nom, corps })
}

fn executer(p: &Programme, opp_source: &str, k: usize) -> Result<String, String> {
    let mut vars: HashMap<String, i64> = HashMap::new();
    for ins in &p.corps {
        match ins {
            Instr::Retour(v) => return Ok(v.clone()),
            Instr::Affecte(n, v) => {
                vars.insert(n.clone(), *v);
            }
            Instr::RetourChr(arg) => {
                let code = match arg.parse::<i64>() {
                    Ok(v) => v,
                    Err(_) => *vars.get(arg).ok_or(format!("nom inconnu : {}", arg))?,
                };
                return char::from_u32(code as u32).map(|c| c.to_string()).ok_or(format!("chr hors domaine : {}", code));
            }
            Instr::Si { negation, but, retour } => {
                if proof_search(k, opp_source, but) != *negation {
                    return Ok(retour.clone());
                }
            }
        }
    }
    Err(format!("{} se termine sans return", p.nom))
}

type Issue = (String, String, (i64, i64));

struct Moteur { programmes: Vec<(String, Programme)> }

impl Moteur {
    // PROGRAMMES = {nom: charger(SOURCES[nom])[nom] for nom in ORDRE}
    fn charger_tout() -> Moteur {
        let programmes = ORDRE.iter().map(|nom| {
            let p = charger(source(nom)).expect("texte publie interpretable");
            assert!(p.nom == *nom, "la source ne definit pas {}", nom);
            (nom.to_string(), p)
        }).collect();
        Moteur { programmes }
    }
    fn programme(&self, nom: &str) -> &Programme {
        &self.programmes.iter().find(|(n, _)| n == nom).expect("programme charge").1
    }
    fn action(&self, nom: &str, adverse: &str, k: usize) -> String {
        executer(self.programme(nom), source(adverse), k).expect("execution du programme")
    }
    fn issue(&self, nom_a: &str, nom_b: &str, k: usize) -> Issue {
        let act_a = self.action(nom_a, nom_b, k);
        let act_b = self.action(nom_b, nom_a, k);
        let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
        (act_a, act_b, g)
    }
}

fn paires() -> Vec<(String, String)> {
    ORDRE.iter().flat_map(|a| ORDRE.iter().map(move |b| (a.to_string(), b.to_string()))).collect()
}

// ISSUE_MOTEUR = {(a, b): issue(a, b) for a, b in itertools.product(ORDRE, repeat=2)} (ordre d'insertion)
fn table_moteur(m: &Moteur) -> Vec<((String, String), Issue)> {
    paires().into_iter().map(|(a, b)| {
        let i = m.issue(&a, &b, K_DEFAUT);
        ((a, b), i)
    }).collect()
}

fn chercher<'a>(table: &'a [((String, String), Issue)], a: &str, b: &str) -> &'a Issue {
    &table.iter().find(|((x, y), _)| x == a && y == b).expect("paire du registre").1
}

fn py_list(items: &BTreeSet<String>) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn py_pair(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }
fn py_gains(g: (i64, i64)) -> String { format!("({}, {})", g.0, g.1) }
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
// ---- fin du bloc moteur ----

// ---------------------------------------------------------------------------
// Balayage de la borne k : le seul parametre libre du modele.
// ---------------------------------------------------------------------------
let m = Moteur::charger_tout();
println!("{:>2} | {:>12} | {:>12} | {:>15} | {:>13}",
         "k", "CUPOD vs DB", "DUPOC vs CB", "CIMCIC/CIMCIC", "DUPOC/DUPOC");
for k in 0..5usize {
    let a = m.issue("CUPOD", "DefectBot", k);
    let b = m.issue("DUPOC", "CooperateBot", k);
    let c = m.issue("CIMCIC", "CIMCIC", k);
    let d = m.issue("DUPOC", "DUPOC", k);
    println!("{:>2} | {:>12} | {:>12} | {:>15} | {:>13}",
             k, py_pair(&a.0, &a.1), py_pair(&b.0, &b.1), py_pair(&c.0, &c.1), py_pair(&d.0, &d.1));
}

let i1 = m.issue("CUPOD", "DefectBot", 1);
assert!((i1.0.as_str(), i1.1.as_str()) == ("C", "D"));
let i2 = m.issue("CUPOD", "DefectBot", 2);
assert!((i2.0.as_str(), i2.1.as_str()) == ("D", "D"));
println!();
println!("CUPOD contre DefectBot bascule de (C, D) a (D, D) des que k atteint 2 tours.");
println!("La cloture est monotone et se stabilise : k plus grand n'ajoute plus aucun fait.");


 k |  CUPOD vs DB |  DUPOC vs CB |   CIMCIC/CIMCIC |   DUPOC/DUPOC


 0 |   ('C', 'D') |   ('D', 'C') |      ('D', 'D') |    ('D', 'D')


 1 |   ('C', 'D') |   ('D', 'C') |      ('C', 'C') |    ('D', 'D')


 2 |   ('D', 'D') |   ('C', 'C') |      ('C', 'C') |    ('D', 'D')


 3 |   ('D', 'D') |   ('C', 'C') |      ('C', 'C') |    ('D', 'D')


 4 |   ('D', 'D') |   ('C', 'C') |      ('C', 'C') |    ('D', 'D')


CUPOD contre DefectBot bascule de (C, D) a (D, D) des que k atteint 2 tours.


La cloture est monotone et se stabilise : k plus grand n'ajoute plus aucun fait.


### Exercice 2 — borne de stabilisation sur toutes les paires

L'exemple guidé balaie quatre confrontations choisies. À vous de traiter le registre entier.

Objectif : écrire `exercice_2(k_max)` qui rend, pour chaque paire du registre, la **plus petite** borne `k` telle que l'issue ne change plus pour tout `k` jusqu'à `k_max` — une `BTreeMap` complète des 49 paires, ou `None` tant que la table est incomplète.

Contraintes : n'utiliser que `Moteur::issue`, `ORDRE` et `paires()` ; ne pas recopier la table déjà affichée.

Indices : « l'issue ne change plus » signifie que toutes les issues de `k` à `k_max` sont égales à celle en `k_max`.
Cas limite à signaler explicitement : une paire peut déjà être stable en `k = 0`.

In [17]:
// ---- Bloc moteur Dynamics-08 (std seul) — a recopier tel quel en tete des cellules 10 a 35 ----
use std::collections::{BTreeMap, BTreeSet, HashMap};

const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

fn gain(mien: &str, adverse: &str) -> i64 {
    if mien == "C" { if adverse == "C" { R } else { S } } else if adverse == "C" { T } else { P }
}

const K_DEFAUT: usize = 3;

const REGLES: [(&[&str], &[&str]); 9] = [
    (&["uncond_coop"], &["intent_coop"]),
    (&["uncond_defect"], &["intent_defect"]),
    (&["self_cond_coop"], &["implies_own_coop"]),
    (&["intent_coop"], &["coops_against_any", "coops_with_db"]),
    (&["intent_defect"], &["defects_against_any", "defects_with_db"]),
    (&["needs_proof_of_coop"], &["defects_with_db"]),
    (&["needs_proof_of_defect"], &["defects_with_db"]),
    (&["self_cond_coop"], &["defects_with_db"]),
    (&["prudent"], &["defects_with_db"]),
];

// Textes publies, ordre d'insertion du dict source (valeurs exactes des litteraux Python).
const SOURCES: [(&str, &str); 7] = [
    ("CooperateBot", r#"def CooperateBot(opp_source, k):
    return "C"
# BYLAW: uncond_coop
bylaws_json = '{"atoms": ["uncond_coop"]}'
"#),
    ("DefectBot", r#"def DefectBot(opp_source, k):
    return "D"
# BYLAW: uncond_defect
bylaws_json = '{"atoms": ["uncond_defect"]}'
"#),
    ("CooperateBotOpake", r#"def CooperateBotOpake(opp_source, k):
    _code = 67
    return chr(_code)
# BYLAW:
bylaws_json = '{"atoms": []}'
"#),
    ("CUPOD", r#"def CUPOD(opp_source, k):
    if proof_search(k, opp_source, "defects_against_any"):
        return "D"
    return "C"
# BYLAW: needs_proof_of_defect
bylaws_json = '{"atoms": ["needs_proof_of_defect"]}'
"#),
    ("DUPOC", r#"def DUPOC(opp_source, k):
    if proof_search(k, opp_source, "coops_against_any"):
        return "C"
    return "D"
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["needs_proof_of_coop"]}'
"#),
    ("CIMCIC", r#"def CIMCIC(opp_source, k):
    if proof_search(k, opp_source, "implies_own_coop"):
        return "C"
    return "D"
# BYLAW: self_cond_coop
bylaws_json = '{"atoms": ["self_cond_coop"]}'
"#),
    ("PrudentBot_borne", r#"def PrudentBot_borne(opp_source, k):
    if not proof_search(k, opp_source, "coops_against_any"):
        return "D"
    if not proof_search(k, opp_source, "defects_with_db"):
        return "D"
    return "C"
# BYLAW: prudent
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["prudent", "needs_proof_of_coop"]}'
"#),
];

const ORDRE: [&str; 7] = ["CooperateBot", "DefectBot", "CooperateBotOpake",
                          "CUPOD", "DUPOC", "CIMCIC", "PrudentBot_borne"];

fn source(nom: &str) -> &'static str {
    SOURCES.iter().find(|(n, _)| *n == nom).map(|(_, s)| *s).expect("programme publie")
}

// Equivalent ligne a ligne de re.compile(r"^#\s*BYLAW:\s*([a-z_]+)\s*$", re.MULTILINE).
fn atome_bylaw(ligne: &str) -> Option<String> {
    let reste = ligne.strip_prefix('#')?.trim_start().strip_prefix("BYLAW:")?.trim();
    if !reste.is_empty() && reste.chars().all(|c| c.is_ascii_lowercase() || c == '_') {
        Some(reste.to_string())
    } else {
        None
    }
}

fn parse_certificat_a(source: &str) -> BTreeSet<String> {
    source.lines().filter_map(atome_bylaw).collect()
}

fn cloture_a(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut grossi = faits.clone();
        for (premisse, conclusions) in REGLES.iter() {
            if premisse.iter().all(|p| faits.contains(*p)) {
                grossi.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if grossi == faits {
            break;
        }
        faits = grossi;
    }
    faits
}

fn proof_search(k: usize, opp_source: &str, but: &str) -> bool {
    cloture_a(&parse_certificat_a(opp_source), k).contains(but)
}

// Interprete du sous-langage des textes publies (remplace exec : la decision vient du texte).
#[derive(Debug)]
enum Instr {
    Retour(String),                                     // return "X"
    Affecte(String, i64),                               // _code = 67
    RetourChr(String),                                  // return chr(_code) | return chr(67)
    Si { negation: bool, but: String, retour: String }, // if [not] proof_search(k, opp_source, "but"): return "X"
}

#[derive(Debug)]
struct Programme { nom: String, corps: Vec<Instr> }

fn chaine_python(s: &str) -> Option<String> {
    let s = s.trim();
    let q = s.chars().next()?;
    if (q == '"' || q == '\'') && s.len() >= 2 && s.ends_with(q) {
        let interieur = &s[1..s.len() - 1];
        if !interieur.contains(q) && !interieur.contains('\\') {
            return Some(interieur.to_string());
        }
    }
    None
}

fn charger(source: &str) -> Result<Programme, String> {
    let mut lignes = source.lines();
    let tete = lignes.next().ok_or("source vide")?;
    let nom = tete.strip_prefix("def ").and_then(|r| r.strip_suffix("(opp_source, k):"))
        .ok_or(format!("en-tete non reconnu : {}", tete))?.to_string();
    let corps_lignes: Vec<&str> = lignes.take_while(|l| l.starts_with("    ")).collect();
    let mut corps = Vec::new();
    let mut i = 0;
    while i < corps_lignes.len() {
        let l = &corps_lignes[i][4..];
        if let Some(cond) = l.strip_prefix("if ").and_then(|r| r.strip_suffix(':')) {
            let (negation, appel) = match cond.strip_prefix("not ") {
                Some(r) => (true, r),
                None => (false, cond),
            };
            let but = appel.strip_prefix("proof_search(k, opp_source, ").and_then(|r| r.strip_suffix(')'))
                .and_then(chaine_python).ok_or(format!("condition non reconnue : {}", l))?;
            let retour = corps_lignes.get(i + 1).and_then(|s| s.strip_prefix("        return "))
                .and_then(chaine_python).ok_or(format!("bloc if non reconnu apres : {}", l))?;
            corps.push(Instr::Si { negation, but, retour });
            i += 2;
            continue;
        }
        if let Some(e) = l.strip_prefix("return ") {
            if let Some(v) = chaine_python(e) {
                corps.push(Instr::Retour(v));
            } else if let Some(arg) = e.strip_prefix("chr(").and_then(|r| r.strip_suffix(')')) {
                corps.push(Instr::RetourChr(arg.trim().to_string()));
            } else {
                return Err(format!("retour non reconnu : {}", l));
            }
        } else if let Some((var, val)) = l.split_once(" = ") {
            let v: i64 = val.trim().parse().map_err(|_| format!("affectation non reconnue : {}", l))?;
            corps.push(Instr::Affecte(var.trim().to_string(), v));
        } else {
            return Err(format!("instruction non reconnue : {}", l));
        }
        i += 1;
    }
    Ok(Programme { nom, corps })
}

fn executer(p: &Programme, opp_source: &str, k: usize) -> Result<String, String> {
    let mut vars: HashMap<String, i64> = HashMap::new();
    for ins in &p.corps {
        match ins {
            Instr::Retour(v) => return Ok(v.clone()),
            Instr::Affecte(n, v) => {
                vars.insert(n.clone(), *v);
            }
            Instr::RetourChr(arg) => {
                let code = match arg.parse::<i64>() {
                    Ok(v) => v,
                    Err(_) => *vars.get(arg).ok_or(format!("nom inconnu : {}", arg))?,
                };
                return char::from_u32(code as u32).map(|c| c.to_string()).ok_or(format!("chr hors domaine : {}", code));
            }
            Instr::Si { negation, but, retour } => {
                if proof_search(k, opp_source, but) != *negation {
                    return Ok(retour.clone());
                }
            }
        }
    }
    Err(format!("{} se termine sans return", p.nom))
}

type Issue = (String, String, (i64, i64));

struct Moteur { programmes: Vec<(String, Programme)> }

impl Moteur {
    // PROGRAMMES = {nom: charger(SOURCES[nom])[nom] for nom in ORDRE}
    fn charger_tout() -> Moteur {
        let programmes = ORDRE.iter().map(|nom| {
            let p = charger(source(nom)).expect("texte publie interpretable");
            assert!(p.nom == *nom, "la source ne definit pas {}", nom);
            (nom.to_string(), p)
        }).collect();
        Moteur { programmes }
    }
    fn programme(&self, nom: &str) -> &Programme {
        &self.programmes.iter().find(|(n, _)| n == nom).expect("programme charge").1
    }
    fn action(&self, nom: &str, adverse: &str, k: usize) -> String {
        executer(self.programme(nom), source(adverse), k).expect("execution du programme")
    }
    fn issue(&self, nom_a: &str, nom_b: &str, k: usize) -> Issue {
        let act_a = self.action(nom_a, nom_b, k);
        let act_b = self.action(nom_b, nom_a, k);
        let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
        (act_a, act_b, g)
    }
}

fn paires() -> Vec<(String, String)> {
    ORDRE.iter().flat_map(|a| ORDRE.iter().map(move |b| (a.to_string(), b.to_string()))).collect()
}

// ISSUE_MOTEUR = {(a, b): issue(a, b) for a, b in itertools.product(ORDRE, repeat=2)} (ordre d'insertion)
fn table_moteur(m: &Moteur) -> Vec<((String, String), Issue)> {
    paires().into_iter().map(|(a, b)| {
        let i = m.issue(&a, &b, K_DEFAUT);
        ((a, b), i)
    }).collect()
}

fn chercher<'a>(table: &'a [((String, String), Issue)], a: &str, b: &str) -> &'a Issue {
    &table.iter().find(|((x, y), _)| x == a && y == b).expect("paire du registre").1
}

fn py_list(items: &BTreeSet<String>) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn py_pair(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }
fn py_gains(g: (i64, i64)) -> String { format!("({}, {})", g.0, g.1) }
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
// ---- fin du bloc moteur ----

fn exercice_2(k_max: usize) -> Option<BTreeMap<(String, String), usize>> {
    // Etape 1 : pour une paire (a, b), calculer l'issue de reference en k_max.
    // Etape 2 : balayer k de 0 a k_max et renvoyer le premier k stable.
    // Etape 3 : appliquer la fonction aux 49 paires ordonnees de paires()
    //           et renvoyer la table {paire: borne}.
    // TODO etudiant : completer les etapes ci-dessus.
    let result: Option<BTreeMap<(String, String), usize>> = None;
    result
}

// Affichage stable : None tant que l'exercice est a completer.
fn py_none<T>(o: &Option<T>) -> &'static str {
    if o.is_some() { "Some(...)" } else { "None" }
}

println!("Exercice 2 a completer -> {}", py_none(&exercice_2(5)));
// Deux bornes, 49 paires et tout le suffixe : une table partielle est refusée.
fn oracle_stabilisation(m: &Moteur, k_max: usize) -> BTreeMap<(String, String), usize> {
    let mut result = BTreeMap::new();
    for (a, b) in paires() {
        let v: Vec<Issue> = (0..=k_max).map(|k| m.issue(&a, &b, k)).collect();
        let borne = (0..=k_max).find(|k| v[*k..].iter().all(|x| *x == v[v.len() - 1]))
            .expect("suffixe stable en k_max");
        result.insert((a, b), borne);
    }
    result
}
let m = Moteur::charger_tout();
let reps2: Vec<Option<BTreeMap<(String, String), usize>>> = [0usize, 2, 5].iter().map(|k| exercice_2(*k)).collect();
let tous_none = reps2.iter().all(|r| r.is_none());
let tous_some = reps2.iter().all(|r| r.is_some());
assert!(tous_none || tous_some, "Réponse partielle");
if tous_some {
    for (rep, k) in reps2.iter().zip([0usize, 2, 5].iter()) {
        assert!(*rep == Some(oracle_stabilisation(&m, *k)), "k_max = {}", k);
    }
};


Exercice 2 a completer -> None


## 8. Vérificateur indépendant (implémentation 2)

Un résultat calculé par un seul programme n'est pas vérifié. On recalcule toute la table par un
**second chemin de calcul**, écrit indépendamment sur trois plans :

| Plan | Moteur principal | Vérificateur |
| --- | --- | --- |
| Lecture du certificat | marqueurs `# BYLAW:` (lecture ligne à ligne) | champ `bylaws_json` (lecteur dédié, sans bibliothèque) |
| Clôture | parcours d'un tableau de règles à prémisses ensemblistes, arrêt au point fixe | itération naïve avec retour anticipé sur un tableau prémisses vers conclusions |
| Décision | **interprétation** du texte du programme (sans `exec`) | **réécriture** de la règle de décision par rôle, sans appel au moteur |

La **spécification** — atomes, règles, table de gains — est commune : c'est elle que la
comparaison met à l'épreuve. Ce qui est indépendant, c'est le code qui l'implémente.

In [18]:
// ---------------------------------------------------------------------------
// Implementation 2 : verificateur ecrit independamment du moteur.
// ---------------------------------------------------------------------------
// ---- Bloc moteur Dynamics-08 (std seul) — a recopier tel quel en tete des cellules 10 a 35 ----
use std::collections::{BTreeMap, BTreeSet, HashMap};

const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

fn gain(mien: &str, adverse: &str) -> i64 {
    if mien == "C" { if adverse == "C" { R } else { S } } else if adverse == "C" { T } else { P }
}

const K_DEFAUT: usize = 3;

const REGLES: [(&[&str], &[&str]); 9] = [
    (&["uncond_coop"], &["intent_coop"]),
    (&["uncond_defect"], &["intent_defect"]),
    (&["self_cond_coop"], &["implies_own_coop"]),
    (&["intent_coop"], &["coops_against_any", "coops_with_db"]),
    (&["intent_defect"], &["defects_against_any", "defects_with_db"]),
    (&["needs_proof_of_coop"], &["defects_with_db"]),
    (&["needs_proof_of_defect"], &["defects_with_db"]),
    (&["self_cond_coop"], &["defects_with_db"]),
    (&["prudent"], &["defects_with_db"]),
];

// Textes publies, ordre d'insertion du dict source (valeurs exactes des litteraux Python).
const SOURCES: [(&str, &str); 7] = [
    ("CooperateBot", r#"def CooperateBot(opp_source, k):
    return "C"
# BYLAW: uncond_coop
bylaws_json = '{"atoms": ["uncond_coop"]}'
"#),
    ("DefectBot", r#"def DefectBot(opp_source, k):
    return "D"
# BYLAW: uncond_defect
bylaws_json = '{"atoms": ["uncond_defect"]}'
"#),
    ("CooperateBotOpake", r#"def CooperateBotOpake(opp_source, k):
    _code = 67
    return chr(_code)
# BYLAW:
bylaws_json = '{"atoms": []}'
"#),
    ("CUPOD", r#"def CUPOD(opp_source, k):
    if proof_search(k, opp_source, "defects_against_any"):
        return "D"
    return "C"
# BYLAW: needs_proof_of_defect
bylaws_json = '{"atoms": ["needs_proof_of_defect"]}'
"#),
    ("DUPOC", r#"def DUPOC(opp_source, k):
    if proof_search(k, opp_source, "coops_against_any"):
        return "C"
    return "D"
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["needs_proof_of_coop"]}'
"#),
    ("CIMCIC", r#"def CIMCIC(opp_source, k):
    if proof_search(k, opp_source, "implies_own_coop"):
        return "C"
    return "D"
# BYLAW: self_cond_coop
bylaws_json = '{"atoms": ["self_cond_coop"]}'
"#),
    ("PrudentBot_borne", r#"def PrudentBot_borne(opp_source, k):
    if not proof_search(k, opp_source, "coops_against_any"):
        return "D"
    if not proof_search(k, opp_source, "defects_with_db"):
        return "D"
    return "C"
# BYLAW: prudent
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["prudent", "needs_proof_of_coop"]}'
"#),
];

const ORDRE: [&str; 7] = ["CooperateBot", "DefectBot", "CooperateBotOpake",
                          "CUPOD", "DUPOC", "CIMCIC", "PrudentBot_borne"];

fn source(nom: &str) -> &'static str {
    SOURCES.iter().find(|(n, _)| *n == nom).map(|(_, s)| *s).expect("programme publie")
}

// Equivalent ligne a ligne de re.compile(r"^#\s*BYLAW:\s*([a-z_]+)\s*$", re.MULTILINE).
fn atome_bylaw(ligne: &str) -> Option<String> {
    let reste = ligne.strip_prefix('#')?.trim_start().strip_prefix("BYLAW:")?.trim();
    if !reste.is_empty() && reste.chars().all(|c| c.is_ascii_lowercase() || c == '_') {
        Some(reste.to_string())
    } else {
        None
    }
}

fn parse_certificat_a(source: &str) -> BTreeSet<String> {
    source.lines().filter_map(atome_bylaw).collect()
}

fn cloture_a(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut grossi = faits.clone();
        for (premisse, conclusions) in REGLES.iter() {
            if premisse.iter().all(|p| faits.contains(*p)) {
                grossi.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if grossi == faits {
            break;
        }
        faits = grossi;
    }
    faits
}

fn proof_search(k: usize, opp_source: &str, but: &str) -> bool {
    cloture_a(&parse_certificat_a(opp_source), k).contains(but)
}

// Interprete du sous-langage des textes publies (remplace exec : la decision vient du texte).
#[derive(Debug)]
enum Instr {
    Retour(String),                                     // return "X"
    Affecte(String, i64),                               // _code = 67
    RetourChr(String),                                  // return chr(_code) | return chr(67)
    Si { negation: bool, but: String, retour: String }, // if [not] proof_search(k, opp_source, "but"): return "X"
}

#[derive(Debug)]
struct Programme { nom: String, corps: Vec<Instr> }

fn chaine_python(s: &str) -> Option<String> {
    let s = s.trim();
    let q = s.chars().next()?;
    if (q == '"' || q == '\'') && s.len() >= 2 && s.ends_with(q) {
        let interieur = &s[1..s.len() - 1];
        if !interieur.contains(q) && !interieur.contains('\\') {
            return Some(interieur.to_string());
        }
    }
    None
}

fn charger(source: &str) -> Result<Programme, String> {
    let mut lignes = source.lines();
    let tete = lignes.next().ok_or("source vide")?;
    let nom = tete.strip_prefix("def ").and_then(|r| r.strip_suffix("(opp_source, k):"))
        .ok_or(format!("en-tete non reconnu : {}", tete))?.to_string();
    let corps_lignes: Vec<&str> = lignes.take_while(|l| l.starts_with("    ")).collect();
    let mut corps = Vec::new();
    let mut i = 0;
    while i < corps_lignes.len() {
        let l = &corps_lignes[i][4..];
        if let Some(cond) = l.strip_prefix("if ").and_then(|r| r.strip_suffix(':')) {
            let (negation, appel) = match cond.strip_prefix("not ") {
                Some(r) => (true, r),
                None => (false, cond),
            };
            let but = appel.strip_prefix("proof_search(k, opp_source, ").and_then(|r| r.strip_suffix(')'))
                .and_then(chaine_python).ok_or(format!("condition non reconnue : {}", l))?;
            let retour = corps_lignes.get(i + 1).and_then(|s| s.strip_prefix("        return "))
                .and_then(chaine_python).ok_or(format!("bloc if non reconnu apres : {}", l))?;
            corps.push(Instr::Si { negation, but, retour });
            i += 2;
            continue;
        }
        if let Some(e) = l.strip_prefix("return ") {
            if let Some(v) = chaine_python(e) {
                corps.push(Instr::Retour(v));
            } else if let Some(arg) = e.strip_prefix("chr(").and_then(|r| r.strip_suffix(')')) {
                corps.push(Instr::RetourChr(arg.trim().to_string()));
            } else {
                return Err(format!("retour non reconnu : {}", l));
            }
        } else if let Some((var, val)) = l.split_once(" = ") {
            let v: i64 = val.trim().parse().map_err(|_| format!("affectation non reconnue : {}", l))?;
            corps.push(Instr::Affecte(var.trim().to_string(), v));
        } else {
            return Err(format!("instruction non reconnue : {}", l));
        }
        i += 1;
    }
    Ok(Programme { nom, corps })
}

fn executer(p: &Programme, opp_source: &str, k: usize) -> Result<String, String> {
    let mut vars: HashMap<String, i64> = HashMap::new();
    for ins in &p.corps {
        match ins {
            Instr::Retour(v) => return Ok(v.clone()),
            Instr::Affecte(n, v) => {
                vars.insert(n.clone(), *v);
            }
            Instr::RetourChr(arg) => {
                let code = match arg.parse::<i64>() {
                    Ok(v) => v,
                    Err(_) => *vars.get(arg).ok_or(format!("nom inconnu : {}", arg))?,
                };
                return char::from_u32(code as u32).map(|c| c.to_string()).ok_or(format!("chr hors domaine : {}", code));
            }
            Instr::Si { negation, but, retour } => {
                if proof_search(k, opp_source, but) != *negation {
                    return Ok(retour.clone());
                }
            }
        }
    }
    Err(format!("{} se termine sans return", p.nom))
}

type Issue = (String, String, (i64, i64));

struct Moteur { programmes: Vec<(String, Programme)> }

impl Moteur {
    // PROGRAMMES = {nom: charger(SOURCES[nom])[nom] for nom in ORDRE}
    fn charger_tout() -> Moteur {
        let programmes = ORDRE.iter().map(|nom| {
            let p = charger(source(nom)).expect("texte publie interpretable");
            assert!(p.nom == *nom, "la source ne definit pas {}", nom);
            (nom.to_string(), p)
        }).collect();
        Moteur { programmes }
    }
    fn programme(&self, nom: &str) -> &Programme {
        &self.programmes.iter().find(|(n, _)| n == nom).expect("programme charge").1
    }
    fn action(&self, nom: &str, adverse: &str, k: usize) -> String {
        executer(self.programme(nom), source(adverse), k).expect("execution du programme")
    }
    fn issue(&self, nom_a: &str, nom_b: &str, k: usize) -> Issue {
        let act_a = self.action(nom_a, nom_b, k);
        let act_b = self.action(nom_b, nom_a, k);
        let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
        (act_a, act_b, g)
    }
}

fn paires() -> Vec<(String, String)> {
    ORDRE.iter().flat_map(|a| ORDRE.iter().map(move |b| (a.to_string(), b.to_string()))).collect()
}

// ISSUE_MOTEUR = {(a, b): issue(a, b) for a, b in itertools.product(ORDRE, repeat=2)} (ordre d'insertion)
fn table_moteur(m: &Moteur) -> Vec<((String, String), Issue)> {
    paires().into_iter().map(|(a, b)| {
        let i = m.issue(&a, &b, K_DEFAUT);
        ((a, b), i)
    }).collect()
}

fn chercher<'a>(table: &'a [((String, String), Issue)], a: &str, b: &str) -> &'a Issue {
    &table.iter().find(|((x, y), _)| x == a && y == b).expect("paire du registre").1
}

fn py_list(items: &BTreeSet<String>) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn py_pair(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }
fn py_gains(g: (i64, i64)) -> String { format!("({}, {})", g.0, g.1) }
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
// ---- fin du bloc moteur ----

// ---- Bloc verificateur Dynamics-08 (std seul) — en tete des cellules 29, 31 et 33, apres le bloc moteur ----
const REGLES_B: [(&str, &[&str]); 8] = [
    ("uncond_coop", &["intent_coop"]),
    ("uncond_defect", &["intent_defect"]),
    ("self_cond_coop", &["implies_own_coop", "defects_with_db"]),
    ("intent_coop", &["coops_against_any", "coops_with_db"]),
    ("intent_defect", &["defects_against_any", "defects_with_db"]),
    ("needs_proof_of_coop", &["defects_with_db"]),
    ("needs_proof_of_defect", &["defects_with_db"]),
    ("prudent", &["defects_with_db"]),
];

// Equivalent de re.search(r"bylaws_json\s*=\s*'([^']*)'", source) : premiere occurrence complete.
fn champ_bylaws_json(source: &str) -> Option<&str> {
    for (i, _) in source.match_indices("bylaws_json") {
        let r = source[i + "bylaws_json".len()..].trim_start();
        if let Some(r) = r.strip_prefix('=') {
            if let Some(r) = r.trim_start().strip_prefix('\'') {
                if let Some(fin) = r.find('\'') {
                    return Some(&r[..fin]);
                }
            }
        }
    }
    None
}

// Lecteur du seul format publie : {"atoms": ["a", "b", ...]} (chaines sans echappement).
fn atomes_json(texte: &str) -> Result<BTreeSet<String>, String> {
    let interieur = texte.trim().strip_prefix('{').and_then(|r| r.strip_suffix('}')).ok_or("objet JSON attendu")?;
    let liste = interieur.trim().strip_prefix("\"atoms\"").map(str::trim_start).and_then(|r| r.strip_prefix(':'))
        .map(str::trim).and_then(|r| r.strip_prefix('[')).and_then(|r| r.strip_suffix(']'))
        .ok_or("champ atoms attendu")?;
    let mut atomes = BTreeSet::new();
    for brut in liste.split(',').map(str::trim).filter(|s| !s.is_empty()) {
        let a = brut.strip_prefix('"').and_then(|r| r.strip_suffix('"'))
            .ok_or(format!("chaine JSON attendue : {}", brut))?;
        if a.contains('"') || a.contains('\\') {
            return Err(format!("echappement non gere : {}", brut));
        }
        atomes.insert(a.to_string());
    }
    Ok(atomes)
}

fn parse_certificat_b(source: &str) -> BTreeSet<String> {
    let champ = champ_bylaws_json(source).expect("champ bylaws_json absent");
    atomes_json(champ).expect("JSON du certificat")
}

fn cloture_b(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut ajouts = BTreeSet::new();
        for (premisse, conclusions) in REGLES_B.iter() {
            if faits.contains(*premisse) {
                ajouts.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if ajouts.is_subset(&faits) {
            return faits;
        }
        faits.extend(ajouts);
    }
    faits
}

fn decision_verificateur(nom: &str, source: &str, k: usize) -> Result<String, String> {
    let faits = cloture_b(&parse_certificat_b(source), k);
    let choix = |b: bool, si: &str, sinon: &str| if b { si.to_string() } else { sinon.to_string() };
    match nom {
        "CooperateBot" | "CooperateBotOpake" => Ok("C".to_string()),
        "DefectBot" => Ok("D".to_string()),
        "CUPOD" => Ok(choix(faits.contains("defects_against_any"), "D", "C")),
        "DUPOC" => Ok(choix(faits.contains("coops_against_any"), "C", "D")),
        "CIMCIC" => Ok(choix(faits.contains("implies_own_coop"), "C", "D")),
        "PrudentBot_borne" => Ok(if !faits.contains("coops_against_any") {
            "D".to_string()
        } else {
            choix(faits.contains("defects_with_db"), "C", "D")
        }),
        _ => Err(format!("role inconnu : {}", nom)),
    }
}

fn issue_verificateur(nom_a: &str, nom_b: &str, k: usize) -> Issue {
    let act_a = decision_verificateur(nom_a, source(nom_b), k).expect("role connu");
    let act_b = decision_verificateur(nom_b, source(nom_a), k).expect("role connu");
    let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
    (act_a, act_b, g)
}
// ---- fin du bloc verificateur ----

for nom in ORDRE.iter() {
    assert!(parse_certificat_a(source(nom)) == parse_certificat_b(source(nom)), "{}", nom);
}
println!("Analyses du certificat : accord sur les sept programmes.");
println!("  marqueurs `# BYLAW:` (moteur) == champ `bylaws_json` (verificateur).");
println!("Regles du verificateur : {} premisses, ecrites independamment.", REGLES_B.len());


Analyses du certificat : accord sur les sept programmes.


  marqueurs `# BYLAW:` (moteur) == champ `bylaws_json` (verificateur).


Regles du verificateur : 8 premisses, ecrites independamment.


### 8.1 Comparaison mécanique des deux implémentations

On compare non seulement les actions, mais la **relation de coopération** et la **matrice de
gains**. Un désaccord sur n'importe laquelle des trois serait un diagnostic.

In [20]:
// ---- Bloc moteur Dynamics-08 (std seul) — a recopier tel quel en tete des cellules 10 a 35 ----
use std::collections::{BTreeMap, BTreeSet, HashMap};

const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

fn gain(mien: &str, adverse: &str) -> i64 {
    if mien == "C" { if adverse == "C" { R } else { S } } else if adverse == "C" { T } else { P }
}

const K_DEFAUT: usize = 3;

const REGLES: [(&[&str], &[&str]); 9] = [
    (&["uncond_coop"], &["intent_coop"]),
    (&["uncond_defect"], &["intent_defect"]),
    (&["self_cond_coop"], &["implies_own_coop"]),
    (&["intent_coop"], &["coops_against_any", "coops_with_db"]),
    (&["intent_defect"], &["defects_against_any", "defects_with_db"]),
    (&["needs_proof_of_coop"], &["defects_with_db"]),
    (&["needs_proof_of_defect"], &["defects_with_db"]),
    (&["self_cond_coop"], &["defects_with_db"]),
    (&["prudent"], &["defects_with_db"]),
];

// Textes publies, ordre d'insertion du dict source (valeurs exactes des litteraux Python).
const SOURCES: [(&str, &str); 7] = [
    ("CooperateBot", r#"def CooperateBot(opp_source, k):
    return "C"
# BYLAW: uncond_coop
bylaws_json = '{"atoms": ["uncond_coop"]}'
"#),
    ("DefectBot", r#"def DefectBot(opp_source, k):
    return "D"
# BYLAW: uncond_defect
bylaws_json = '{"atoms": ["uncond_defect"]}'
"#),
    ("CooperateBotOpake", r#"def CooperateBotOpake(opp_source, k):
    _code = 67
    return chr(_code)
# BYLAW:
bylaws_json = '{"atoms": []}'
"#),
    ("CUPOD", r#"def CUPOD(opp_source, k):
    if proof_search(k, opp_source, "defects_against_any"):
        return "D"
    return "C"
# BYLAW: needs_proof_of_defect
bylaws_json = '{"atoms": ["needs_proof_of_defect"]}'
"#),
    ("DUPOC", r#"def DUPOC(opp_source, k):
    if proof_search(k, opp_source, "coops_against_any"):
        return "C"
    return "D"
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["needs_proof_of_coop"]}'
"#),
    ("CIMCIC", r#"def CIMCIC(opp_source, k):
    if proof_search(k, opp_source, "implies_own_coop"):
        return "C"
    return "D"
# BYLAW: self_cond_coop
bylaws_json = '{"atoms": ["self_cond_coop"]}'
"#),
    ("PrudentBot_borne", r#"def PrudentBot_borne(opp_source, k):
    if not proof_search(k, opp_source, "coops_against_any"):
        return "D"
    if not proof_search(k, opp_source, "defects_with_db"):
        return "D"
    return "C"
# BYLAW: prudent
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["prudent", "needs_proof_of_coop"]}'
"#),
];

const ORDRE: [&str; 7] = ["CooperateBot", "DefectBot", "CooperateBotOpake",
                          "CUPOD", "DUPOC", "CIMCIC", "PrudentBot_borne"];

fn source(nom: &str) -> &'static str {
    SOURCES.iter().find(|(n, _)| *n == nom).map(|(_, s)| *s).expect("programme publie")
}

// Equivalent ligne a ligne de re.compile(r"^#\s*BYLAW:\s*([a-z_]+)\s*$", re.MULTILINE).
fn atome_bylaw(ligne: &str) -> Option<String> {
    let reste = ligne.strip_prefix('#')?.trim_start().strip_prefix("BYLAW:")?.trim();
    if !reste.is_empty() && reste.chars().all(|c| c.is_ascii_lowercase() || c == '_') {
        Some(reste.to_string())
    } else {
        None
    }
}

fn parse_certificat_a(source: &str) -> BTreeSet<String> {
    source.lines().filter_map(atome_bylaw).collect()
}

fn cloture_a(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut grossi = faits.clone();
        for (premisse, conclusions) in REGLES.iter() {
            if premisse.iter().all(|p| faits.contains(*p)) {
                grossi.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if grossi == faits {
            break;
        }
        faits = grossi;
    }
    faits
}

fn proof_search(k: usize, opp_source: &str, but: &str) -> bool {
    cloture_a(&parse_certificat_a(opp_source), k).contains(but)
}

// Interprete du sous-langage des textes publies (remplace exec : la decision vient du texte).
#[derive(Debug)]
enum Instr {
    Retour(String),                                     // return "X"
    Affecte(String, i64),                               // _code = 67
    RetourChr(String),                                  // return chr(_code) | return chr(67)
    Si { negation: bool, but: String, retour: String }, // if [not] proof_search(k, opp_source, "but"): return "X"
}

#[derive(Debug)]
struct Programme { nom: String, corps: Vec<Instr> }

fn chaine_python(s: &str) -> Option<String> {
    let s = s.trim();
    let q = s.chars().next()?;
    if (q == '"' || q == '\'') && s.len() >= 2 && s.ends_with(q) {
        let interieur = &s[1..s.len() - 1];
        if !interieur.contains(q) && !interieur.contains('\\') {
            return Some(interieur.to_string());
        }
    }
    None
}

fn charger(source: &str) -> Result<Programme, String> {
    let mut lignes = source.lines();
    let tete = lignes.next().ok_or("source vide")?;
    let nom = tete.strip_prefix("def ").and_then(|r| r.strip_suffix("(opp_source, k):"))
        .ok_or(format!("en-tete non reconnu : {}", tete))?.to_string();
    let corps_lignes: Vec<&str> = lignes.take_while(|l| l.starts_with("    ")).collect();
    let mut corps = Vec::new();
    let mut i = 0;
    while i < corps_lignes.len() {
        let l = &corps_lignes[i][4..];
        if let Some(cond) = l.strip_prefix("if ").and_then(|r| r.strip_suffix(':')) {
            let (negation, appel) = match cond.strip_prefix("not ") {
                Some(r) => (true, r),
                None => (false, cond),
            };
            let but = appel.strip_prefix("proof_search(k, opp_source, ").and_then(|r| r.strip_suffix(')'))
                .and_then(chaine_python).ok_or(format!("condition non reconnue : {}", l))?;
            let retour = corps_lignes.get(i + 1).and_then(|s| s.strip_prefix("        return "))
                .and_then(chaine_python).ok_or(format!("bloc if non reconnu apres : {}", l))?;
            corps.push(Instr::Si { negation, but, retour });
            i += 2;
            continue;
        }
        if let Some(e) = l.strip_prefix("return ") {
            if let Some(v) = chaine_python(e) {
                corps.push(Instr::Retour(v));
            } else if let Some(arg) = e.strip_prefix("chr(").and_then(|r| r.strip_suffix(')')) {
                corps.push(Instr::RetourChr(arg.trim().to_string()));
            } else {
                return Err(format!("retour non reconnu : {}", l));
            }
        } else if let Some((var, val)) = l.split_once(" = ") {
            let v: i64 = val.trim().parse().map_err(|_| format!("affectation non reconnue : {}", l))?;
            corps.push(Instr::Affecte(var.trim().to_string(), v));
        } else {
            return Err(format!("instruction non reconnue : {}", l));
        }
        i += 1;
    }
    Ok(Programme { nom, corps })
}

fn executer(p: &Programme, opp_source: &str, k: usize) -> Result<String, String> {
    let mut vars: HashMap<String, i64> = HashMap::new();
    for ins in &p.corps {
        match ins {
            Instr::Retour(v) => return Ok(v.clone()),
            Instr::Affecte(n, v) => {
                vars.insert(n.clone(), *v);
            }
            Instr::RetourChr(arg) => {
                let code = match arg.parse::<i64>() {
                    Ok(v) => v,
                    Err(_) => *vars.get(arg).ok_or(format!("nom inconnu : {}", arg))?,
                };
                return char::from_u32(code as u32).map(|c| c.to_string()).ok_or(format!("chr hors domaine : {}", code));
            }
            Instr::Si { negation, but, retour } => {
                if proof_search(k, opp_source, but) != *negation {
                    return Ok(retour.clone());
                }
            }
        }
    }
    Err(format!("{} se termine sans return", p.nom))
}

type Issue = (String, String, (i64, i64));

struct Moteur { programmes: Vec<(String, Programme)> }

impl Moteur {
    // PROGRAMMES = {nom: charger(SOURCES[nom])[nom] for nom in ORDRE}
    fn charger_tout() -> Moteur {
        let programmes = ORDRE.iter().map(|nom| {
            let p = charger(source(nom)).expect("texte publie interpretable");
            assert!(p.nom == *nom, "la source ne definit pas {}", nom);
            (nom.to_string(), p)
        }).collect();
        Moteur { programmes }
    }
    fn programme(&self, nom: &str) -> &Programme {
        &self.programmes.iter().find(|(n, _)| n == nom).expect("programme charge").1
    }
    fn action(&self, nom: &str, adverse: &str, k: usize) -> String {
        executer(self.programme(nom), source(adverse), k).expect("execution du programme")
    }
    fn issue(&self, nom_a: &str, nom_b: &str, k: usize) -> Issue {
        let act_a = self.action(nom_a, nom_b, k);
        let act_b = self.action(nom_b, nom_a, k);
        let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
        (act_a, act_b, g)
    }
}

fn paires() -> Vec<(String, String)> {
    ORDRE.iter().flat_map(|a| ORDRE.iter().map(move |b| (a.to_string(), b.to_string()))).collect()
}

// ISSUE_MOTEUR = {(a, b): issue(a, b) for a, b in itertools.product(ORDRE, repeat=2)} (ordre d'insertion)
fn table_moteur(m: &Moteur) -> Vec<((String, String), Issue)> {
    paires().into_iter().map(|(a, b)| {
        let i = m.issue(&a, &b, K_DEFAUT);
        ((a, b), i)
    }).collect()
}

fn chercher<'a>(table: &'a [((String, String), Issue)], a: &str, b: &str) -> &'a Issue {
    &table.iter().find(|((x, y), _)| x == a && y == b).expect("paire du registre").1
}

fn py_list(items: &BTreeSet<String>) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn py_pair(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }
fn py_gains(g: (i64, i64)) -> String { format!("({}, {})", g.0, g.1) }
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
// ---- fin du bloc moteur ----

// ---- Bloc verificateur Dynamics-08 (std seul) — en tete des cellules 29, 31 et 33, apres le bloc moteur ----
const REGLES_B: [(&str, &[&str]); 8] = [
    ("uncond_coop", &["intent_coop"]),
    ("uncond_defect", &["intent_defect"]),
    ("self_cond_coop", &["implies_own_coop", "defects_with_db"]),
    ("intent_coop", &["coops_against_any", "coops_with_db"]),
    ("intent_defect", &["defects_against_any", "defects_with_db"]),
    ("needs_proof_of_coop", &["defects_with_db"]),
    ("needs_proof_of_defect", &["defects_with_db"]),
    ("prudent", &["defects_with_db"]),
];

// Equivalent de re.search(r"bylaws_json\s*=\s*'([^']*)'", source) : premiere occurrence complete.
fn champ_bylaws_json(source: &str) -> Option<&str> {
    for (i, _) in source.match_indices("bylaws_json") {
        let r = source[i + "bylaws_json".len()..].trim_start();
        if let Some(r) = r.strip_prefix('=') {
            if let Some(r) = r.trim_start().strip_prefix('\'') {
                if let Some(fin) = r.find('\'') {
                    return Some(&r[..fin]);
                }
            }
        }
    }
    None
}

// Lecteur du seul format publie : {"atoms": ["a", "b", ...]} (chaines sans echappement).
fn atomes_json(texte: &str) -> Result<BTreeSet<String>, String> {
    let interieur = texte.trim().strip_prefix('{').and_then(|r| r.strip_suffix('}')).ok_or("objet JSON attendu")?;
    let liste = interieur.trim().strip_prefix("\"atoms\"").map(str::trim_start).and_then(|r| r.strip_prefix(':'))
        .map(str::trim).and_then(|r| r.strip_prefix('[')).and_then(|r| r.strip_suffix(']'))
        .ok_or("champ atoms attendu")?;
    let mut atomes = BTreeSet::new();
    for brut in liste.split(',').map(str::trim).filter(|s| !s.is_empty()) {
        let a = brut.strip_prefix('"').and_then(|r| r.strip_suffix('"'))
            .ok_or(format!("chaine JSON attendue : {}", brut))?;
        if a.contains('"') || a.contains('\\') {
            return Err(format!("echappement non gere : {}", brut));
        }
        atomes.insert(a.to_string());
    }
    Ok(atomes)
}

fn parse_certificat_b(source: &str) -> BTreeSet<String> {
    let champ = champ_bylaws_json(source).expect("champ bylaws_json absent");
    atomes_json(champ).expect("JSON du certificat")
}

fn cloture_b(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut ajouts = BTreeSet::new();
        for (premisse, conclusions) in REGLES_B.iter() {
            if faits.contains(*premisse) {
                ajouts.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if ajouts.is_subset(&faits) {
            return faits;
        }
        faits.extend(ajouts);
    }
    faits
}

fn decision_verificateur(nom: &str, source: &str, k: usize) -> Result<String, String> {
    let faits = cloture_b(&parse_certificat_b(source), k);
    let choix = |b: bool, si: &str, sinon: &str| if b { si.to_string() } else { sinon.to_string() };
    match nom {
        "CooperateBot" | "CooperateBotOpake" => Ok("C".to_string()),
        "DefectBot" => Ok("D".to_string()),
        "CUPOD" => Ok(choix(faits.contains("defects_against_any"), "D", "C")),
        "DUPOC" => Ok(choix(faits.contains("coops_against_any"), "C", "D")),
        "CIMCIC" => Ok(choix(faits.contains("implies_own_coop"), "C", "D")),
        "PrudentBot_borne" => Ok(if !faits.contains("coops_against_any") {
            "D".to_string()
        } else {
            choix(faits.contains("defects_with_db"), "C", "D")
        }),
        _ => Err(format!("role inconnu : {}", nom)),
    }
}

fn issue_verificateur(nom_a: &str, nom_b: &str, k: usize) -> Issue {
    let act_a = decision_verificateur(nom_a, source(nom_b), k).expect("role connu");
    let act_b = decision_verificateur(nom_b, source(nom_a), k).expect("role connu");
    let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
    (act_a, act_b, g)
}
// ---- fin du bloc verificateur ----

// ---------------------------------------------------------------------------
// Comparaison mecanique : actions, relation de cooperation, matrice de gains.
// ---------------------------------------------------------------------------
let m = Moteur::charger_tout();
let ps = paires();
let im = table_moteur(&m);
let desaccords: Vec<((String, String), Issue)> = im.iter()
    .filter(|((a, b), i)| *i != issue_verificateur(a, b, K_DEFAUT))
    .map(|(p, i)| (p.clone(), i.clone()))
    .collect();
assert!(desaccords.is_empty(), "{} desaccords", desaccords.len());

let relation_moteur: BTreeSet<(String, String)> = im.iter()
    .filter(|(_, i)| i.0 == "C")
    .map(|((a, b), _)| (a.clone(), b.clone()))
    .collect();
let relation_verif: BTreeSet<(String, String)> = ps.iter()
    .filter(|(a, b)| issue_verificateur(a, b, K_DEFAUT).0 == "C")
    .map(|(a, b)| (a.clone(), b.clone()))
    .collect();
let gains_moteur: BTreeMap<(String, String), (i64, i64)> = im.iter()
    .map(|((a, b), i)| ((a.clone(), b.clone()), i.2))
    .collect();
let gains_verif: BTreeMap<(String, String), (i64, i64)> = ps.iter()
    .map(|(a, b)| ((a.clone(), b.clone()), issue_verificateur(a, b, K_DEFAUT).2))
    .collect();

assert!(relation_moteur == relation_verif);
assert!(gains_moteur == gains_verif);

println!("Paires ordonnees comparees : {}", ps.len());
println!("Desaccords moteur / verificateur : {}", desaccords.len());
println!("Relation de cooperation identique : {}", py_bool(relation_moteur == relation_verif));
println!("Matrice de gains identique        : {}", py_bool(gains_moteur == gains_verif));
println!();
println!("Deux implementations independantes, un seul verdict sur les trois objets compares.");


Paires ordonnees comparees : 49


Desaccords moteur / verificateur : 0


Relation de cooperation identique : True


Matrice de gains identique        : True


Deux implementations independantes, un seul verdict sur les trois objets compares.


### Exercice 3 — une troisième vérification des paires

Le vérificateur de la section 8 est un second chemin. Écrivez un **troisième** contrôle
qui reconstruit et agrège les paires à partir de sa fonction de décision, puis retrouve la
non-exploitabilité de `CUPOD`, `DUPOC` et `CIMCIC`
(propositions 3.1, 3.2 et 5.1) sur **toutes** les paires du registre. Cette reconstruction
est indépendante de la table déjà calculée, mais partage la décision du moteur B : ce n'est
pas une troisième implémentation indépendante de la règle de décision.

Objectif : rendre une `BTreeMap` de témoins `{role: paire_temoin ou None}`, où une paire
témoin est une paire dont le couple d'actions est l'issue interdite du rôle
(`CUPOD` interdit `DC`, `DUPOC` interdit `CD`, `CIMCIC` interdit `CD`).

Contraintes : ne pas utiliser la table du moteur (c'est le but), ni `issue_verificateur`, ni `exploite_par`. Partir de `decision_verificateur` et `gain` uniquement.

Indice : `decision_verificateur(nom, source, k)` rend une action isolée ; appelez-la deux fois
pour reconstruire un couple d'actions. Le contrôle réussi rend trois valeurs `None`.

Les orientations diffèrent selon le rôle : pour CUPOD, le témoin interdit D/C vérifierait qu’il profite de la coopération adverse ; pour DUPOC et CIMCIC, C/D signifierait qu’ils subissent la défection adverse. Le contrôle n’assimile donc pas ces deux sens de non-exploitation.


In [22]:
// ---- Bloc moteur Dynamics-08 (std seul) — a recopier tel quel en tete des cellules 10 a 35 ----
use std::collections::{BTreeMap, BTreeSet, HashMap};

const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

fn gain(mien: &str, adverse: &str) -> i64 {
    if mien == "C" { if adverse == "C" { R } else { S } } else if adverse == "C" { T } else { P }
}

const K_DEFAUT: usize = 3;

const REGLES: [(&[&str], &[&str]); 9] = [
    (&["uncond_coop"], &["intent_coop"]),
    (&["uncond_defect"], &["intent_defect"]),
    (&["self_cond_coop"], &["implies_own_coop"]),
    (&["intent_coop"], &["coops_against_any", "coops_with_db"]),
    (&["intent_defect"], &["defects_against_any", "defects_with_db"]),
    (&["needs_proof_of_coop"], &["defects_with_db"]),
    (&["needs_proof_of_defect"], &["defects_with_db"]),
    (&["self_cond_coop"], &["defects_with_db"]),
    (&["prudent"], &["defects_with_db"]),
];

// Textes publies, ordre d'insertion du dict source (valeurs exactes des litteraux Python).
const SOURCES: [(&str, &str); 7] = [
    ("CooperateBot", r#"def CooperateBot(opp_source, k):
    return "C"
# BYLAW: uncond_coop
bylaws_json = '{"atoms": ["uncond_coop"]}'
"#),
    ("DefectBot", r#"def DefectBot(opp_source, k):
    return "D"
# BYLAW: uncond_defect
bylaws_json = '{"atoms": ["uncond_defect"]}'
"#),
    ("CooperateBotOpake", r#"def CooperateBotOpake(opp_source, k):
    _code = 67
    return chr(_code)
# BYLAW:
bylaws_json = '{"atoms": []}'
"#),
    ("CUPOD", r#"def CUPOD(opp_source, k):
    if proof_search(k, opp_source, "defects_against_any"):
        return "D"
    return "C"
# BYLAW: needs_proof_of_defect
bylaws_json = '{"atoms": ["needs_proof_of_defect"]}'
"#),
    ("DUPOC", r#"def DUPOC(opp_source, k):
    if proof_search(k, opp_source, "coops_against_any"):
        return "C"
    return "D"
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["needs_proof_of_coop"]}'
"#),
    ("CIMCIC", r#"def CIMCIC(opp_source, k):
    if proof_search(k, opp_source, "implies_own_coop"):
        return "C"
    return "D"
# BYLAW: self_cond_coop
bylaws_json = '{"atoms": ["self_cond_coop"]}'
"#),
    ("PrudentBot_borne", r#"def PrudentBot_borne(opp_source, k):
    if not proof_search(k, opp_source, "coops_against_any"):
        return "D"
    if not proof_search(k, opp_source, "defects_with_db"):
        return "D"
    return "C"
# BYLAW: prudent
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["prudent", "needs_proof_of_coop"]}'
"#),
];

const ORDRE: [&str; 7] = ["CooperateBot", "DefectBot", "CooperateBotOpake",
                          "CUPOD", "DUPOC", "CIMCIC", "PrudentBot_borne"];

fn source(nom: &str) -> &'static str {
    SOURCES.iter().find(|(n, _)| *n == nom).map(|(_, s)| *s).expect("programme publie")
}

// Equivalent ligne a ligne de re.compile(r"^#\s*BYLAW:\s*([a-z_]+)\s*$", re.MULTILINE).
fn atome_bylaw(ligne: &str) -> Option<String> {
    let reste = ligne.strip_prefix('#')?.trim_start().strip_prefix("BYLAW:")?.trim();
    if !reste.is_empty() && reste.chars().all(|c| c.is_ascii_lowercase() || c == '_') {
        Some(reste.to_string())
    } else {
        None
    }
}

fn parse_certificat_a(source: &str) -> BTreeSet<String> {
    source.lines().filter_map(atome_bylaw).collect()
}

fn cloture_a(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut grossi = faits.clone();
        for (premisse, conclusions) in REGLES.iter() {
            if premisse.iter().all(|p| faits.contains(*p)) {
                grossi.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if grossi == faits {
            break;
        }
        faits = grossi;
    }
    faits
}

fn proof_search(k: usize, opp_source: &str, but: &str) -> bool {
    cloture_a(&parse_certificat_a(opp_source), k).contains(but)
}

// Interprete du sous-langage des textes publies (remplace exec : la decision vient du texte).
#[derive(Debug)]
enum Instr {
    Retour(String),                                     // return "X"
    Affecte(String, i64),                               // _code = 67
    RetourChr(String),                                  // return chr(_code) | return chr(67)
    Si { negation: bool, but: String, retour: String }, // if [not] proof_search(k, opp_source, "but"): return "X"
}

#[derive(Debug)]
struct Programme { nom: String, corps: Vec<Instr> }

fn chaine_python(s: &str) -> Option<String> {
    let s = s.trim();
    let q = s.chars().next()?;
    if (q == '"' || q == '\'') && s.len() >= 2 && s.ends_with(q) {
        let interieur = &s[1..s.len() - 1];
        if !interieur.contains(q) && !interieur.contains('\\') {
            return Some(interieur.to_string());
        }
    }
    None
}

fn charger(source: &str) -> Result<Programme, String> {
    let mut lignes = source.lines();
    let tete = lignes.next().ok_or("source vide")?;
    let nom = tete.strip_prefix("def ").and_then(|r| r.strip_suffix("(opp_source, k):"))
        .ok_or(format!("en-tete non reconnu : {}", tete))?.to_string();
    let corps_lignes: Vec<&str> = lignes.take_while(|l| l.starts_with("    ")).collect();
    let mut corps = Vec::new();
    let mut i = 0;
    while i < corps_lignes.len() {
        let l = &corps_lignes[i][4..];
        if let Some(cond) = l.strip_prefix("if ").and_then(|r| r.strip_suffix(':')) {
            let (negation, appel) = match cond.strip_prefix("not ") {
                Some(r) => (true, r),
                None => (false, cond),
            };
            let but = appel.strip_prefix("proof_search(k, opp_source, ").and_then(|r| r.strip_suffix(')'))
                .and_then(chaine_python).ok_or(format!("condition non reconnue : {}", l))?;
            let retour = corps_lignes.get(i + 1).and_then(|s| s.strip_prefix("        return "))
                .and_then(chaine_python).ok_or(format!("bloc if non reconnu apres : {}", l))?;
            corps.push(Instr::Si { negation, but, retour });
            i += 2;
            continue;
        }
        if let Some(e) = l.strip_prefix("return ") {
            if let Some(v) = chaine_python(e) {
                corps.push(Instr::Retour(v));
            } else if let Some(arg) = e.strip_prefix("chr(").and_then(|r| r.strip_suffix(')')) {
                corps.push(Instr::RetourChr(arg.trim().to_string()));
            } else {
                return Err(format!("retour non reconnu : {}", l));
            }
        } else if let Some((var, val)) = l.split_once(" = ") {
            let v: i64 = val.trim().parse().map_err(|_| format!("affectation non reconnue : {}", l))?;
            corps.push(Instr::Affecte(var.trim().to_string(), v));
        } else {
            return Err(format!("instruction non reconnue : {}", l));
        }
        i += 1;
    }
    Ok(Programme { nom, corps })
}

fn executer(p: &Programme, opp_source: &str, k: usize) -> Result<String, String> {
    let mut vars: HashMap<String, i64> = HashMap::new();
    for ins in &p.corps {
        match ins {
            Instr::Retour(v) => return Ok(v.clone()),
            Instr::Affecte(n, v) => {
                vars.insert(n.clone(), *v);
            }
            Instr::RetourChr(arg) => {
                let code = match arg.parse::<i64>() {
                    Ok(v) => v,
                    Err(_) => *vars.get(arg).ok_or(format!("nom inconnu : {}", arg))?,
                };
                return char::from_u32(code as u32).map(|c| c.to_string()).ok_or(format!("chr hors domaine : {}", code));
            }
            Instr::Si { negation, but, retour } => {
                if proof_search(k, opp_source, but) != *negation {
                    return Ok(retour.clone());
                }
            }
        }
    }
    Err(format!("{} se termine sans return", p.nom))
}

type Issue = (String, String, (i64, i64));

struct Moteur { programmes: Vec<(String, Programme)> }

impl Moteur {
    // PROGRAMMES = {nom: charger(SOURCES[nom])[nom] for nom in ORDRE}
    fn charger_tout() -> Moteur {
        let programmes = ORDRE.iter().map(|nom| {
            let p = charger(source(nom)).expect("texte publie interpretable");
            assert!(p.nom == *nom, "la source ne definit pas {}", nom);
            (nom.to_string(), p)
        }).collect();
        Moteur { programmes }
    }
    fn programme(&self, nom: &str) -> &Programme {
        &self.programmes.iter().find(|(n, _)| n == nom).expect("programme charge").1
    }
    fn action(&self, nom: &str, adverse: &str, k: usize) -> String {
        executer(self.programme(nom), source(adverse), k).expect("execution du programme")
    }
    fn issue(&self, nom_a: &str, nom_b: &str, k: usize) -> Issue {
        let act_a = self.action(nom_a, nom_b, k);
        let act_b = self.action(nom_b, nom_a, k);
        let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
        (act_a, act_b, g)
    }
}

fn paires() -> Vec<(String, String)> {
    ORDRE.iter().flat_map(|a| ORDRE.iter().map(move |b| (a.to_string(), b.to_string()))).collect()
}

// ISSUE_MOTEUR = {(a, b): issue(a, b) for a, b in itertools.product(ORDRE, repeat=2)} (ordre d'insertion)
fn table_moteur(m: &Moteur) -> Vec<((String, String), Issue)> {
    paires().into_iter().map(|(a, b)| {
        let i = m.issue(&a, &b, K_DEFAUT);
        ((a, b), i)
    }).collect()
}

fn chercher<'a>(table: &'a [((String, String), Issue)], a: &str, b: &str) -> &'a Issue {
    &table.iter().find(|((x, y), _)| x == a && y == b).expect("paire du registre").1
}

fn py_list(items: &BTreeSet<String>) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn py_pair(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }
fn py_gains(g: (i64, i64)) -> String { format!("({}, {})", g.0, g.1) }
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
// ---- fin du bloc moteur ----

// ---- Bloc verificateur Dynamics-08 (std seul) — en tete des cellules 29, 31 et 33, apres le bloc moteur ----
const REGLES_B: [(&str, &[&str]); 8] = [
    ("uncond_coop", &["intent_coop"]),
    ("uncond_defect", &["intent_defect"]),
    ("self_cond_coop", &["implies_own_coop", "defects_with_db"]),
    ("intent_coop", &["coops_against_any", "coops_with_db"]),
    ("intent_defect", &["defects_against_any", "defects_with_db"]),
    ("needs_proof_of_coop", &["defects_with_db"]),
    ("needs_proof_of_defect", &["defects_with_db"]),
    ("prudent", &["defects_with_db"]),
];

// Equivalent de re.search(r"bylaws_json\s*=\s*'([^']*)'", source) : premiere occurrence complete.
fn champ_bylaws_json(source: &str) -> Option<&str> {
    for (i, _) in source.match_indices("bylaws_json") {
        let r = source[i + "bylaws_json".len()..].trim_start();
        if let Some(r) = r.strip_prefix('=') {
            if let Some(r) = r.trim_start().strip_prefix('\'') {
                if let Some(fin) = r.find('\'') {
                    return Some(&r[..fin]);
                }
            }
        }
    }
    None
}

// Lecteur du seul format publie : {"atoms": ["a", "b", ...]} (chaines sans echappement).
fn atomes_json(texte: &str) -> Result<BTreeSet<String>, String> {
    let interieur = texte.trim().strip_prefix('{').and_then(|r| r.strip_suffix('}')).ok_or("objet JSON attendu")?;
    let liste = interieur.trim().strip_prefix("\"atoms\"").map(str::trim_start).and_then(|r| r.strip_prefix(':'))
        .map(str::trim).and_then(|r| r.strip_prefix('[')).and_then(|r| r.strip_suffix(']'))
        .ok_or("champ atoms attendu")?;
    let mut atomes = BTreeSet::new();
    for brut in liste.split(',').map(str::trim).filter(|s| !s.is_empty()) {
        let a = brut.strip_prefix('"').and_then(|r| r.strip_suffix('"'))
            .ok_or(format!("chaine JSON attendue : {}", brut))?;
        if a.contains('"') || a.contains('\\') {
            return Err(format!("echappement non gere : {}", brut));
        }
        atomes.insert(a.to_string());
    }
    Ok(atomes)
}

fn parse_certificat_b(source: &str) -> BTreeSet<String> {
    let champ = champ_bylaws_json(source).expect("champ bylaws_json absent");
    atomes_json(champ).expect("JSON du certificat")
}

fn cloture_b(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut ajouts = BTreeSet::new();
        for (premisse, conclusions) in REGLES_B.iter() {
            if faits.contains(*premisse) {
                ajouts.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if ajouts.is_subset(&faits) {
            return faits;
        }
        faits.extend(ajouts);
    }
    faits
}

fn decision_verificateur(nom: &str, source: &str, k: usize) -> Result<String, String> {
    let faits = cloture_b(&parse_certificat_b(source), k);
    let choix = |b: bool, si: &str, sinon: &str| if b { si.to_string() } else { sinon.to_string() };
    match nom {
        "CooperateBot" | "CooperateBotOpake" => Ok("C".to_string()),
        "DefectBot" => Ok("D".to_string()),
        "CUPOD" => Ok(choix(faits.contains("defects_against_any"), "D", "C")),
        "DUPOC" => Ok(choix(faits.contains("coops_against_any"), "C", "D")),
        "CIMCIC" => Ok(choix(faits.contains("implies_own_coop"), "C", "D")),
        "PrudentBot_borne" => Ok(if !faits.contains("coops_against_any") {
            "D".to_string()
        } else {
            choix(faits.contains("defects_with_db"), "C", "D")
        }),
        _ => Err(format!("role inconnu : {}", nom)),
    }
}

fn issue_verificateur(nom_a: &str, nom_b: &str, k: usize) -> Issue {
    let act_a = decision_verificateur(nom_a, source(nom_b), k).expect("role connu");
    let act_b = decision_verificateur(nom_b, source(nom_a), k).expect("role connu");
    let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
    (act_a, act_b, g)
}
// ---- fin du bloc verificateur ----

type Decideur = fn(&str, &str, usize) -> Result<String, String>;

fn exercice_3(decider: Decideur) -> Option<BTreeMap<String, Option<(String, String)>>> {
    // Etape 1 : issues_interdites = {"CUPOD": "DC", "DUPOC": "CD", "CIMCIC": "CD"}
    // Etape 2 : parcourir les paires ordonnees et reconstruire l'issue via decider.
    // Etape 3 : collecter un temoin par role, puis renvoyer le dictionnaire.
    // TODO etudiant : completer les etapes ci-dessus.
    let result: Option<BTreeMap<String, Option<(String, String)>>> = None;
    result
}

// Affichage stable : None tant que l'exercice est a completer.
fn py_none<T>(o: &Option<T>) -> &'static str {
    if o.is_some() { "Some(...)" } else { "None" }
}

println!("Exercice 3 a completer -> {}", py_none(&exercice_3(decision_verificateur)));
// Le témoin synthétique doit être découvert : toujours renvoyer None est un mutant.
fn decision_fausse(nom: &str, _source: &str, _k: usize) -> Result<String, String> {
    Ok(if nom == "CUPOD" { "D".to_string() } else { "C".to_string() })
}
let attendu3: BTreeMap<String, Option<(String, String)>> = [
    ("CUPOD", Some(("CUPOD".to_string(), ORDRE[0].to_string()))),
    ("DUPOD", Some(("DUPOD".to_string(), "CUPOD".to_string()))),
    ("CIMCIC", Some(("CIMCIC".to_string(), "CUPOD".to_string()))),
].iter().map(|(k, v)| (k.to_string(), v.clone())).collect();
let reps3 = [exercice_3(decision_verificateur), exercice_3(decision_fausse)];
let tous_none = reps3.iter().all(|r| r.is_none());
let tous_some = reps3.iter().all(|r| r.is_some());
assert!(tous_none || tous_some, "Réponse partielle");
if tous_some {
    let aucun: BTreeMap<String, Option<(String, String)>> = ["CUPOD", "DUPOC", "CIMCIC"]
        .iter().map(|n| (n.to_string(), None)).collect();
    let rep0 = reps3[0].as_ref().expect("verificateur some");
    let rep1 = reps3[1].as_ref().expect("mutant some");
    assert!(*rep0 == aucun, "temoins attendus : aucun, {:?}", rep0);
    assert!(*rep1 == attendu3, "temoins attendus chez le mutant, {:?}", rep1);
};


Exercice 3 a completer -> None


## 9. Open Problem 3 : DUPOC(k) contre CUPOD(k)

L'Open Problem 3 (p. 18) conjecture, pour `k` grand,
`outcome(DUPOC(k), CUPOD(k)) == (D, C)`, et demande si c'est bien le cas. L'article explique
pourquoi la question est difficile : l'argument de symétrie naïf est invalide, car `DUPOC(k)`
n'est pas le miroir exact de `CUPOD(k)` — les lettres `C` et `D` ne sont pas échangées dans le
`proof_checker`. Il faut raisonner sur l'échec d'une recherche de preuve d'un agent pendant que
l'autre ne peut pas prouver cet échec, ce que PBLT ne permet pas.

**Cet Open Problem reste explicitement ouvert.** Le modèle borné calcule une issue pour cette
paire, comme pour toutes les autres, mais ce calcul **ne tranche pas** la conjecture : il
n'implémente pas la notion de preuve non bornée sur laquelle porte l'énoncé. La cellule qui suit
affiche l'observation et le dit ; aucun exercice du notebook ne porte sur ce problème.

In [25]:
// ---- Bloc moteur Dynamics-08 (std seul) — a recopier tel quel en tete des cellules 10 a 35 ----
use std::collections::{BTreeMap, BTreeSet, HashMap};

const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

fn gain(mien: &str, adverse: &str) -> i64 {
    if mien == "C" { if adverse == "C" { R } else { S } } else if adverse == "C" { T } else { P }
}

const K_DEFAUT: usize = 3;

const REGLES: [(&[&str], &[&str]); 9] = [
    (&["uncond_coop"], &["intent_coop"]),
    (&["uncond_defect"], &["intent_defect"]),
    (&["self_cond_coop"], &["implies_own_coop"]),
    (&["intent_coop"], &["coops_against_any", "coops_with_db"]),
    (&["intent_defect"], &["defects_against_any", "defects_with_db"]),
    (&["needs_proof_of_coop"], &["defects_with_db"]),
    (&["needs_proof_of_defect"], &["defects_with_db"]),
    (&["self_cond_coop"], &["defects_with_db"]),
    (&["prudent"], &["defects_with_db"]),
];

// Textes publies, ordre d'insertion du dict source (valeurs exactes des litteraux Python).
const SOURCES: [(&str, &str); 7] = [
    ("CooperateBot", r#"def CooperateBot(opp_source, k):
    return "C"
# BYLAW: uncond_coop
bylaws_json = '{"atoms": ["uncond_coop"]}'
"#),
    ("DefectBot", r#"def DefectBot(opp_source, k):
    return "D"
# BYLAW: uncond_defect
bylaws_json = '{"atoms": ["uncond_defect"]}'
"#),
    ("CooperateBotOpake", r#"def CooperateBotOpake(opp_source, k):
    _code = 67
    return chr(_code)
# BYLAW:
bylaws_json = '{"atoms": []}'
"#),
    ("CUPOD", r#"def CUPOD(opp_source, k):
    if proof_search(k, opp_source, "defects_against_any"):
        return "D"
    return "C"
# BYLAW: needs_proof_of_defect
bylaws_json = '{"atoms": ["needs_proof_of_defect"]}'
"#),
    ("DUPOC", r#"def DUPOC(opp_source, k):
    if proof_search(k, opp_source, "coops_against_any"):
        return "C"
    return "D"
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["needs_proof_of_coop"]}'
"#),
    ("CIMCIC", r#"def CIMCIC(opp_source, k):
    if proof_search(k, opp_source, "implies_own_coop"):
        return "C"
    return "D"
# BYLAW: self_cond_coop
bylaws_json = '{"atoms": ["self_cond_coop"]}'
"#),
    ("PrudentBot_borne", r#"def PrudentBot_borne(opp_source, k):
    if not proof_search(k, opp_source, "coops_against_any"):
        return "D"
    if not proof_search(k, opp_source, "defects_with_db"):
        return "D"
    return "C"
# BYLAW: prudent
# BYLAW: needs_proof_of_coop
bylaws_json = '{"atoms": ["prudent", "needs_proof_of_coop"]}'
"#),
];

const ORDRE: [&str; 7] = ["CooperateBot", "DefectBot", "CooperateBotOpake",
                          "CUPOD", "DUPOC", "CIMCIC", "PrudentBot_borne"];

fn source(nom: &str) -> &'static str {
    SOURCES.iter().find(|(n, _)| *n == nom).map(|(_, s)| *s).expect("programme publie")
}

// Equivalent ligne a ligne de re.compile(r"^#\s*BYLAW:\s*([a-z_]+)\s*$", re.MULTILINE).
fn atome_bylaw(ligne: &str) -> Option<String> {
    let reste = ligne.strip_prefix('#')?.trim_start().strip_prefix("BYLAW:")?.trim();
    if !reste.is_empty() && reste.chars().all(|c| c.is_ascii_lowercase() || c == '_') {
        Some(reste.to_string())
    } else {
        None
    }
}

fn parse_certificat_a(source: &str) -> BTreeSet<String> {
    source.lines().filter_map(atome_bylaw).collect()
}

fn cloture_a(graine: &BTreeSet<String>, k: usize) -> BTreeSet<String> {
    let mut faits = graine.clone();
    for _ in 0..k {
        let mut grossi = faits.clone();
        for (premisse, conclusions) in REGLES.iter() {
            if premisse.iter().all(|p| faits.contains(*p)) {
                grossi.extend(conclusions.iter().map(|c| c.to_string()));
            }
        }
        if grossi == faits {
            break;
        }
        faits = grossi;
    }
    faits
}

fn proof_search(k: usize, opp_source: &str, but: &str) -> bool {
    cloture_a(&parse_certificat_a(opp_source), k).contains(but)
}

// Interprete du sous-langage des textes publies (remplace exec : la decision vient du texte).
#[derive(Debug)]
enum Instr {
    Retour(String),                                     // return "X"
    Affecte(String, i64),                               // _code = 67
    RetourChr(String),                                  // return chr(_code) | return chr(67)
    Si { negation: bool, but: String, retour: String }, // if [not] proof_search(k, opp_source, "but"): return "X"
}

#[derive(Debug)]
struct Programme { nom: String, corps: Vec<Instr> }

fn chaine_python(s: &str) -> Option<String> {
    let s = s.trim();
    let q = s.chars().next()?;
    if (q == '"' || q == '\'') && s.len() >= 2 && s.ends_with(q) {
        let interieur = &s[1..s.len() - 1];
        if !interieur.contains(q) && !interieur.contains('\\') {
            return Some(interieur.to_string());
        }
    }
    None
}

fn charger(source: &str) -> Result<Programme, String> {
    let mut lignes = source.lines();
    let tete = lignes.next().ok_or("source vide")?;
    let nom = tete.strip_prefix("def ").and_then(|r| r.strip_suffix("(opp_source, k):"))
        .ok_or(format!("en-tete non reconnu : {}", tete))?.to_string();
    let corps_lignes: Vec<&str> = lignes.take_while(|l| l.starts_with("    ")).collect();
    let mut corps = Vec::new();
    let mut i = 0;
    while i < corps_lignes.len() {
        let l = &corps_lignes[i][4..];
        if let Some(cond) = l.strip_prefix("if ").and_then(|r| r.strip_suffix(':')) {
            let (negation, appel) = match cond.strip_prefix("not ") {
                Some(r) => (true, r),
                None => (false, cond),
            };
            let but = appel.strip_prefix("proof_search(k, opp_source, ").and_then(|r| r.strip_suffix(')'))
                .and_then(chaine_python).ok_or(format!("condition non reconnue : {}", l))?;
            let retour = corps_lignes.get(i + 1).and_then(|s| s.strip_prefix("        return "))
                .and_then(chaine_python).ok_or(format!("bloc if non reconnu apres : {}", l))?;
            corps.push(Instr::Si { negation, but, retour });
            i += 2;
            continue;
        }
        if let Some(e) = l.strip_prefix("return ") {
            if let Some(v) = chaine_python(e) {
                corps.push(Instr::Retour(v));
            } else if let Some(arg) = e.strip_prefix("chr(").and_then(|r| r.strip_suffix(')')) {
                corps.push(Instr::RetourChr(arg.trim().to_string()));
            } else {
                return Err(format!("retour non reconnu : {}", l));
            }
        } else if let Some((var, val)) = l.split_once(" = ") {
            let v: i64 = val.trim().parse().map_err(|_| format!("affectation non reconnue : {}", l))?;
            corps.push(Instr::Affecte(var.trim().to_string(), v));
        } else {
            return Err(format!("instruction non reconnue : {}", l));
        }
        i += 1;
    }
    Ok(Programme { nom, corps })
}

fn executer(p: &Programme, opp_source: &str, k: usize) -> Result<String, String> {
    let mut vars: HashMap<String, i64> = HashMap::new();
    for ins in &p.corps {
        match ins {
            Instr::Retour(v) => return Ok(v.clone()),
            Instr::Affecte(n, v) => {
                vars.insert(n.clone(), *v);
            }
            Instr::RetourChr(arg) => {
                let code = match arg.parse::<i64>() {
                    Ok(v) => v,
                    Err(_) => *vars.get(arg).ok_or(format!("nom inconnu : {}", arg))?,
                };
                return char::from_u32(code as u32).map(|c| c.to_string()).ok_or(format!("chr hors domaine : {}", code));
            }
            Instr::Si { negation, but, retour } => {
                if proof_search(k, opp_source, but) != *negation {
                    return Ok(retour.clone());
                }
            }
        }
    }
    Err(format!("{} se termine sans return", p.nom))
}

type Issue = (String, String, (i64, i64));

struct Moteur { programmes: Vec<(String, Programme)> }

impl Moteur {
    // PROGRAMMES = {nom: charger(SOURCES[nom])[nom] for nom in ORDRE}
    fn charger_tout() -> Moteur {
        let programmes = ORDRE.iter().map(|nom| {
            let p = charger(source(nom)).expect("texte publie interpretable");
            assert!(p.nom == *nom, "la source ne definit pas {}", nom);
            (nom.to_string(), p)
        }).collect();
        Moteur { programmes }
    }
    fn programme(&self, nom: &str) -> &Programme {
        &self.programmes.iter().find(|(n, _)| n == nom).expect("programme charge").1
    }
    fn action(&self, nom: &str, adverse: &str, k: usize) -> String {
        executer(self.programme(nom), source(adverse), k).expect("execution du programme")
    }
    fn issue(&self, nom_a: &str, nom_b: &str, k: usize) -> Issue {
        let act_a = self.action(nom_a, nom_b, k);
        let act_b = self.action(nom_b, nom_a, k);
        let g = (gain(&act_a, &act_b), gain(&act_b, &act_a));
        (act_a, act_b, g)
    }
}

fn paires() -> Vec<(String, String)> {
    ORDRE.iter().flat_map(|a| ORDRE.iter().map(move |b| (a.to_string(), b.to_string()))).collect()
}

// ISSUE_MOTEUR = {(a, b): issue(a, b) for a, b in itertools.product(ORDRE, repeat=2)} (ordre d'insertion)
fn table_moteur(m: &Moteur) -> Vec<((String, String), Issue)> {
    paires().into_iter().map(|(a, b)| {
        let i = m.issue(&a, &b, K_DEFAUT);
        ((a, b), i)
    }).collect()
}

fn chercher<'a>(table: &'a [((String, String), Issue)], a: &str, b: &str) -> &'a Issue {
    &table.iter().find(|((x, y), _)| x == a && y == b).expect("paire du registre").1
}

fn py_list(items: &BTreeSet<String>) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn py_pair(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }
fn py_gains(g: (i64, i64)) -> String { format!("({}, {})", g.0, g.1) }
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
// ---- fin du bloc moteur ----

// ---------------------------------------------------------------------------
// Open Problem 3 : observation bornee, sans valeur de resolution.
// ---------------------------------------------------------------------------
let m = Moteur::charger_tout();
let du_cu = m.issue("DUPOC", "CUPOD", K_DEFAUT);
let cu_du = m.issue("CUPOD", "DUPOC", K_DEFAUT);
println!("Observation bornee : outcome(DUPOC, CUPOD) = {} {}", py_pair(&du_cu.0, &du_cu.1), py_gains(du_cu.2));
println!("Observation bornee : outcome(CUPOD, DUPOC) = {} {}", py_pair(&cu_du.0, &cu_du.1), py_gains(cu_du.2));
println!();
println!("Open Problem 3 (p. 18) : la conjecture outcome(DUPOC(k), CUPOD(k)) == (D, C) reste OUVERTE.");
println!("Le present modele ne la tranche pas : sa cloture finie ne reproduit pas le raisonnement sur");
println!("l'echec d'une recherche de preuve non bornee dont l'article a besoin, et PBLT ne s'y");
println!("applique pas (l'article mentionne lui-meme l'absence de 'self-fulfilling prophecy' evident).");
println!();
println!("Aucune cellule de ce notebook ne presente cette observation comme un resultat, et aucun");
println!("exercice ne porte sur Open Problem 3.");


Observation bornee : outcome(DUPOC, CUPOD) = ('D', 'C') (5, 0)


Observation bornee : outcome(CUPOD, DUPOC) = ('C', 'D') (0, 5)


Open Problem 3 (p. 18) : la conjecture outcome(DUPOC(k), CUPOD(k)) == (D, C) reste OUVERTE.


Le present modele ne la tranche pas : sa cloture finie ne reproduit pas le raisonnement sur


l'echec d'une recherche de preuve non bornee dont l'article a besoin, et PBLT ne s'y


applique pas (l'article mentionne lui-meme l'absence de 'self-fulfilling prophecy' evident).


Aucune cellule de ce notebook ne presente cette observation comme un resultat, et aucun


exercice ne porte sur Open Problem 3.


## 10. Les dix problèmes ouverts de la source

| # | Énoncé | Statut dans la source | Prérequis | Expérience finie possible |
| --- | --- | --- | --- | --- |
| 1 | Prouver le théorème de Löb sans le point fixe modal `Psi <-> (Box Psi -> C)` | Open Problem 1, p. 15, théorie de la preuve | Logique de la prouvabilité, théorème du point fixe modal | Non : la question est l'existence d'une preuve, non un calcul. Au mieux, tester une formalisation partielle. |
| 2 | Conditions sous lesquelles une population contenant des DUPOC évolue vers des agents G-fair | Open Problem 2, p. 18, dynamique de populations | Dynamique évolutionnaire, simulations | Oui : tournoi répété sur une population finie de petits certificats, mesurer la part de G-fair. |
| 3 | Pour `k` grand, `outcome(DUPOC(k), CUPOD(k)) == (D, C)` | Open Problem 3, p. 18 — **problème explicitement ouvert** | Raisonner sur l'échec d'une recherche de preuve non bornée ; PBLT ne s'applique pas | **Non résolutoire** : le modèle borné calcule une issue, mais celle-ci ne tranche pas l'énoncé. Aucun exercice ne porte sur ce problème. |
| 4 | Implémenter DUPOC par recherche heuristique de preuve en HOL/ML ou Coq, et vérifier l'arrêt coopératif sur une machine de bureau | Open Problem 4, p. 20, ingénierie de la preuve | HOL/ML ou Coq, tactiques heuristiques | Oui, mais hors notebook : exige un assistant de preuve réel. |
| 5 | Que vaut `outcome(CUPOD(k), CIMCIC(k))` | Open Problem 5, p. 22 | Même difficulté que le problème 3 | Non résolutoire : observation bornée possible, sans valeur de preuve. |
| 6 | Que vaut `outcome(DUPOC(k), DIMCID(k))` | Open Problem 6, p. 24 | Définition de DIMCID, p. 22 | Non résolutoire ; DIMCID n'est pas défini dans ce notebook. |
| 7 | Généraliser PBLT en analogue borné de Gödel-Löb suivant les longueurs de preuve | Open Problem 7, p. 25 et 26 | Logique modale, PBLT, Critch 2016 théorème 4.2 | Partielle : instrumenter les longueurs dans un système de preuve jouet. |
| 8 | Appliquer le résultat du problème 7 pour améliorer l'algorithme Haskell du dépôt publique `klao/provability` | Open Problem 8, p. 26 | Haskell, sémantique de Kripke | Oui, mais dépend du problème 7. |
| 9 | Existe-t-il une version bornée de PrudentBot | Open Problem 9, p. 26 | Définition de PrudentBot (LaVictoire et al. 2017, cité p. 26) | Oui : le candidat `PrudentBot_borne` de ce notebook en est un essai. Il ne coopère avec aucun agent du registre, ce qui **illustre** la difficulté sans résoudre le problème 9. |
| 10 | Implémenter un CDEBot borné obtenant `(C, C)` contre DUPOC(k) et `(D, D)` contre EUPOD(k) | Open Problem 10, p. 27 | PD étendu à trois actions (E), application conditionnelle de PBLT | Oui : matrice bornée sur le jeu étendu à trois actions. |

Les dix énoncés ci-dessus sont ceux de la source, avec leur numérotation et leur page. Les
problèmes 3, 5 et 6 sont ceux que l'article désigne lui-même comme ne relevant pas d'une
« self-fulfilling prophecy » démontrable par PBLT (p. 32 et 33).

## 11. Limites et résidu

- **Le modèle n'est pas la source.** La clôture finie n'est pas le `proof_search` de l'article :
  aucun énoncé impliquant PBLT (théorèmes 3.4, 3.7, 5.2(b)) n'est reproduit. Ces énoncés sont
  cités comme tels dans le tableau de la section 4 et dans le contrôle négatif.
- **Les seuils de `k` ne sont pas transposables.** L'article compte des caractères de preuve, ce
  notebook des tours d'inférence. Seule la monotonie est commune.
- **`PrudentBot_borne` est un candidat, pas une solution.** Sa ligne dans la matrice est
  entièrement `D` : au sens du modèle, il ne coopère avec aucun agent du registre. C'est
  cohérent avec le fait que le PrudentBot de LaVictoire et al. repose sur une **recherche de
  preuve supplémentaire** dont l'analogue borné est précisément l'Open Problem 9, laissé ouvert.
- **Les certificats sont déclaratifs.** Un programme peut mentir sur son certificat ; le modèle
  suppose la publication sincère, comme l'article suppose un `proof_check` correct. La question
  de la vérification des certificats n'est pas traitée ici.
- **`CooperateBotOpake` est un contrôle local**, sans équivalent publié : il sert uniquement à
  isoler l'effet de la représentation dans la section 6.

## 12. Conclusion

Ce notebook a défini une sémantique **bornée et terminante** des agents `CUPOD`, `DUPOC`,
`PrudentBot` et `CIMCIC`, puis a classé chaque affirmation dans un des trois régimes
— **observation finie**, **résultat reproduit**, **théorème cité**.

À retenir :

1. Dans le jeu de programmes transparents, la lecture des certificats change l'issue observée : `DUPOC` obtient
   `(C, C)` contre `CooperateBot`, et `(D, D)` contre `DefectBot`.
2. La transparence **seule** ne suffit pas : `DUPOC` contre `DUPOC` donne `(D, D)` dans le
   modèle borné, alors que le théorème 3.7 donne `(C, C)` via PBLT. Le lemme manquant est
   exactement ce que le modèle ne peut pas reproduire.
3. La **représentation** change l'issue à comportement constant : `CooperateBot` et
   `CooperateBotOpake` coopèrent tous deux contre tous, mais `DUPOC` ne récompense que le
   premier — le point (3) de la p. 18.
4. Les propositions 3.1, 3.2 et 5.1 sont reproduites sur tout le registre, et le théorème 5.2(a)
   l'est aussi, parce que sa preuve se réduit à une tautologie.
5. Vérifier demande **deux chemins** : le moteur et le vérificateur sont indépendants dans leur
   lecture du certificat, leur algorithme de clôture et leur évaluateur de décision, et
   s'accordent sur les actions, la relation de coopération et la matrice de gains.
6. **Open Problem 3 reste ouvert** : le notebook l'observe sans le trancher, et n'en fait ni un
   exercice ni un résultat.

Pour aller plus loin : les dix problèmes ouverts de la section 10, et les notebooks voisins
`GameTheory-06e` (cadre des jeux-programmes) et `GameTheory-06g` (analogue par simulation).

## À retenir

- **La transparence seule ne suffit pas** : `Observation finie : outcome(DUPOC, DUPOC) = ('D', 'D') (1, 1)` — le théorème 3.7 donne `(C, C)` pour `k` grand via PBLT : sans le lemme de prouvabilité, les deux font défection.
- **L'écart est en miroir** : `Observation finie : outcome(CUPOD, CUPOD) = ('C', 'C') (3, 3)` là où le théorème 3.4 donne `(D, D)` — le modèle borné est incomplet dans les deux sens, et le carnet le dit.
- **La représentation décide du mérite** : `DUPOC vs CooperateBot      : ('C', 'C') (3, 3)` et `DUPOC vs CooperateBotOpake : ('D', 'C') (5, 0)` — deux adversaires au comportement identique, l'un au certificat publié et l'autre au certificat vide : l'issue change.
- **Moteur et vérificateur s'accordent** : `Desaccords moteur / verificateur : 0` sur `Paires ordonnees comparees : 49`, avec `Relation de cooperation identique : True` et `Matrice de gains identique        : True`.
- **La clôture est monotone** : `La cloture est monotone et se stabilise : k plus grand n'ajoute plus aucun fait.` — et `CUPOD contre DefectBot bascule de (C, D) a (D, D) des que k atteint 2 tours.`
- **Un problème ouvert reste ouvert** : `Open Problem 3 (p. 18)` — le carnet l'observe sans le trancher, et aucun exercice ne porte dessus.


## Vérifiez votre compréhension

1. Que montrent `Observation finie : outcome(DUPOC, DUPOC) = ('D', 'D') (1, 1)` et le théorème 3.7 cité, et que doit-on en conclure ?

   <details><summary>Réponse</summary>

   **Réponse.** Le modèle borné produit `('D', 'D')` pour `DUPOC` contre `DUPOC`, là où le théorème 3.7 (p. 14) annonce `outcome(DUPOC(k), DUPOC(k)) == (C, C)` pour `k` grand, via le lemme 3.6 (PBLT). La conclusion honnête est double : le modèle **ne reproduit pas** le théorème, il en mesure l'écart — la transparence mutuelle ne suffit pas sans le raisonnement de prouvabilité, que la borne ne sait pas représenter. L'observation est un **résultat du modèle**, pas un résultat de la source.

   </details>

2. Que prouve `Desaccords moteur / verificateur : 0` sur `Paires ordonnees comparees : 49` — et que ne prouve pas ?

   <details><summary>Réponse</summary>

   **Réponse.** Que les deux implémentations indépendantes — le moteur, qui lit les marqueurs `# BYLAW:`, et le vérificateur, dont `Regles du verificateur : 8 premisses, ecrites independamment.` — produisent le même verdict sur 49 paires ordonnées : même relation de coopération, même matrice de gains. Cela **contrôle l'implémentation**, pas la sémantique : les deux peuvent se tromper ensemble sur la question de la bornitude, et le carnet le dit dans sa section des limites.

   </details>


## Pour aller plus loin

**Carnet suivant.** [Dynamics-09 — Biens publics et N personnes](Dynamics-09-Biens-Publics-N-Personnes-Python.ipynb) — le passage de deux agents à N joueurs : ce que la transparence des programmes dit du bien public.

**Carnets voisins.**

- [Dynamics-04 — Open-source game theory](Dynamics-04-Open-Source-Game-Theory-Python.ipynb) — le moteur dont ce carnet reprend l'instrument de simulation.
- [Dynamics-04b — FairBot et Loeb (Lean)](Dynamics-04b-FairBot-Loeb-Lean.ipynb) — la version formelle d'un agent qui lit la source de l'adversaire.
- [Dynamics-05 — Agents bornés (Python)](Dynamics-05-Agents-Bornes-Python.ipynb) — la famille de budgets et le miroir décidable.

**Lectures.**

- Critch, Dennis & Russell (2022), *Cooperative and uncooperative institution designs: Surprises and problems in open-source game theory*, arXiv:2208.07006 — notice héritée (théorèmes 3.4 et 3.7, lemme 3.6, problèmes ouverts).
- LaVictoire et al. (2017) — `PrudentBot`, l'agent non borné dont ce carnet implémente une version bornée (p. 26 de la source).
